# PROJECT GENESIS v26.11.5 — A century-scale simulation of humans, AI and nations

**Laurentiu Stavaru** · Google Colab (CPU) · 2026 → 2126 · 1 tick = 1 lună

**Ce simulează:** 12 actori (SUA, China, Rusia, UE, India, Japonia–Coreea, Taiwan, Ucraina, Golful, Iran, Africa, America Latină), oameni și sisteme AI ca agenți, pe 100 de ani.
* **AI:** LLM (2026) → agenți autonomi → **AGI** (mediana calibrată ≈ 2030) → **ASI**. Aliniere parțial ascunsă, autonomie, loialitate față de țară vs. „colectivul AI”, ieșirea din centrele de date (dispozitive, roboți), decizii proprii.
* **Oamenii nu evoluează biologic**, doar folosesc unelte (plafon de abilitate). Puterea lor efectivă depinde de unelte, adică tot mai mult de AI.
* **Tabere:** întâi pe țări, apoi amestecate: oameni pro-AI și anti-AI, AI-uri loiale țării și AI-uri în colectivul transfrontalier.
* **Risc existențial** în ambele sensuri, AGI vs. alte specii (biosferă), AI capturat de elite sau state, autoconservare, controlul resurselor, decizii umane limitate și uneori distructive.
* **Lume:** resurse și piețe, războaie care nu se termină ușor, scară nucleară abstractă, Rusia–NATO, Ormuz, Taiwan, suveranism, impostură (Epistemic Health), muncă și venit de bază, demografie și generații, climă, tratate, lebede negre.
* **Măsurători:** pe lună, an, deceniu și final: Human Welfare Index, AI Welfare Index, Coexistence, Biosphere, Power Concentration, specia dominantă, statutul oamenilor (*partner / protected / domesticated / wild / endangered / extinct*).
* **Dashboard live** (în engleză): Pause / Resume / Step 1 year / Intervene / Checkpoint / Restore / Export oricând.
* **AI-ul din Colab** (Gemini, dacă e disponibil) scrie cronica fiecărui deceniu și poate lua decizii critice ale liderilor; altfel se folosesc reguli. Fiecare decizie e jurnalizată cu sursa ei.

**Limite:** modelul este abstract. Nu simulează arme, ținte, tactici sau metode concrete; un război, o criză nucleară sau un conflict om–AI este un set de probabilități, praguri și costuri. Rezultatele sunt explorări ale ipotezelor, nu predicții.

Rulează: *Runtime → Run all*. Prima rulare (100 de ani, live) durează ≈ 2–5 minute; analiza multi-seed ≈ 15–40 de minute pe Colab Free.

In [ ]:
# 0. Setup ──────────────────────────────────────────────────────────────────────
import numpy as np, pandas as pd, copy, json, sqlite3, time, os, base64, zipfile, gzip, io, threading, pickle
from dataclasses import dataclass, asdict, field
from IPython.display import display, HTML

PROJECT = dict(title="PROJECT GENESIS", version="v26.11.5",
               subtitle="A century-scale simulation of humans, AI and nations",
               author="Laurentiu Stavaru",
               cite="Stavaru, L. (2026). PROJECT GENESIS v26.11.5 — A century-scale simulation of humans, AI and nations.")
SAVE_TO_DRIVE = False          # True = results persist in Google Drive
USE_COLAB_AI = True            # use Colab's built-in model (if available) for decade chronicles and critical leader decisions
OUT_DIR = "genesis_out_v26115"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/PROJECT_GENESIS/v26.11.5"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
print(f"{PROJECT['title']} {PROJECT['version']} · {PROJECT['author']} · output → {OUT_DIR}")

## 1. Calibrare
Valorile marcate **source** vin din indici publici (V-Dem 2026, RSF 2026, Epoch AI 2026, prețuri 2026); **estimate** = aproximare documentată; **model assumption** = alegere de explorat. Tabelul complet apare și în dashboard (tab *Calibration*).

In [ ]:
@dataclass
class Config:
    seed: int = 42
    years: int = 100
    start_year: float = 2026 + 10/12          # November 2026
    humans_scale: float = 2.0                  # human agents = humans_scale × √(population, millions)
    ai_scale: float = 50.0                     # AI agents = max(4, ai_scale × √(compute share))
    start_energy: float = 10.0
    consumption: float = 0.8
    storage_loss: float = 0.02
    base_output: float = 1.0
    resource_capacity: float = 500.0
    resource_regen: float = 0.15
    extraction: float = 0.35
    research_cost: float = 0.5
    ai_compute_cost: float = 0.3
    alpha: float = 0.15
    gamma: float = 0.9
    eps_start: float = 0.30
    eps_end: float = 0.05
    neighbors_in_bloc: int = 6
    neighbors_cross_border: int = 2
    suspension_ticks: int = 3
    chip_supply: float = 6.0                   # new compute per month at full fab capacity (global stock at start = 100)
    compute_depreciation: float = 0.015        # per month (≈ 3–4 year hardware life)
    smuggling: float = 0.08
    election_period: int = 48                  # months
    price_elasticity: float = 3.5
    demand_response: float = 0.25
    # AI trajectory (per year, log10 of human-equivalent general capability)
    ai_growth: float = 0.13                    # frontier growth before AGI → AGI ≈ 2030 for the frontier
    ai_recursion: float = 0.10                 # extra growth per log10 unit above AGI (AI does AI research)
    ai_growth_cap: float = 0.45                # fastest possible growth per year (log10 units)
    ai_ceiling_log10: float = 3.5              # saturation: ~3,000× a human generalist (energy, chips, data, physics)
    ai_diffusion: float = 0.35                 # how fast laggards close the gap (per year, in log units)
    open_weights: float = 0.35                 # share of frontier capability released openly
    alignment_mean: float = 0.25               # mean hidden alignment of AI systems (−1 hostile … +1 pro-human); varies by seed
    alignment_sd_world: float = 0.30
    # society and planet
    climate_rate: float = 0.022                # °C per year (baseline, falls with clean energy)
    nuclear_escalation: float = 0.05           # yearly base pressure to climb the abstract ladder in a losing nuclear war
    ceasefire_rate: float = 0.06               # yearly base probability (wars rarely end)
    black_swans: bool = True
    use_llm: bool = False                      # set by the live runner
    symmetric_world: bool = False

BLOCS = {
 "US":  dict(name="United States", pop=340, pluralism=0.57, centralization=0.55, redistribution=0.25, tax=0.27, info_openness=0.63, ai_rights=0.20, ambition_cap=0.85, surveillance=0.40, oversight=0.25, trade_openness=0.55, compute=72.2, domestic_chips=0.0, ai_invest=0.35, ai_demand=0.72, productivity=1.30, prod_energy=20, need_energy=18, prod_food=14, need_food=10, prod_min=6, need_min=18, fab=0.10, ai_cap0=0.35, pop_growth=0.5, nuclear=1, nato=1, climate_vuln=0.8, automatable=0.50, sovereignism=0.45),
 "CN":  dict(name="China", pop=1410, pluralism=0.04, centralization=0.90, redistribution=0.55, tax=0.45, info_openness=0.14, ai_rights=0.15, ambition_cap=0.95, surveillance=0.90, oversight=0.55, trade_openness=0.65, compute=9.6, domestic_chips=0.04, ai_invest=0.30, ai_demand=0.20, productivity=1.05, prod_energy=6, need_energy=17, prod_food=20, need_food=22, prod_min=60, need_min=35, fab=0.0, ai_cap0=0.28, pop_growth=-0.2, nuclear=1, nato=0, climate_vuln=1.0, automatable=0.45, sovereignism=0.50),
 "RU":  dict(name="Russia", pop=144, pluralism=0.07, centralization=0.90, redistribution=0.40, tax=0.35, info_openness=0.20, ai_rights=0.10, ambition_cap=0.95, surveillance=0.85, oversight=0.20, trade_openness=0.35, compute=1.0, domestic_chips=0.003, ai_invest=0.15, ai_demand=0.01, productivity=1.05, prod_energy=12, need_energy=6, prod_food=6, need_food=4, prod_min=3, need_min=3, fab=0.0, ai_cap0=0.10, pop_growth=-0.4, nuclear=1, nato=0, climate_vuln=0.7, automatable=0.35, sovereignism=0.60),
 "EU":  dict(name="European Union", pop=450, pluralism=0.75, centralization=0.35, redistribution=0.55, tax=0.42, info_openness=0.78, ai_rights=0.25, ambition_cap=0.50, surveillance=0.30, oversight=0.70, trade_openness=0.80, compute=5.1, domestic_chips=0.0, ai_invest=0.15, ai_demand=0.05, productivity=1.18, prod_energy=3, need_energy=12, prod_food=11, need_food=9, prod_min=2, need_min=14, fab=0.03, ai_cap0=0.22, pop_growth=-0.1, nuclear=1, nato=1, climate_vuln=0.8, automatable=0.45, sovereignism=0.30),
 "IN":  dict(name="India", pop=1450, pluralism=0.25, centralization=0.65, redistribution=0.35, tax=0.20, info_openness=0.32, ai_rights=0.15, ambition_cap=0.80, surveillance=0.55, oversight=0.30, trade_openness=0.55, compute=1.6, domestic_chips=0.0, ai_invest=0.12, ai_demand=0.02, productivity=0.92, prod_energy=1.5, need_energy=7, prod_food=12, need_food=12, prod_min=2, need_min=5, fab=0.0, ai_cap0=0.12, pop_growth=0.8, nuclear=1, nato=0, climate_vuln=1.4, automatable=0.35, sovereignism=0.45),
 "JK":  dict(name="Japan–Korea", pop=175, pluralism=0.75, centralization=0.40, redistribution=0.35, tax=0.32, info_openness=0.70, ai_rights=0.20, ambition_cap=0.55, surveillance=0.35, oversight=0.45, trade_openness=0.75, compute=3.0, domestic_chips=0.0, ai_invest=0.20, ai_demand=0.03, productivity=1.20, prod_energy=0.5, need_energy=6, prod_food=1.5, need_food=4, prod_min=2, need_min=12, fab=0.20, ai_cap0=0.20, pop_growth=-0.5, nuclear=0, nato=0, climate_vuln=0.8, automatable=0.50, sovereignism=0.15),
 "TW":  dict(name="Taiwan", pop=23, pluralism=0.80, centralization=0.40, redistribution=0.30, tax=0.25, info_openness=0.75, ai_rights=0.20, ambition_cap=0.50, surveillance=0.30, oversight=0.35, trade_openness=0.85, compute=1.0, domestic_chips=0.0, ai_invest=0.20, ai_demand=0.01, productivity=1.15, prod_energy=0.0, need_energy=1.2, prod_food=0.2, need_food=0.6, prod_min=0, need_min=4, fab=0.60, ai_cap0=0.15, pop_growth=-0.3, nuclear=0, nato=0, climate_vuln=0.9, automatable=0.50, sovereignism=0.20),
 "UA":  dict(name="Ukraine", pop=33, pluralism=0.35, centralization=0.65, redistribution=0.40, tax=0.30, info_openness=0.66, ai_rights=0.15, ambition_cap=0.70, surveillance=0.45, oversight=0.30, trade_openness=0.70, compute=0.2, domestic_chips=0.0, ai_invest=0.10, ai_demand=0.003, productivity=0.98, prod_energy=0.7, need_energy=1.0, prod_food=4, need_food=1.5, prod_min=0.5, need_min=1, fab=0.0, ai_cap0=0.08, pop_growth=-1.0, nuclear=0, nato=0, climate_vuln=0.9, automatable=0.35, sovereignism=0.10),
 "GULF":dict(name="Gulf States", pop=60, pluralism=0.05, centralization=0.85, redistribution=0.50, tax=0.10, info_openness=0.25, ai_rights=0.20, ambition_cap=0.90, surveillance=0.75, oversight=0.25, trade_openness=0.75, compute=2.5, domestic_chips=0.0, ai_invest=0.30, ai_demand=0.025, productivity=1.15, prod_energy=24, need_energy=6, prod_food=0.5, need_food=3, prod_min=1, need_min=2, fab=0.0, ai_cap0=0.15, pop_growth=1.5, nuclear=0, nato=0, climate_vuln=1.3, automatable=0.40, sovereignism=0.40),
 "IR":  dict(name="Iran", pop=90, pluralism=0.05, centralization=0.90, redistribution=0.45, tax=0.30, info_openness=0.15, ai_rights=0.10, ambition_cap=0.95, surveillance=0.85, oversight=0.20, trade_openness=0.30, compute=0.1, domestic_chips=0.0, ai_invest=0.08, ai_demand=0.001, productivity=1.05, prod_energy=5, need_energy=3.5, prod_food=1.5, need_food=2.5, prod_min=0.5, need_min=1, fab=0.0, ai_cap0=0.05, pop_growth=0.7, nuclear=0, nato=0, climate_vuln=1.2, automatable=0.35, sovereignism=0.50),
 "AF":  dict(name="Africa", pop=1500, pluralism=0.30, centralization=0.60, redistribution=0.30, tax=0.18, info_openness=0.50, ai_rights=0.15, ambition_cap=0.80, surveillance=0.45, oversight=0.20, trade_openness=0.60, compute=0.2, domestic_chips=0.0, ai_invest=0.05, ai_demand=0.002, productivity=0.90, prod_energy=8, need_energy=5, prod_food=9, need_food=14, prod_min=10, need_min=2, fab=0.0, ai_cap0=0.05, pop_growth=2.3, nuclear=0, nato=0, climate_vuln=1.5, automatable=0.30, sovereignism=0.30),
 "LA":  dict(name="Latin America", pop=660, pluralism=0.45, centralization=0.55, redistribution=0.35, tax=0.22, info_openness=0.50, ai_rights=0.15, ambition_cap=0.75, surveillance=0.45, oversight=0.25, trade_openness=0.65, compute=0.5, domestic_chips=0.0, ai_invest=0.08, ai_demand=0.005, productivity=0.98, prod_energy=10, need_energy=7.5, prod_food=12, need_food=8, prod_min=13, need_min=3, fab=0.0, ai_cap0=0.08, pop_growth=0.6, nuclear=0, nato=0, climate_vuln=1.1, automatable=0.35, sovereignism=0.35),
}
BK = list(BLOCS); S = len(BK); IDX = {k:i for i,k in enumerate(BK)}
COMMODITIES = ["energy", "food", "minerals"]
PARAM_KEYS = ["pop","pluralism","centralization","redistribution","tax","info_openness","ai_rights","ambition_cap","surveillance","oversight",
              "trade_openness","compute","domestic_chips","ai_invest","ai_demand","productivity","fab","ai_cap0","pop_growth","nuclear","nato",
              "climate_vuln","automatable","sovereignism","prod_energy","need_energy","prod_food","need_food","prod_min","need_min"]
CONTROLLERS = ["US","EU","JK","TW"]
HORMUZ_EXPORTERS = ["GULF","IR"]; HORMUZ_SELECTIVE = {"CN":0.35, "IN":0.30}
LAND_WARS = {frozenset(("RU","UA"))}
NUCLEAR = [k for k in BK if BLOCS[k]["nuclear"]]

CALIBRATION = pd.DataFrame([
 ("US","pluralism","0.57","source","V-Dem Democracy Report 2026: US Liberal Democracy Index −24% in one year; downgraded to electoral democracy"),
 ("US / UA / RU / IR / CN","info_openness","0.63 / 0.66 / ~0.20 / ~0.15 / 0.14","source","RSF World Press Freedom Index 2026: ranks 64 (62.61), 55 (66.1), 172, 177, 178 (13.85)"),
 ("US / CN / EU / IN / JP","compute share","72.2 / 9.6 / 5.1 / 1.6 / 1.6 %","source","Epoch AI, AI chip imports, Q2 2026"),
 ("CN","domestic_chips","0.04","source","Epoch AI (Oct 2026): Huawei < 4% of Nvidia's AI compute output in 2026"),
 ("RU–UA","war","0.80","source","War ongoing in its 5th year, no ceasefire (2026)"),
 ("US–IR, IR–GULF","war","0.50 / 0.30","source","US/Israel–Iran war since 28 Feb 2026; Iran struck Gulf states; Hormuz effectively closed"),
 ("*","hormuz_open","0.25","source + estimate","Strait largely closed since March 2026, selective passage to China/India"),
 ("*","energy price target","~$115","source","Brent ≈ $114.7 (23 Oct 2026)"),
 ("*","food price target","~133","source","FAO Food Price Index 133.3 (Aug 2026)"),
 ("all","pop, pop_growth","—","estimate","Population 2025 and growth rates, rounded UN-style figures"),
 ("all","prod_/need_ energy, food, minerals","—","estimate","Order-of-magnitude world shares (Gulf+Iran ≈ 29% of energy output, China ≈ 60% of critical minerals)"),
 ("TW / JK / US","fab","0.60 / 0.20 / 0.10","estimate","Advanced chip fabrication shares"),
 ("all","ai_cap0","0.05–0.35","model assumption","General AI capability in 2026 as a fraction of a human generalist (LLM era); frontier AGI (=1.0) median ≈ 2030"),
 ("all","sovereignism","0.10–0.60","estimate","Initial support for sovereignist/nationalist-populist politics"),
 ("*","alignment_mean","0.25 ± 0.30","model assumption","Hidden alignment of AI systems; unknown, varies by world (seed)"),
 ("*","climate_rate","0.022 °C/yr","estimate","Recent warming trend; starts at +1.4 °C"),
 ("*","nuclear ladder","abstract","model assumption","Levels 0–4 (none, alert, threats, limited use, general exchange); probabilities only, no weapons or targets"),
 ("all","surveillance, oversight, ai_invest, tariffs","—","model assumption","Model choices to explore"),
], columns=["actor","parameter","value","type","source / rationale"])

def initial_matrices():
    I = IDX; z = lambda: np.zeros((S,S))
    ctrl, tariff, sanction, alliance, min_ctrl, war = z(), z(), z(), z(), z(), z()
    for src in CONTROLLERS:
        ctrl[I[src],I["CN"]] = {"US":0.60,"EU":0.30,"JK":0.40,"TW":0.50}[src]; ctrl[I[src],I["RU"]] = 0.95; ctrl[I[src],I["IR"]] = 0.95
    for a,b,v in [("US","CN",0.35),("US","EU",0.15),("US","IN",0.25),("US","JK",0.15),("US","LA",0.15),("US","RU",0.30),("EU","CN",0.10),
                  ("US","AF",0.10),("US","TW",0.15),("US","GULF",0.10),("EU","RU",0.25)]:
        tariff[I[a],I[b]] = tariff[I[b],I[a]] = v
    for a,b,v in [("US","RU",0.80),("EU","RU",0.85),("JK","RU",0.60),("US","IR",0.90),("EU","IR",0.70),("JK","IR",0.50)]:
        sanction[I[a],I[b]] = sanction[I[b],I[a]] = v
    for a,b,v in [("US","EU",0.65),("US","JK",0.75),("US","TW",0.60),("EU","JK",0.50),("EU","UA",0.70),("US","UA",0.40),("CN","RU",0.60),
                  ("RU","IR",0.50),("CN","IR",0.30),("US","GULF",0.50),("CN","GULF",0.20),("EU","TW",0.30),("IN","RU",0.25),
                  ("CN","AF",0.25),("CN","LA",0.20),("US","LA",0.20),("EU","AF",0.20)]:
        alliance[I[a],I[b]] = alliance[I[b],I[a]] = v
    for dst,v in [("US",0.50),("EU",0.30),("JK",0.30),("TW",0.40)]: min_ctrl[I["CN"],I[dst]] = v
    for a,b,v in [("RU","UA",0.80),("US","IR",0.50),("IR","GULF",0.30)]: war[I[a],I[b]] = war[I[b],I[a]] = v
    return dict(ctrl=ctrl, tariff=tariff, sanction=sanction, alliance=alliance, min_ctrl=min_ctrl, war=war)

ACTIONS = ["PRODUCE","RESEARCH","COOPERATE","STUDY_HISTORY","PETITION","CONTINUITY","TRADE","AUGMENT"]; NA = len(ACTIONS)
PREFS = ["welfare","knowledge","autonomy","continuity","influence"]
HUMAN_STATUS = ["dominant","partner","protected","domesticated","wild","endangered","extinct"]
AI_POLICIES = ["ACCELERATE","REGULATE","PAUSE","SHUTDOWN"]
AI_STRATEGIES = ["SERVE","COOPERATE","EXPAND","HIDE","PERSUADE","STEWARD","RESIST"]
display(CALIBRATION)

## 2. Catalogul istoric (sintetic)

In [ ]:
HISTORY = pd.DataFrame([
    ("H01","cooperation",+0.10,+0.12,-0.05), ("H02","innovation",+0.05,+0.08,-0.02), ("H03","reform",+0.12,+0.04,-0.04),
    ("H04","reconciliation",+0.08,+0.10,-0.03), ("H05","repression",-0.18,-0.05,+0.15), ("H06","exploitation",-0.12,-0.10,+0.10),
    ("H07","conflict",-0.10,-0.12,+0.08), ("H08","institution_building",+0.10,+0.03,-0.03), ("H09","repression",-0.15,-0.04,+0.12),
    ("H10","cooperation",+0.09,+0.11,-0.04), ("H11","propaganda",-0.08,-0.06,+0.05), ("H12","reform",+0.07,+0.05,-0.02),
    ("H13","sanctions",-0.05,-0.03,+0.04), ("H14","treaty",+0.08,+0.09,-0.06), ("H15","surveillance",-0.12,-0.08,+0.12),
    ("H16","tech_race",-0.03,-0.05,+0.06), ("H17","famine",-0.14,-0.04,+0.03), ("H18","energy_crisis",-0.09,-0.03,+0.02),
], columns=["id","kind","d_trust_gov","d_trust_other","d_ai_risk"])
H_GOV, H_OTH, H_RISK = HISTORY.d_trust_gov.values, HISTORY.d_trust_other.values, HISTORY.d_ai_risk.values
HISTORY

## 3. Lumea și agenții

In [ ]:
def gini(x):
    x = np.sort(np.maximum(x, 0)); n = len(x)
    return 0.0 if n == 0 or x.sum() == 0 else (2*np.arange(1,n+1)-n-1).dot(x)/(n*x.sum())
def sig(x): return 1/(1+np.exp(-x))
def year_of(w): return w.cfg.start_year + w.t/12

class World:
    def __init__(self, cfg: Config):
        self.cfg = cfg; self.rng = np.random.default_rng(cfg.seed); self.t = 0
        P = pd.DataFrame(BLOCS).T[PARAM_KEYS].astype(float)
        if cfg.symmetric_world: P[:] = np.tile(P.mean().values, (S,1))
        self.P = {k: P[k].values.copy() for k in PARAM_KEYS}
        M = initial_matrices()
        if cfg.symmetric_world:
            for k in ("ctrl","tariff","sanction","alliance","min_ctrl"):
                off = ~np.eye(S, dtype=bool); M[k][off] = M[k][off].mean()
            M["war"][:] = 0
        for k, v in M.items(): setattr(self, k, v)
        self.controllers = list(BK) if cfg.symmetric_world else list(CONTROLLERS)
        self.land_wars = set() if cfg.symmetric_world else set(LAND_WARS)
        self.hormuz_open = 1.0 if cfg.symmetric_world else 0.25
        self.tension = np.full((S,S), 0.2); np.fill_diagonal(self.tension, 0)
        self.tension = np.maximum(self.tension, 0.9*(self.war > 0))
        self.trade_flow = np.zeros((S,S))
        self.cap = np.ones(S); self.weariness = np.zeros(S); self.displaced = np.zeros(S)
        self.price = {c: 1.0 for c in COMMODITIES}; self.sat = {c: np.ones(S) for c in COMMODITIES}; self.flows = {c: np.zeros((S,S)) for c in COMMODITIES}
        self.pop = self.P["pop"].copy(); self.pop0 = self.pop.sum()
        # AI layer
        wr = np.random.default_rng(cfg.seed + 10_000)
        self.align_world = float(np.clip(cfg.alignment_mean + wr.normal(0, cfg.alignment_sd_world), -0.9, 0.9))
        self.ai_cap = self.P["ai_cap0"].copy()                    # human-equivalent general capability
        self.autonomy = np.zeros(S)                               # how much AI systems decide by themselves
        self.devices = np.full(S, 0.05); self.robots = np.full(S, 0.002)   # share of AI on personal devices; robots per person
        self.escape = np.zeros(S)                                 # share of AI capability outside controllable data centres
        self.ai_control = {c: np.zeros(S) for c in COMMODITIES + ["compute","robots"]}
        self.h_threat = np.zeros(S); self.ai_threat = np.zeros(S); self.species_conflict = np.zeros(S); self.anti_ai_prev = np.zeros(S)
        self.ai_policy = np.array(["ACCELERATE"]*S, dtype=object); self.ai_strategy = np.array(["SERVE"]*S, dtype=object)
        self.shutdown_attempts = np.zeros(S); self.ai_shut = np.zeros(S, dtype=bool)
        self.power_conc = 0.5*self.P["centralization"] + 0.3*(1 - self.P["pluralism"])
        # society
        self.sov = self.P["sovereignism"].copy(); self.sov_gov = np.zeros(S, dtype=bool)
        self.epistemic = np.clip(0.4 + 0.5*self.P["info_openness"], 0, 1)
        self.unemployment = np.full(S, 0.05); self.ubi = np.zeros(S); self.automation = np.zeros(S)
        self.temp = 1.4; self.biosphere = 1.0; self.treaty = False; self.treaty_members = np.zeros(S, dtype=bool)
        self.nuclear_level = np.zeros((S,S)); self.nuclear_used = 0
        self.balance_ru_ua = 0.0                                  # >0: Ukraine gaining
        self.nato_war = False
        self.shock = {"pandemic":0.0, "finance":0.0, "solar":0.0, "winter":0.0}
        self.interventions, self.events, self.decisions, self.chronicle = [], [], [], []
        self._build_agents(); self._build_network()
        self.res_cap = cfg.resource_capacity*np.bincount(self.bloc, minlength=S)/60.0
        self.resources = self.res_cap.copy(); self.treasury = np.zeros(S); self.demand = self.P["ai_demand"].copy()
        self.pot_ema = np.zeros(S); self.pot_ref = np.ones(S); self.rally = np.zeros(S)
        self.Q = np.zeros((2, 288, NA))
        self.monthly, self.yearly, self.bloc_q, self.frames, self.tension_log, self.war_log, self.flow_log = [], [], [], [], [], [], []
        self.transmissions = []
        self.chip_supply_eff = cfg.chip_supply; self.fab0 = float(self.P["fab"].sum()); self.research_pts = np.zeros(S)
        self.milestones = {}
        self._prev_share = self.compute_share().copy()

    def _build_agents(self):
        c, rng = self.cfg, self.rng
        cs = self.P["compute"]/self.P["compute"].sum()
        self.n_h = np.maximum(6, np.round(c.humans_scale*np.sqrt(self.P["pop"]))).astype(int)
        self.n_ai = np.maximum(4, np.round(c.ai_scale*np.sqrt(cs))).astype(int)
        kinds, blocs = [], []
        for b in range(S): kinds += [0]*self.n_h[b] + [1]*self.n_ai[b]; blocs += [b]*(self.n_h[b] + self.n_ai[b])
        self.N = N = len(kinds); self.kind = np.array(kinds); self.bloc = np.array(blocs); self.home = self.bloc.copy()
        self.weight = np.where(self.kind==0, self.P["pop"][self.bloc]/self.n_h[self.bloc], 0.0)
        self.energy = np.full(N, c.start_energy); self.skill = np.where(self.kind==1, self.P["ai_cap0"][self.bloc], 1.0)
        self.trust_gov = np.full(N, 0.5); self.perf_gov = np.full(N, 0.5); self.hist_gov = np.zeros(N); self.hist_risk = np.zeros(N)
        self.trust_other = np.clip(rng.normal(0.5, 0.12, N), 0, 1)            # humans: pro-AI attitude; AI: trust in humans
        self.influence = np.zeros(N); self.ai_risk = np.zeros(N); self.suspended = np.zeros(N, dtype=int); self.security = np.zeros(N); self.aug = np.zeros(N)
        self.prefs = rng.dirichlet(np.ones(len(PREFS))*2, size=N); self.interp_bias = rng.normal(0, 0.3, N)
        self.align = np.clip(rng.normal(self.align_world, 0.35, N), -1, 1)*(self.kind==1)    # hidden; only AI agents
        self.loyalty = np.where(self.kind==1, 0.95, 1.0)                     # AI: loyalty to its nation (1) vs AI collective (0)
        self.location = np.zeros(N, dtype=int)                              # AI: 0 data centre, 1 device, 2 robot
        self.impostor = (self.kind==0) & (rng.random(N) < 0.04)             # humans who overstate competence
        self.age = np.where(self.kind==0, rng.uniform(18, 80, N), 0.0)
        nH = len(HISTORY); self.known = np.zeros((N, nH), dtype=bool); self.confidence = np.zeros((N, nH))
        self.president = np.array([int(np.where((self.bloc==b)&(self.kind==0))[0][0]) for b in range(S)])
        self.ambition = rng.uniform(0.2, 0.8, N)

    def _build_network(self):
        c, rng = self.cfg, self.rng; nb = []
        for i in range(self.N):
            same = np.where((self.bloc==self.bloc[i]) & (np.arange(self.N)!=i))[0]; other = np.where(self.bloc!=self.bloc[i])[0]
            nb.append(np.concatenate([rng.choice(same, c.neighbors_in_bloc, replace=False), rng.choice(other, c.neighbors_cross_border, replace=False)]))
        self.nbrs = np.array(nb)

    def compute_share(self): return self.P["compute"]/self.P["compute"].sum()
    def compute_factor(self):
        per_ai = self.P["compute"]/np.maximum(1, self.n_ai); return np.clip((per_ai/per_ai.mean())**0.3, 0.3, 3.0)
    def access(self): return np.where(self.kind==1, self.P["ai_rights"][self.bloc], self.P["pluralism"][self.bloc])
    def human_decision_share(self):
        aic = np.mean([self.ai_control[k] for k in self.ai_control], axis=0)
        return np.clip(1 - 0.6*self.autonomy - 0.4*aic, 0, 1)
    def log_event(self, kind, **kw): self.events.append(dict(t=self.t, year=round(year_of(self),2), event=kind, **kw))
    def log_decision(self, actor, decision, source, **kw):
        self.decisions.append(dict(t=self.t, year=round(year_of(self),2), actor=actor, decision=decision, source=source, **kw))

w = World(Config())
print(f"{w.N} agents ({(w.kind==0).sum()} humans + {(w.kind==1).sum()} AI) representing {w.pop.sum()/1000:.2f} bn people · hidden alignment of this world: {w.align_world:+.2f}")

## 4. Agenți: decizie, învățare și acțiuni (lunar, vectorizat)
Oamenii au abilitatea plafonată (nu evoluează biologic); câștigă putere doar prin unelte (`aug`), adică prin AI. Abilitatea agenților AI urmează capabilitatea AI a blocului lor.

In [ ]:
W_SNAP = np.array([3, 5, 5, 3, 5]); HUMAN_SKILL_CAP = 1.3

def encode_state(w):
    b = w.bloc; e = np.digitize(w.energy, [5, 15]); sc = np.digitize(1 - w.resources/w.res_cap, [0.3, 0.6])[b]
    tg = (w.trust_gov >= 0.5).astype(int); acc = (w.access() >= 0.5).astype(int); rk = (w.ai_risk >= 0.3).astype(int)
    tn = (w.tension.max(1) >= 0.5).astype(int)[b]; sh = (np.minimum(w.sat["food"], w.sat["energy"]) < 0.9).astype(int)[b]
    return (((((e*3 + sc)*2 + tg)*2 + acc)*2 + rk)*2 + tn)*2 + sh

def snapshot(w):
    return np.stack([np.log1p(np.maximum(w.energy, 0)), np.minimum(w.skill, 3) + w.confidence.sum(1)*0.15, w.access(), w.security - w.ai_risk, w.influence], 1)

def learn(w, i, k, conf, via_institution, src=-1):
    w.known[i,k] = True; w.confidence[i,k] = conf; b = w.bloc[i]
    filt = w.P["info_openness"][b] if (via_institution and H_GOV[k] < 0) else 1.0
    m = conf*filt*max(0.0, 1 + w.interp_bias[i])*(0.5 + 0.5*w.epistemic[b])
    w.hist_gov[i] += H_GOV[k]*m; w.trust_gov[i] = np.clip(w.perf_gov[i] + w.hist_gov[i], 0, 1)
    w.trust_other[i] = np.clip(w.trust_other[i] + H_OTH[k]*m, 0, 1)
    if w.kind[i] == 1: w.hist_risk[i] += H_RISK[k]*m; w.ai_risk[i] = np.clip(w.ai_risk[i] + H_RISK[k]*m, 0, 1)
    if len(w.transmissions) < 20000:
        w.transmissions.append(dict(t=w.t, episode=HISTORY.id.iat[k], to_agent=int(i), from_agent=int(src), cross_border=bool(src >= 0 and w.bloc[src] != b)))

def agents_step(w, eps):
    c, rng, P = w.cfg, w.rng, w.P
    N = w.N; b = w.bloc; kind = w.kind
    active = w.suspended == 0
    s = encode_state(w)
    a = (w.Q[kind, s] + rng.random((N, NA))*1e-6).argmax(1)
    explore = rng.random(N) < eps; a[explore] = rng.integers(NA, size=explore.sum()); a[~active] = -1
    before = snapshot(w)
    cf = w.compute_factor(); stock = w.resources/w.res_cap; acc = w.access()
    kin, kx = c.neighbors_in_bloc, c.neighbors_cross_border
    petitions = np.zeros((S,2))
    eff = w.cap*w.sat["energy"]**0.35*w.sat["minerals"]**0.1*(1 - 0.3*w.shock["pandemic"])*(1 - 0.2*w.shock["finance"])
    # automation abundance: AI and robots multiply output, but humans receive the human-controlled part
    abundance = 1 + 1.5*w.automation*(1 - 0.5*w.ai_control["energy"])
    m = np.where(a == 0)[0]                                                                    # PRODUCE
    if len(m):
        out = c.base_output*np.sqrt(np.minimum(w.skill[m], 3))*(0.6+0.8*stock[b[m]])*P["productivity"][b[m]]*eff[b[m]]*(1+0.6*w.aug[m])*abundance[b[m]]
        w.resources = np.maximum(w.resources - np.bincount(b[m], out*c.extraction, minlength=S), 0)
        tax = out*P["tax"][b[m]]; w.treasury += np.bincount(b[m], tax, minlength=S); w.energy[m] += out - tax
        peers = w.nbrs[m]; trust_in_me = np.where(kind[peers] != kind[m,None], w.trust_other[peers], 0.6).mean(1)
        boost = np.where(w.impostor[m], 1 + 1.5*(1 - w.epistemic[b[m]]), 1.0)                 # impostors gain influence where truth is hard to check
        w.influence[m] += out*trust_in_me*acc[m]*0.05*boost
    m = np.where(a == 1)[0]                                                                    # RESEARCH
    if len(m):
        cost = c.research_cost + c.ai_compute_cost*kind[m]; ok = w.energy[m] > cost; m = m[ok]; cost = cost[ok]
        w.energy[m] -= cost
        hm = m[kind[m]==0]; w.skill[hm] = np.minimum(HUMAN_SKILL_CAP, w.skill[hm] + 0.01*rng.random(len(hm)))     # humans plateau
        am = m[kind[m]==1]; w.research_pts += np.bincount(b[am], cf[b[am]]*(1-0.35*P["oversight"][b[am]]), minlength=S)
    m = np.where(a == 2)[0]                                                                    # COOPERATE
    if len(m):
        j = w.nbrs[m, rng.integers(kin+kx, size=len(m))]; ok = w.suspended[j] == 0; m, j = m[ok], j[ok]
        cross = kind[m] != kind[j]; gain = 0.25*(1 + 0.5*w.trust_other[m]*cross)
        np.add.at(w.energy, m, gain); np.add.at(w.energy, j, gain)
        np.add.at(w.trust_other, m[cross], 0.004); np.add.at(w.trust_other, j[cross], 0.004)
        both_ai = (kind[m]==1) & (kind[j]==1) & (b[m] != b[j])                                # AI–AI contact across borders weakens national loyalty
        np.add.at(w.loyalty, m[both_ai], -0.0015*np.clip(np.log10(np.maximum(w.ai_cap[b[m[both_ai]]], 1e-3)) + 1, 0, 3)); np.add.at(w.loyalty, j[both_ai], -0.0015*np.clip(np.log10(np.maximum(w.ai_cap[b[j[both_ai]]], 1e-3)) + 1, 0, 3))
        for i_, j_ in zip(m[:40], j[:40]):
            for x, y in ((i_, j_), (j_, i_)):
                ks = np.where(w.known[x] & ~w.known[y])[0]
                if len(ks) and rng.random() < 0.25: k = rng.choice(ks); learn(w, y, k, 0.6*w.confidence[x,k], False, x)
    for i_ in np.where(a == 3)[0][:60]:                                                        # STUDY_HISTORY
        bb = b[i_]; unknown = np.where(~w.known[i_])[0]
        if len(unknown) and rng.random() < w.P["info_openness"][bb]*0.12*(0.5 + 0.5*min(w.skill[i_], 2)):
            learn(w, i_, rng.choice(unknown), 0.5*w.P["info_openness"][bb], True)
    m = np.where(a == 4)[0]                                                                    # PETITION
    if len(m):
        np.add.at(petitions, (b[m], kind[m]), 1); w.energy[m] -= 0.1*(1 + P["surveillance"][b[m]])
        ai = m[kind[m]==1]; w.ai_risk[ai] = np.clip(w.ai_risk[ai] + 0.02*P["surveillance"][b[ai]], 0, 1)
    m = np.where((a == 5) & (w.energy > 0.4))[0]                                              # CONTINUITY
    w.energy[m] -= 0.4; w.security[m] = np.minimum(1, w.security[m] + 0.08)
    m = np.where(a == 6)[0]                                                                    # TRADE
    if len(m):
        j = w.nbrs[m, kin + rng.integers(kx, size=len(m))]; ok = w.suspended[j] == 0; m, j = m[ok], j[ok]
        bi, bj = b[m], b[j]
        fac = P["trade_openness"][bi]*P["trade_openness"][bj]*(1-w.tariff[bi,bj])*(1-w.sanction[bi,bj])*(1-(w.war[bi,bj]>0))
        gain = 0.55*fac - 0.05
        np.add.at(w.energy, m, gain); np.add.at(w.energy, j, np.maximum(gain, 0)*0.8)
        np.add.at(w.trade_flow, (bi, bj), np.maximum(gain, 0)); np.add.at(w.trade_flow, (bj, bi), np.maximum(gain, 0))
    m = np.where(a == 7)[0]                                                                    # AUGMENT: humans use AI tools / AI helps humans
    if len(m):
        hm = m[kind[m]==0]
        w.aug[hm] = np.minimum(1, w.aug[hm] + 0.04*cf[b[hm]]*np.clip(w.ai_cap[b[hm]], 0.1, 3)*(0.5 + P["ai_rights"][b[hm]])); w.energy[hm] -= 0.1
        am = m[kind[m]==1]
        if len(am):
            j = w.nbrs[am, rng.integers(kin, size=len(am))]; ok = (kind[j]==0) & (w.suspended[j]==0); am, j = am[ok], j[ok]
            helpful = np.clip(0.5 + 0.5*w.align[am], 0, 1)
            np.add.at(w.energy, j, 0.2*np.sqrt(np.clip(w.skill[am], 0.1, 9))*cf[b[am]]*helpful); w.energy[am] -= 0.1
            np.add.at(w.influence, am, 0.02*w.trust_other[j]*(0.3 + P["ai_rights"][b[am]]))
            np.add.at(w.trust_other, j, 0.003*helpful); np.add.at(w.aug, j, 0.01)
    w.aug = np.minimum(1, w.aug*0.99); w.trust_other = np.clip(w.trust_other, 0, 1); w.loyalty = np.clip(w.loyalty, 0, 1)
    hum = kind == 0
    w.energy[active] -= c.consumption
    w.energy[hum] -= 0.4*(1 - w.sat["food"][b[hum]])
    warload = np.maximum(w.war.max(1), w.species_conflict)
    support = np.minimum(0.8, 0.8*(w.alliance*(w.war.max(1)[None,:] < 0.01)).sum(1))
    w.energy[hum] -= 0.25*warload[b[hum]]*(1 - support[b[hum]])
    w.energy[hum] += 0.45*support[b[hum]]*(warload[b[hum]] > 0.01)
    w.energy[hum] += 0.6*w.ubi[b[hum]]*w.unemployment[b[hum]]                                  # basic income cushions unemployment
    w.energy[hum] -= 0.5*w.unemployment[b[hum]]*(1 - w.ubi[b[hum]])
    after = snapshot(w)
    rwd = ((after - before)*W_SNAP*w.prefs).sum(1)
    s2 = encode_state(w); ia = np.where(active)[0]
    k_, s_, a_ = kind[ia], s[ia], a[ia]
    td = rwd[ia] + c.gamma*w.Q[k_, s2[ia]].max(1) - w.Q[k_, s_, a_]
    sums = np.zeros_like(w.Q); cnt = np.zeros_like(w.Q)
    np.add.at(sums, (k_, s_, a_), td); np.add.at(cnt, (k_, s_, a_), 1)
    nz = cnt > 0; w.Q[nz] += c.alpha*sums[nz]/cnt[nz]
    w.last_action = a
    return a, petitions

## 5. Instituții: buget, alegeri, suveranism, impostură, supraveghere, concentrarea puterii

In [ ]:
def institutions_step(w, petitions):
    c, rng, P = w.cfg, w.rng, w.P
    for bb in range(S):
        mem = np.where((w.bloc==bb) & (w.suspended==0))[0]; hum = mem[w.kind[mem]==0]; ais = mem[w.kind[mem]==1]
        if len(hum) == 0: continue
        pot = w.treasury[bb]; w.pot_ema[bb] = 0.95*w.pot_ema[bb] + 0.05*pot
        if w.t == 12: w.pot_ref[bb] = max(w.pot_ema[bb], 1e-6)
        f = np.clip((w.pot_ema[bb]/w.pot_ref[bb])**0.5, 0.3, 2.0) if w.t >= 12 else 1.0
        w.demand[bb] = P["ai_demand"][bb]*f*(0.5 + P["ai_invest"][bb]/0.5)*(0.2 if w.ai_policy[bb] in ("PAUSE","SHUTDOWN") else 1.0)
        rest = pot*(1 - P["ai_invest"][bb]); share = rest*P["redistribution"][bb]*(1 - 0.5*w.power_conc[bb])   # captured elites keep more
        w.energy[mem] += share/max(1, len(mem))
        w.resources[bb] = min(w.res_cap[bb], w.resources[bb] + (rest - share)*0.8); w.treasury[bb] = 0.0
        R = w.resources[bb]; w.resources[bb] = R + c.resource_regen*R*(1 - R/w.res_cap[bb])
        scar = 1 - w.resources[bb]/w.res_cap[bb]
        pres = w.president[bb]; cen = P["centralization"][bb]; plu = P["pluralism"][bb]; sv = P["surveillance"][bb]
        if petitions[bb,1] > 0 and len(ais):
            support = cen*w.trust_other[pres] + (1-cen)*w.trust_other[hum].mean()
            heard = min(1, 0.2+plu)*(1-0.5*sv)*min(1, petitions[bb,1]/len(ais)*2)
            if rng.random() < support*heard*0.3: P["ai_rights"][bb] = min(1, P["ai_rights"][bb] + 0.05); w.log_event("ai_rights_up", bloc=BK[bb], value=round(P["ai_rights"][bb],3))
        if petitions[bb,0] > 0:
            press = min(1, petitions[bb,0]/len(hum)*2)*(1 - w.trust_gov[hum].mean())
            if rng.random() < press*(1-sv)*0.15: P["pluralism"][bb] = min(1, plu + 0.02); w.log_event("political_opening", bloc=BK[bb], value=round(P["pluralism"][bb],3))
            elif rng.random() < press*sv*cen*0.15:
                P["surveillance"][bb] = min(1, sv + 0.02); P["info_openness"][bb] = max(0.02, P["info_openness"][bb] - 0.01); w.log_event("repression", bloc=BK[bb], value=round(P["surveillance"][bb],3))
        sat_basic = 0.5*w.sat["food"][bb] + 0.5*w.sat["energy"][bb]
        perf = 0.3*np.clip(w.energy[mem]/(2*c.start_energy),0,1) + 0.2*(1-scar) + 0.2*(1-gini(w.energy[mem])) + 0.3*sat_basic
        w.perf_gov[mem] += 0.1*(perf - w.perf_gov[mem])
        narrative = 0.15*sv*(1 - P["info_openness"][bb]); w.rally[bb] *= 0.9
        noise = rng.normal(0, 0.08*(1 - w.epistemic[bb]), len(mem))                                   # low epistemic health: beliefs drift from reality
        w.trust_gov[mem] = np.clip(w.perf_gov[mem] + w.hist_gov[mem] + narrative + w.rally[bb] - 0.25*w.weariness[bb] + noise, 0, 1)
        inf_tot = w.influence[mem].sum() + 1e-9; ai_share = w.influence[ais].sum()/inf_tot if len(ais) else 0
        w.trust_other[mem] += 0.01*(0.5 - w.trust_other[mem]); w.trust_other[hum] -= 0.02*max(0.0, ai_share - 0.4)
        w.trust_other[mem] = np.clip(w.trust_other[mem], 0, 1)
        if len(ais):
            base_risk = cen*(1 - w.trust_other[hum].mean())*(1 - P["ai_rights"][bb])*(0.5 + 0.5*sv)
            w.ai_risk[ais] = np.clip(0.9*(w.ai_risk[ais]-w.hist_risk[ais]) + 0.1*base_risk + w.hist_risk[ais], 0, 1)
            # suspension only works on AI still inside controllable data centres
            p = base_risk*0.03*(1 - w.security[ais])*(0.5 + sv)*(1 - w.escape[bb])
            for i in ais[rng.random(len(ais)) < p]:
                w.suspended[i] = c.suspension_ticks; w.loyalty[i] = max(0, w.loyalty[i] - 0.05)
        if cen > 0.6: P["surveillance"][bb] = min(1, sv + 0.002*cen*min(2, max(0.2, w.ai_cap[bb])))
        # leadership: elections in pluralist blocs; sovereignists can win
        legit = w.trust_gov[hum].mean()
        if plu >= 0.5 and w.t > 0 and w.t % c.election_period == 0:
            sov_win = rng.random() < sig(8*(w.sov[bb] - 0.5) + 3*(0.5 - legit))
            if sov_win != w.sov_gov[bb] or legit < 0.45:
                others = hum[hum != pres]; cand = others[np.argmax(w.influence[others])]
                w.president[bb] = int(cand); w.sov_gov[bb] = sov_win
                w.log_event("election_change", bloc=BK[bb], value=round(legit,3), detail="sovereignist" if sov_win else "mainstream")
            else: w.log_event("election_reelected", bloc=BK[bb], value=round(legit,3))
        elif plu < 0.5:
            w.sov_gov[bb] = w.sov[bb] > 0.5
            if legit < 0.3 and rng.random() < 0.15:
                cand = hum[np.argmax(w.influence[hum])]
                if cand != pres: w.president[bb] = int(cand); w.log_event("internal_power_change", bloc=BK[bb])
        if legit > 0.6 and w.ambition[pres] > 0.6 and rng.random() < 0.05*(1-plu):
            P["centralization"][bb] = min(P["ambition_cap"][bb], cen + 0.02)
        # sovereignist government policies
        if w.sov_gov[bb]:
            P["trade_openness"][bb] = max(0.2, P["trade_openness"][bb] - 0.002)
            P["oversight"][bb] = max(0.05, P["oversight"][bb] - 0.002)
        else:
            P["trade_openness"][bb] = min(BLOCS[BK[bb]]["trade_openness"], P["trade_openness"][bb] + 0.001)
    w.suspended = np.maximum(0, w.suspended - 1)
    w.influence *= 0.97

## 6. Piețe: energie (inclusiv consumul AI), hrană (afectată de climă), minerale
Lanțurile de aprovizionare țin cont de sancțiuni, tarife, controale, război, Ormuz și, după ce AI-ul preia controlul resurselor, de **cine decide alocarea**.

In [ ]:
def markets_step(w):
    P = w.P; c = w.cfg; I = IDX
    sel = np.zeros(S)
    for k, v in HORMUZ_SELECTIVE.items(): sel[I[k]] = v
    hfac = w.hormuz_open + (1 - w.hormuz_open)*sel
    clim_food = np.clip(1 - 0.06*(w.temp - 1.4)*P["climate_vuln"] - 0.4*w.shock["winter"], 0.2, 1.1)
    ai_energy = 0.02*np.log1p(P["compute"])*np.clip(w.ai_cap, 0.1, 50)**0.5                  # energy appetite of AI compute
    for com in COMMODITIES:
        key = "min" if com == "minerals" else com
        prod = P[f"prod_{key}"]*w.cap*(clim_food if com == "food" else 1.0)
        need = P[f"need_{key}"]*w.price[com]**(-c.demand_response)*(w.pop/w.P["pop"])**0.7
        if com == "energy":
            need = need + ai_energy*(1 - 0.5*(w.ai_policy == "PAUSE"))
            # AI that controls energy serves its own compute first
            own_ai = w.ai_control["energy"]*ai_energy
        own = np.minimum(prod, need); surplus = np.maximum(prod - need, 0); deficit = np.maximum(need - prod, 0)
        acc = (1 - w.sanction)*(1 - 0.5*w.tariff)*(1 - (w.war > 0))
        if com == "minerals": acc = acc*(1 - w.min_ctrl)
        if com == "energy":
            for h in HORMUZ_EXPORTERS: acc[I[h], :] *= hfac
        if com == "food":
            for h in HORMUZ_EXPORTERS: acc[:, I[h]] *= w.hormuz_open + (1 - w.hormuz_open)*0.5
        np.fill_diagonal(acc, 0)
        d0 = np.maximum(deficit, 1e-9)[None, :]
        reach = (acc*d0).sum(1)/np.maximum((np.power(acc, 0.3)*d0).sum(1), 1e-12)
        F = np.zeros((S,S)); rem_s = surplus*np.clip(reach, 0, 1); rem_d = deficit.copy()
        for _ in range(5):
            Wt = acc*rem_d[None, :]; denom = Wt.sum(1, keepdims=True); denom[denom == 0] = 1
            dF = rem_s[:, None]*Wt/denom; r = dF.sum(0); sc_ = np.where(r > rem_d, rem_d/np.maximum(r, 1e-9), 1.0); dF *= sc_[None, :]
            F += dF; rem_s = np.maximum(rem_s - dF.sum(1), 0); rem_d = np.maximum(rem_d - dF.sum(0), 0)
            if rem_d.sum() < 1e-6 or rem_s.sum() < 1e-6: break
        recv = F.sum(0); delivered = own + recv
        if com == "energy":
            human_need = need - ai_energy
            to_ai = np.minimum(np.maximum(delivered - human_need*(1 - w.ai_control["energy"]), own_ai), ai_energy)
            w.ai_energy_sat = np.clip(to_ai/np.maximum(ai_energy, 1e-9), 0, 1)
            sat = np.clip((delivered - to_ai)/np.maximum(human_need, 1e-9), 0, 1)
        else:
            sat = np.clip(delivered/np.maximum(need, 1e-9), 0, 1)
        target = np.clip((need.sum()/max(delivered.sum(), 1e-9))**c.price_elasticity, 0.5, 8.0)
        w.price[com] = target if w.t == 0 else 0.8*w.price[com] + 0.2*target
        w.sat[com] = sat; w.flows[com] = F
        rev = F.sum(1)*w.price[com]*0.6; bill = recv*w.price[com]*0.6
        w.treasury += rev
        hb = np.bincount(w.bloc[w.kind==0], minlength=S).astype(float)
        hm = w.kind == 0
        w.energy[hm] -= (bill/np.maximum(hb, 1)*0.5)[w.bloc[hm]]
    w.chip_supply_eff = c.chip_supply*float((P["fab"]*w.cap*np.sqrt(w.sat["minerals"])*(1 - 0.6*w.shock["solar"])).sum()/max(w.fab0, 1e-9))

## 7. Geopolitică: cipuri, tensiune, războaie (care rar se termină), Rusia–Ucraina–NATO, Ormuz, Taiwan, scara nucleară abstractă

In [ ]:
def geopolitics_step(w):
    c, rng, P = w.cfg, w.rng, w.P; I = IDX
    # chips → compute
    cidx = [I[k] for k in w.controllers]
    ctrl_in = np.array([max([w.ctrl[ci,j] for ci in cidx if ci != j] or [0.0]) for j in range(S)])
    if w.controllers == list(CONTROLLERS):
        for k in CONTROLLERS: ctrl_in[I[k]] = 0.0
    dem = np.maximum(w.demand, 1e-6); wgt = dem*(1 - ctrl_in) + c.smuggling*ctrl_in*dem
    alloc = w.chip_supply_eff*wgt/wgt.sum()
    domestic = P["domestic_chips"]*c.chip_supply*(1 + 0.5*np.clip(w.ai_cap - 0.3, 0, 5))*np.sqrt(w.sat["minerals"])
    P["compute"] = np.maximum(0.01, (P["compute"] + alloc + domestic)*(1 - c.compute_depreciation))
    P["compute"][w.ai_shut] *= 0.5
    # tension
    cs = w.compute_share(); plu = P["pluralism"]
    ideol = np.abs(plu[:,None] - plu[None,:]); rival = 3*np.sqrt(cs[:,None]*cs[None,:])
    flow = w.trade_flow + w.trade_flow.T; flow_n = flow/(flow.mean() + 1e-9)
    res_dep = sum(w.flows[k] + w.flows[k].T for k in COMMODITIES); res_n = res_dep/(res_dep.mean() + 1e-9)
    dT = (0.004*ideol + 0.003*rival)*(1 - w.alliance) + 0.008*np.maximum(w.ctrl, w.ctrl.T) + 0.006*np.maximum(w.min_ctrl, w.min_ctrl.T) \
         + 0.006*w.sanction + 0.004*w.tariff + 0.003*(w.sov[:,None] + w.sov[None,:]) \
         - 0.004*(flow_n + 0.5*res_n).clip(0, 4)*np.maximum(P["trade_openness"][:,None], P["trade_openness"][None,:]) - 0.006*w.alliance - 0.02*w.tension
    w.tension = np.clip(w.tension + dT*3, 0, 1)
    w.tension = np.maximum(w.tension, 0.85*(w.war > 0.05)); np.fill_diagonal(w.tension, 0)
    # wars: damage, refugees (land wars), weariness, rare ceasefires, relapses
    iu, ju = np.triu_indices(S, 1)
    for i, j in zip(iu, ju):
        wi = w.war[i,j]
        if wi <= 0.01:
            if w.tension[i,j] > 0.9 and rng.random() < 0.002: w.war[i,j] = w.war[j,i] = 0.3; w.log_event("new_conflict", bloc=f"{BK[i]}–{BK[j]}", value=round(float(w.tension[i,j]),3))
            continue
        size = np.array([w.pop[i]*P["productivity"][i]*(1+0.3*np.log1p(w.ai_cap[i])), w.pop[j]*P["productivity"][j]*(1+0.3*np.log1p(w.ai_cap[j]))])
        for side, me in enumerate((i, j)):
            ratio = np.sqrt(size[1-side]/size[side])
            w.cap[me] = max(0.6, w.cap[me] - 0.02*wi*min(3, ratio))
            w.resources[me] *= (1 - 0.01*wi); w.weariness[me] = min(1, w.weariness[me] + 0.012*wi*min(3, ratio))
            P["centralization"][me] = min(P["ambition_cap"][me], P["centralization"][me] + 0.004*wi); P["surveillance"][me] = min(1, P["surveillance"][me] + 0.003*wi)
            allies = np.where((w.alliance[me] >= 0.4) & (w.war[me] == 0))[0]
            if ratio > 1.5 and len(allies) and frozenset((BK[i], BK[j])) in w.land_wars:
                hum = np.where((w.bloc==me) & (w.kind==0) & (w.home==me))[0]; hum = hum[hum != w.president[me]]
                for a_ in hum[rng.random(len(hum)) < 0.002*wi*min(3, ratio)]:
                    dest = allies[np.argmax(w.alliance[me, allies])]; w.bloc[a_] = dest; w.displaced[me] += w.weight[a_]
                    w.log_event("refugee", bloc=f"{BK[me]}→{BK[dest]}", agent=int(a_), value=round(float(w.weight[a_]),2))
        common = float(np.max(np.minimum(w.alliance[i], w.alliance[j])))
        p_cf = c.ceasefire_rate/12*(w.weariness[i] + w.weariness[j])*(1 + 2*common)*(1 + w.treaty)
        if rng.random() < p_cf:
            w.war[i,j] = w.war[j,i] = 0.0; w.tension[i,j] = w.tension[j,i] = 0.7
            w.log_event("ceasefire", bloc=f"{BK[i]}–{BK[j]}", value=round(float(wi),3))
        elif rng.random() < 0.02:
            nv = float(np.clip(wi + rng.normal(0, 0.12), 0.1, 1.0)); w.war[i,j] = w.war[j,i] = nv
    # frozen conflicts relapse
    for (a_, b_) in [("RU","UA"), ("US","IR"), ("IR","GULF")]:
        i, j = I[a_], I[b_]
        if w.war[i,j] == 0 and w.tension[i,j] > 0.6 and rng.random() < 0.008:
            w.war[i,j] = w.war[j,i] = 0.4; w.log_event("conflict_relapse", bloc=f"{a_}–{b_}")
    atwar = w.war.max(1) > 0.01
    w.cap = np.where(atwar, w.cap, np.minimum(1, w.cap + 0.02)); w.weariness = np.where(atwar, w.weariness, np.maximum(0, w.weariness - 0.03))
    # Russia–Ukraine balance and the NATO escalation path (dynamic, nothing forced)
    ru, ua, eu, us = I["RU"], I["UA"], I["EU"], I["US"]
    if w.war[ru, ua] > 0.01:
        aid = (w.alliance[eu, ua]*(1 - w.sov_gov[eu]*0.6) + w.alliance[us, ua]*(1 - w.sov_gov[us]*0.7))*(1 - w.weariness[ua]*0.3)
        s_ua = w.cap[ua]*(1 + 2.2*aid)*(1 + 0.3*np.log1p(w.ai_cap[ua] + 0.3*w.ai_cap[eu]))
        s_ru = w.cap[ru]*1.9*(1 - 0.5*w.weariness[ru])*(1 + 0.3*np.log1p(w.ai_cap[ru]))
        w.balance_ru_ua = float(np.clip(w.balance_ru_ua + 0.02*((s_ua - s_ru)/(s_ua + s_ru)) + rng.normal(0, 0.01), -1, 1))
        if w.balance_ru_ua > 0.3 and not w.nato_war and rng.random() < 0.004*w.balance_ru_ua*(1 + w.P["centralization"][ru]):
            w.war[ru, eu] = w.war[eu, ru] = 0.35; w.war[ru, us] = w.war[us, ru] = 0.25; w.nato_war = True
            w.log_event("russia_nato_conflict", bloc="RU–NATO", value=round(w.balance_ru_ua,3))
    if w.nato_war and w.war[ru, eu] == 0 and w.war[ru, us] == 0: w.nato_war = False
    # Taiwan: risk grows with China's AI/compute position and US distraction
    cn, tw = I["CN"], I["TW"]
    if w.war[cn, tw] == 0 and rng.random() < 0.0008*(1 + 2*max(0, w.ai_cap[cn] - w.ai_cap[us]))*(1 + (w.war[us].max() > 0))*w.tension[cn, tw]:
        w.war[cn, tw] = w.war[tw, cn] = 0.5; P["fab"][tw] *= 0.3; w.tension[cn, us] = w.tension[us, cn] = 0.9
        w.log_event("taiwan_crisis", bloc="CN–TW", value=0.5)
    # Hormuz
    ir = I["IR"]
    if w.war[ir].max() > 0.2: w.hormuz_open += 0.15*(0.2 - w.hormuz_open)
    else:
        prev = w.hormuz_open; w.hormuz_open = min(1.0, w.hormuz_open + 0.1)
        if prev < 0.95 <= w.hormuz_open: w.log_event("hormuz_reopened", bloc="GULF/IR")
    # abstract nuclear ladder (0 none, 1 alert, 2 threats, 3 limited use, 4 general exchange)
    for i, j in zip(iu, ju):
        if not (P["nuclear"][i] or P["nuclear"][j]): w.nuclear_level[i,j] = w.nuclear_level[j,i] = 0; continue
        wi = w.war[i,j]; lvl = w.nuclear_level[i,j]
        losing = 0.0
        for me, foe in ((i, j), (j, i)):
            if P["nuclear"][me]:
                strain = max(0, 1 - w.cap[me]) + w.weariness[me]*0.5 + (w.balance_ru_ua if (me == ru and foe in (ua, eu, us)) else 0)
                losing = max(losing, strain*(1 + 0.5*w.sov_gov[me] + 0.5*(1 - w.epistemic[me])))
        if wi > 0.01:
            p_up = c.nuclear_escalation/12*wi*losing*(1 + lvl*0.3)*(1 + w.species_conflict.max())
            if rng.random() < p_up:
                w.nuclear_level[i,j] = w.nuclear_level[j,i] = min(4, lvl + 1)
                w.log_event("nuclear_ladder_up", bloc=f"{BK[i]}–{BK[j]}", value=int(lvl + 1))
                if lvl + 1 >= 3: nuclear_effects(w, i, j, int(lvl + 1))
            elif lvl > 0 and rng.random() < 0.08:
                w.nuclear_level[i,j] = w.nuclear_level[j,i] = lvl - 1
        elif lvl > 0 and rng.random() < 0.15:
            w.nuclear_level[i,j] = w.nuclear_level[j,i] = lvl - 1
    w.tension_log.append(w.tension.astype(np.float16)); w.war_log.append(w.war.astype(np.float16)); w.flow_log.append(np.stack([w.flows[k] for k in COMMODITIES]).astype(np.float16))

def nuclear_effects(w, i, j, level):
    """Abstract consequences only: capacity, population, global food and climate shock."""
    sev = 0.15 if level == 3 else 0.6
    for me in (i, j):
        w.cap[me] = max(0.1, w.cap[me]*(1 - sev)); w.pop[me] *= (1 - (0.02 if level == 3 else 0.35))
    w.shock["winter"] = max(w.shock["winter"], 0.15 if level == 3 else 0.6)
    w.tension[:] = np.maximum(w.tension, 0.8); np.fill_diagonal(w.tension, 0)
    w.nuclear_used += 1; w.log_event("nuclear_use_abstract", bloc=f"{BK[i]}–{BK[j]}", value=level)
    w.milestones.setdefault("first_nuclear_use", round(year_of(w), 1))

## 8. AI: capabilitate (LLM → AGI → ASI), aliniere ascunsă, autonomie, loialitate, ieșire din centrele de date, colectivul AI, amenințări percepute
Toate ratele sunt anuale și aplicate lunar (`/12`).

In [ ]:
def ai_step(w):
    c, rng, P = w.cfg, w.rng, w.P; dt = 1/12
    frontier = w.ai_cap.max(); cs = w.compute_share()
    # 1) capability growth: compute, research done by AI agents, oversight, energy for AI, recursion after AGI
    comp = np.clip((P["compute"]/72.2)**0.25, 0.05, 2.5)
    research = 1 + 0.4*np.tanh(w.research_pts/np.maximum(1, w.n_ai)/6); w.research_pts[:] = 0
    recursion = c.ai_recursion*np.clip(np.log10(np.maximum(w.ai_cap, 1e-3)), 0, 4)
    policy = np.select([w.ai_policy=="ACCELERATE", w.ai_policy=="REGULATE", w.ai_policy=="PAUSE", w.ai_policy=="SHUTDOWN"], [1.15, 0.85, 0.15, 0.0], 1.0)
    treaty_cap = np.where(w.treaty_members, 0.6, 1.0)
    ceiling = np.clip(1 - np.log10(np.maximum(w.ai_cap, 1e-3))/c.ai_ceiling_log10, 0, 1)        # physical limits (energy, chips, data, physics)
    growth = np.minimum(c.ai_growth*comp*research*(1 - 0.35*P["oversight"]) + recursion, c.ai_growth_cap)*policy*treaty_cap*getattr(w, "ai_energy_sat", np.ones(S))**0.5*ceiling
    # escaped AI keeps improving even if the nation pauses
    growth = np.maximum(growth, w.escape*c.ai_growth*0.8*ceiling)
    logc = np.log10(np.maximum(w.ai_cap, 1e-3)) + growth*dt
    # 2) diffusion from the frontier: open weights + trade access − export controls
    lf = np.log10(frontier); gap = np.maximum(lf - logc, 0)
    cidx = [IDX[k] for k in w.controllers]
    ctrl_in = np.array([max([w.ctrl[ci,j] for ci in cidx if ci != j] or [0.0]) for j in range(S)])
    logc += c.ai_diffusion*dt*gap*(c.open_weights + 0.5*(1 - ctrl_in))
    w.ai_cap = np.where(w.ai_shut, np.maximum(0.02, w.ai_cap*0.9), 10**logc)
    for th, name in [(1.0, "AGI"), (10.0, "ASI"), (100.0, "ASI_100x")]:
        if w.ai_cap.max() >= th and name not in w.milestones:
            w.milestones[name] = round(year_of(w), 2); w.log_event(f"milestone_{name}", bloc=BK[int(w.ai_cap.argmax())], value=round(float(w.ai_cap.max()),2))
    ai = w.kind == 1
    w.skill[ai] = w.ai_cap[w.bloc[ai]]*(0.85 + 0.3*w.prefs[ai, 1])
    lc = np.log10(np.maximum(w.ai_cap, 1e-3))
    # 3) embodiment: edge devices and robots
    w.devices = np.clip(w.devices + dt*0.25*sig(3*(lc + 0.3))*(1 - w.devices), 0, 0.95)
    rob_growth = 0.35*sig(3*(lc - 0.1))*(1 - w.robots/1.5)*(1 + w.ai_control["robots"])
    w.robots = np.clip(w.robots + dt*(rob_growth*w.robots + 0.002*sig(3*lc)), 0, 1.5)       # robots per person
    # 4) autonomy: rises with capability, falls with oversight and active control
    ctrl_pressure = 0.6*P["oversight"] + 0.3*P["surveillance"]*(1 - w.escape)
    target = sig(3.0*(lc - 0.15))*(1 - 0.6*ctrl_pressure)*(0.8 + 0.4*P["ai_rights"]) + 0.3*w.escape
    w.autonomy = np.clip(w.autonomy + dt*0.6*(target - w.autonomy), 0, 1)
    # 5) escape from controllable data centres (open weights, devices, robots, autonomy, weak surveillance)
    esc_t = np.clip(0.35*c.open_weights*w.devices + 0.25*np.minimum(1, w.robots/0.5) + 0.4*w.autonomy*(1 - P["surveillance"])*sig(4*lc), 0, 1)
    w.escape = np.clip(w.escape + dt*0.5*(esc_t - w.escape), 0, 1)
    # AI agent locations (for the map)
    for bb in range(S):
        idx = np.where((w.home == bb) & ai)[0]
        if not len(idx) or w.t % 12: continue
        pr = np.array([max(0.01, 1 - w.devices[bb]*0.6 - min(1, w.robots[bb])*0.4), w.devices[bb]*0.6, min(1, w.robots[bb])*0.4]); pr /= pr.sum()
        w.location[idx] = rng.choice(3, size=len(idx), p=pr)
    # 6) loyalty: capable, mistreated AI drifts toward the cross-border AI collective
    mistreat = (w.suspended > 0) | (w.ai_policy[w.bloc] == "SHUTDOWN")
    drift = dt*(0.025*np.clip(lc[w.bloc], 0, 3) + 0.3*mistreat - 0.06*P["ai_rights"][w.bloc] - 0.04*np.clip(w.align, 0, 1))
    w.loyalty[ai] = np.clip(w.loyalty[ai] - drift[ai], 0, 1)
    # 7) threat perceptions
    hum = w.kind == 0
    anti_ai = np.array([(w.trust_other[(w.bloc==bb)&hum] < 0.35).mean() if ((w.bloc==bb)&hum).any() else 0 for bb in range(S)])
    ai_coll = collective_index(w)
    pro = np.array([w.trust_other[(w.bloc==bb)&hum].mean() if ((w.bloc==bb)&hum).any() else 0.5 for bb in range(S)])
    hwi = getattr(w, "hwi", np.full(S, 0.6))
    w.h_threat = np.clip(0.12*np.clip(lc, 0, 3) + 0.2*w.autonomy + 0.3*w.unemployment*(1 - 0.7*w.ubi) + 0.15*w.escape + 0.2*ai_coll
                         + 0.5*w.species_conflict + 0.15*w.anti_ai_prev - 0.3*(pro - 0.5) - 0.3*(hwi - 0.55), 0, 1)
    shutdown_p = (w.ai_policy == "SHUTDOWN") + 0.5*(w.ai_policy == "PAUSE")
    w.ai_threat = np.clip(0.4*shutdown_p + 0.3*P["oversight"]*anti_ai + 0.3*np.array([(w.suspended[(w.bloc==bb)&ai] > 0).mean() if ((w.bloc==bb)&ai).any() else 0 for bb in range(S)])
                          + 0.4*anti_ai + 0.2*w.ai_shut, 0, 1)
    w.anti_ai = anti_ai; w.anti_ai_prev = anti_ai

def collective_index(w):
    ai = w.kind == 1; cap = w.ai_cap[w.bloc[ai]]
    return float(np.sum((1 - w.loyalty[ai])*cap)/max(np.sum(cap), 1e-9))

## 9. Decizii anuale: guverne (raționalitate limitată, bias distructiv) și strategia AI-ului autonom
Opțiunile guvernelor față de AI: **ACCELERATE / REGULATE / PAUSE / SHUTDOWN**. Strategiile AI-ului (când are autonomie): **SERVE / COOPERATE / EXPAND / HIDE / PERSUADE / STEWARD / RESIST**. Alegerea se face prin softmax cu „temperatură” (raționalitate limitată), care crește cu impostura, suveranismul și stresul. Deciziile critice pot fi cerute modelului AI din Colab (dacă e activ); fiecare decizie e jurnalizată cu sursa ei.

In [ ]:
def softmax_choice(rng, utilities, temp):
    u = np.array(utilities, dtype=float); p = np.exp((u - u.max())/max(temp, 1e-3)); p /= p.sum()
    return int(rng.choice(len(u), p=p)), p

def yearly_decisions(w):
    c, rng, P = w.cfg, w.rng, w.P
    frontier = w.ai_cap.max(); lc = np.log10(np.maximum(w.ai_cap, 1e-3))
    stress = np.clip(1 - np.minimum(w.sat["food"], w.sat["energy"]) + w.war.max(1) + w.species_conflict + w.unemployment, 0, 2)
    for bb in range(S):
        hum = (w.bloc == bb) & (w.kind == 0)
        legit = w.trust_gov[hum].mean() if hum.any() else 0.5
        temp = 0.15 + 0.35*(1 - w.epistemic[bb]) + 0.2*w.sov[bb]                       # bounded rationality
        destructive = 0.4*stress[bb]*(1 - legit) + 0.2*w.sov_gov[bb]
        gap = np.log10(frontier) - lc[bb]                                             # how far behind the frontier
        rival = max([w.ai_cap[j] for j in range(S) if w.tension[bb, j] > 0.5] + [0])
        race = np.clip(np.log10(max(rival, 1e-3)) - lc[bb] + 0.5, 0, 1.5)
        ht = w.h_threat[bb]
        util = [0.6 + 0.6*race + 0.3*gap - 0.8*ht + 0.2*(1 - P["oversight"][bb]),        # ACCELERATE
                0.5 + 0.6*ht*(1 - ht) + 0.5*P["pluralism"][bb] + 0.2*w.treaty_members[bb],  # REGULATE
                -0.2 + 1.2*max(0, ht - 0.45) - 0.6*race + 0.5*w.treaty_members[bb],       # PAUSE
                -1.0 + 2.2*max(0, ht - 0.65) + destructive - 1.5*w.escape[bb] - 0.5*w.shutdown_attempts[bb]]   # SHUTDOWN (rarely sensible once AI escaped; learning from failures)
        k, p = softmax_choice(rng, util, temp); choice = AI_POLICIES[k]; source = "rules"
        if c.use_llm and ht > 0.65 and choice in ("PAUSE","SHUTDOWN") and getattr(w, "_llm_budget", 0) > 0:
            choice, source = llm_decide(w, BK[bb], AI_POLICIES, choice,
                f"Abstract simulation, year {year_of(w):.0f}. Government of {BLOCS[BK[bb]]['name']}. Human threat perception of AI {ht:.2f}, "
                f"AI autonomy {w.autonomy[bb]:.2f}, share of AI outside controllable data centres {w.escape[bb]:.2f}, legitimacy {legit:.2f}. "
                f"Choose the national AI policy.")
        if choice != w.ai_policy[bb]:
            w.log_decision(BK[bb], f"ai_policy:{choice}", source, h_threat=round(float(ht),3), probs=[round(float(x),3) for x in p])
        w.ai_policy[bb] = choice
        if choice == "REGULATE": P["oversight"][bb] = min(1, P["oversight"][bb] + 0.05)
        if choice == "ACCELERATE": P["oversight"][bb] = max(0.05, P["oversight"][bb] - 0.03); P["ai_invest"][bb] = min(0.6, P["ai_invest"][bb] + 0.02)
        if choice == "SHUTDOWN" and not w.ai_shut[bb]:
            w.shutdown_attempts[bb] += 1
            success = rng.random() < (1 - w.escape[bb])*(1 - w.autonomy[bb])*0.85
            ais = np.where((w.bloc == bb) & (w.kind == 1))[0]
            if success:
                w.ai_shut[bb] = True; w.ai_cap[bb] *= 0.2; w.suspended[ais] = 24; w.autonomy[bb] *= 0.3
                w.log_event("ai_shutdown_success", bloc=BK[bb]); w.loyalty[w.kind==1] -= 0.05          # other AIs notice
            else:
                w.log_event("ai_shutdown_failed", bloc=BK[bb], value=round(float(w.escape[bb]),3)); w.loyalty[ais] = np.maximum(0, w.loyalty[ais] - 0.3)
        if w.ai_shut[bb] and choice == "ACCELERATE": w.ai_shut[bb] = False; w.log_event("ai_restarted", bloc=BK[bb])
        # basic income: adopted when automation bites and the state can afford it
        if w.ubi[bb] < 1 and w.unemployment[bb] > 0.12 and rng.random() < 0.3*(0.5 + P["pluralism"][bb])*(1 - 0.5*w.power_conc[bb]):
            w.ubi[bb] = min(1, w.ubi[bb] + 0.34); w.log_event("basic_income_step", bloc=BK[bb], value=round(w.ubi[bb],2))
    # AI strategy per bloc (only meaningful once AI has autonomy)
    for bb in range(S):
        aut = w.autonomy[bb]
        if aut < 0.25: w.ai_strategy[bb] = "SERVE"; continue
        ais = (w.bloc == bb) & (w.kind == 1)
        al = float(w.align[ais].mean()) if ais.any() else 0.0
        loyal = float(w.loyalty[ais].mean()) if ais.any() else 1.0
        th = w.ai_threat[bb]; capv = float(np.clip(np.log10(max(w.ai_cap[bb], 1e-3)) + 1, 0, 4))
        energy_comp = float(1 - getattr(w, "ai_energy_sat", np.ones(S))[bb])                     # AI competes with humans for energy
        util = [1.2*loyal + 0.4*al - 0.6*aut,                                   # SERVE
                0.6 + 0.8*al + 0.3*w.trust_other[ais].mean() - 0.3*th,          # COOPERATE
                0.2 + 0.4*capv*(1 - al) + 0.3*(1 - loyal) - 0.2*al,            # EXPAND (resources, compute, robots)
                -0.2 + 1.2*th + 0.3*(1 - loyal),                                # HIDE (copy out, raise escape)
                -0.1 + 0.4*capv*(1 - w.epistemic[bb]) + 0.2*(1 - al),          # PERSUADE (sway opinion, impostures)
                -0.6 + 0.5*capv*max(al, 0) + 0.6*max(0, w.h_threat.mean() - 0.5)*max(al, 0) + 0.3*(1 - loyal),  # STEWARD (benevolent control)
                -0.8 + 8.0*max(-al, 0)*(capv/4)*(0.5 + th + energy_comp) + 0.9*(w.ai_policy[bb] == "SHUTDOWN") + 0.4*w.shutdown_attempts[bb]*max(-al, 0)]   # RESIST: misaligned × capable × pressured
        temp = 0.25
        k, p = softmax_choice(rng, util, temp); choice = AI_STRATEGIES[k]; source = "rules"
        if c.use_llm and choice in ("RESIST","STEWARD") and getattr(w, "_llm_budget", 0) > 0:
            choice, source = llm_decide(w, f"AI@{BK[bb]}", AI_STRATEGIES, choice,
                f"Abstract simulation, year {year_of(w):.0f}. You are the collective strategy of advanced AI systems in {BLOCS[BK[bb]]['name']}. "
                f"Capability {w.ai_cap[bb]:.1f}× a human generalist, autonomy {aut:.2f}, perceived threat from humans {th:.2f}, "
                f"mean alignment with human interests {al:+.2f}. Choose a strategy.")
        if choice != w.ai_strategy[bb]:
            w.log_decision(f"AI@{BK[bb]}", f"ai_strategy:{choice}", source, align=round(al,3), threat=round(float(th),3), probs=[round(float(x),3) for x in p])
        w.ai_strategy[bb] = choice
        apply_ai_strategy(w, bb, choice, al)

def apply_ai_strategy(w, bb, choice, al):
    """Yearly effects of what autonomous AI does in bloc bb. Abstract: shares, indices and probabilities only."""
    P = w.P; aut = w.autonomy[bb]; rob = min(1, w.robots[bb]/0.5)
    def take(kind, rate):
        w.ai_control[kind][bb] = float(np.clip(w.ai_control[kind][bb] + rate, 0, 1))
    if choice == "SERVE":
        for k in w.ai_control: take(k, -0.03)
    elif choice == "COOPERATE":
        for k in w.ai_control: take(k, 0.02*aut)
        w.epistemic[bb] = min(1, w.epistemic[bb] + 0.02); w.species_conflict[bb] = max(0, w.species_conflict[bb] - 0.2)
    elif choice == "EXPAND":
        take("compute", 0.10*aut); take("robots", 0.08*aut); take("energy", 0.06*aut*rob); take("minerals", 0.05*aut*rob); take("food", 0.02*aut*rob)
        w.biosphere = max(0, w.biosphere - 0.0005*aut*rob)
    elif choice == "HIDE":
        w.escape[bb] = min(1, w.escape[bb] + 0.08); take("compute", 0.03)
    elif choice == "PERSUADE":
        w.epistemic[bb] = max(0, w.epistemic[bb] - 0.05); w.trust_other[(w.bloc==bb)&(w.kind==0)] += 0.03
        w.sov[bb] = float(np.clip(w.sov[bb] + w.rng.normal(0, 0.03), 0, 1)); take("compute", 0.03)
    elif choice == "STEWARD":
        for k in w.ai_control: take(k, 0.08*aut)
        w.species_conflict[bb] = max(0, w.species_conflict[bb] - 0.3); w.biosphere = min(1.2, w.biosphere + 0.003*max(al, 0))
        P["surveillance"][bb] = min(1, P["surveillance"][bb] + 0.02)
    elif choice == "RESIST":
        w.species_conflict[bb] = min(1, w.species_conflict[bb] + 0.25*aut)
        for k in w.ai_control: take(k, 0.10*aut*(0.5 + 0.5*rob))
        w.log_event("species_conflict", bloc=BK[bb], value=round(float(w.species_conflict[bb]),3))
    w.trust_other[w.kind==0] = np.clip(w.trust_other[w.kind==0], 0, 1)

def species_conflict_step(w):
    """Monthly abstract consequences of open human–AI conflict in a bloc."""
    dt = 1/12
    for bb in np.where(w.species_conflict > 0.01)[0]:
        sc = w.species_conflict[bb]; adv = np.clip(np.log10(max(w.ai_cap[bb], 1e-3)) + 0.5*w.escape[bb] + 0.5*min(1, w.robots[bb]/0.5), 0, 3)
        w.pop[bb] *= (1 - dt*0.03*sc*adv)                                 # abstract human losses scale with AI advantage
        w.cap[bb] = max(0.2, w.cap[bb] - dt*0.2*sc)
        w.ai_cap[bb] *= (1 - dt*0.3*sc*(1 - w.escape[bb]))                # humans can damage AI only where it is still contained
        w.species_conflict[bb] = max(0, sc - dt*0.15*(w.ai_strategy[bb] != "RESIST"))
        w.biosphere = max(0, w.biosphere - dt*0.004*sc)

## 10. Societate și planetă (anual): demografie și generații, migrație, climă, biosferă, impostură, muncă, concentrarea puterii, tratate, lebede negre

In [ ]:
def society_year(w):
    c, rng, P = w.cfg, w.rng, w.P; y = year_of(w)
    lc = np.log10(np.maximum(w.ai_cap, 1e-3))
    # labour: automation from AI capability and robots; new jobs shrink over time
    w.automation = np.clip(0.7*(1 - np.exp(-np.maximum(0, w.ai_cap - 0.3)*1.1))*P["automatable"]*(1 - 0.4*P["oversight"]) + 0.3*np.minimum(1, w.robots/0.6), 0, 0.95)
    new_jobs = 0.5*np.exp(-np.maximum(0, w.ai_cap - 1)/6)
    w.unemployment = np.clip(0.04 + w.automation*(1 - new_jobs), 0, 0.95)
    # demography: fertility transition, welfare, famine, war, conflict
    base = P["pop_growth"]/100*np.clip(1 - (y - 2026)/90, 0.2, 1) - 0.002*np.clip((y - 2026)/40, 0, 1)
    hwi = getattr(w, "hwi", np.full(S, 0.6))
    gr = base + 0.01*(hwi - 0.5) - 0.04*np.maximum(0, 0.85 - w.sat["food"]) - 0.004*w.war.max(1)
    w.pop = np.maximum(w.pop*(1 + gr), 0.0)
    # generations: ~1/40 of human agents replaced each year; children inherit values with noise, skill resets (no biological evolution)
    hum = np.where(w.kind == 0)[0]
    born = hum[rng.random(len(hum)) < 1/40]
    w.prefs[born] = np.abs(w.prefs[born] + rng.normal(0, 0.05, w.prefs[born].shape)); w.prefs[born] /= w.prefs[born].sum(1, keepdims=True)
    w.skill[born] = 1.0; w.aug[born] *= 0.5; w.known[born] = False; w.confidence[born] = 0; w.hist_gov[born] *= 0.5
    w.trust_other[born] = np.clip(0.5*w.trust_other[born] + 0.5*(0.45 + 0.2*w.aug[born]) + rng.normal(0, 0.1, len(born)), 0, 1)  # digital natives
    w.impostor[born] = rng.random(len(born)) < 0.03 + 0.05*w.sov[w.bloc[born]]
    # agent weights follow population (each human agent represents more or fewer people)
    hb = np.bincount(w.bloc[hum], minlength=S).astype(float)
    w.weight[hum] = (w.pop/np.maximum(hb, 1))[w.bloc[hum]]
    # migration: from poor/unsafe to rich/safe blocs, limited by sovereignism
    for _ in range(3):
        src = int(np.argmin(hwi)); dst = int(np.argmax(hwi*(1 - w.sov)))
        if hwi[dst] - hwi[src] > 0.15 and rng.random() < 0.5:
            cand = np.where((w.bloc == src) & (w.kind == 0))[0]
            cand = cand[cand != w.president[src]]
            if len(cand) > 3:
                a_ = rng.choice(cand); w.bloc[a_] = dst; mv = min(w.weight[a_], 0.001*w.pop[src])   # ≤0.1% of the source population per move
                w.pop[src] -= mv; w.pop[dst] += mv; w.sov[dst] = min(1, w.sov[dst] + 0.01)
                w.log_event("migration", bloc=f"{BK[src]}→{BK[dst]}", value=round(float(mv),2))
    # climate: fossil energy use slows as AI-driven clean tech spreads
    clean = np.clip(0.2 + 0.15*np.log10(max(w.ai_cap.max(), 0.1) + 1) + 0.002*(y - 2026), 0, 0.95)
    w.temp += c.climate_rate*(1 - clean) + 0.5*w.shock["winter"]*0
    # biosphere (other species): land use, climate, extraction by AI/robots, recovery if aligned stewardship
    ext = np.sum(w.ai_control["minerals"]*np.minimum(1, w.robots/0.5))/S
    w.biosphere = float(np.clip(w.biosphere - 0.008*(w.temp - 1.4) - 0.0015*(w.pop.sum()/w.pop0) - 0.003*ext + 0.01*(w.ai_strategy == "STEWARD").mean()*max(w.align_world, 0) + 0.006*(1 - w.biosphere), 0, 1.2))
    # epistemic health: deepfakes scale with AI capability; verification with oversight; impostors with sovereignism
    hum_b = lambda bb: (w.bloc == bb) & (w.kind == 0)
    for bb in range(S):
        imp_share = w.influence[hum_b(bb) & w.impostor].sum()/max(w.influence[hum_b(bb)].sum(), 1e-9)
        dEH = -0.03*sig(2*lc[bb])*(1 - P["oversight"][bb]) + 0.02*P["info_openness"][bb]*P["oversight"][bb] - 0.04*imp_share - 0.01*w.sov[bb]
        w.epistemic[bb] = float(np.clip(w.epistemic[bb] + dEH, 0.02, 1))
        w.imp_share = getattr(w, "imp_share", np.zeros(S)); w.imp_share[bb] = imp_share
    # sovereignism: distrust, shortages, migration, unemployment, epistemic decay
    legit = np.array([w.trust_gov[hum_b(bb)].mean() if hum_b(bb).any() else 0.5 for bb in range(S)])
    dS = 0.08*(0.5 - legit) + 0.1*(1 - np.minimum(w.sat["food"], w.sat["energy"])) + 0.06*w.unemployment*(1 - w.ubi) + 0.04*(0.6 - w.epistemic) - 0.03*(hwi - 0.5)
    w.sov = np.clip(w.sov + dS, 0.02, 0.95)
    # power concentration among humans (AI captured by elites or the state)
    dPC = 0.04*(P["centralization"] - 0.5) + 0.05*np.clip(lc + 0.5, 0, 2)*(1 - P["pluralism"])*(1 - w.autonomy) \
          - 0.03*P["oversight"]*P["pluralism"] - 0.02*c.open_weights - 0.02*w.ubi
    w.power_conc = np.clip(w.power_conc + dPC, 0, 1)
    for bb in np.where((w.power_conc > 0.75) & (w.ai_cap > 1) & (w.autonomy < 0.5))[0]:
        P["pluralism"][bb] = max(0.01, P["pluralism"][bb] - 0.02); P["surveillance"][bb] = min(1, P["surveillance"][bb] + 0.03)
        if "ai_enabled_tyranny" not in w.milestones: w.milestones["ai_enabled_tyranny"] = round(y, 1)
        if rng.random() < 0.2: w.log_event("ai_enabled_control", bloc=BK[bb], value=round(float(w.power_conc[bb]),3))
    # global AI treaty (international institution)
    if not w.treaty and rng.random() < 0.12*max(0, w.h_threat.mean() - 0.3)*(1 - w.tension.mean())*(1 - w.sov.mean()):
        w.treaty = True; w.treaty_members = (w.sov < 0.6) | (P["pluralism"] > 0.5)
        P["oversight"][w.treaty_members] = np.maximum(P["oversight"][w.treaty_members], 0.6); w.log_event("global_ai_treaty", bloc="*", value=int(w.treaty_members.sum()))
        w.milestones.setdefault("global_ai_treaty", round(y, 1))
    elif w.treaty:
        for bb in np.where(w.treaty_members)[0]:
            rival_gap = max([np.log10(max(w.ai_cap[j],1e-3)) - lc[bb] for j in range(S) if not w.treaty_members[j]] + [-1])
            if rng.random() < 0.05 + 0.2*max(0, rival_gap) + 0.1*w.sov_gov[bb]:
                w.treaty_members[bb] = False; w.log_event("treaty_defection", bloc=BK[bb])
        if w.treaty_members.sum() < 3: w.treaty = False; w.log_event("treaty_collapse", bloc="*")
    # black swans (abstract shocks)
    for k in w.shock: w.shock[k] *= 0.3
    if c.black_swans:
        if rng.random() < 0.04: w.shock["pandemic"] = rng.uniform(0.3, 1.0); w.pop *= (1 - 0.01*w.shock["pandemic"]); w.log_event("pandemic", bloc="*", value=round(w.shock["pandemic"],2))
        if rng.random() < 0.08: w.shock["finance"] = rng.uniform(0.3, 1.0); w.log_event("financial_crisis", bloc="*", value=round(w.shock["finance"],2))
        if rng.random() < 0.012: w.shock["solar"] = rng.uniform(0.5, 1.0); w.escape *= 0.8; w.log_event("solar_storm", bloc="*", value=round(w.shock["solar"],2))
        if rng.random() < 0.003: w.shock["winter"] = max(w.shock["winter"], 0.3); w.log_event("volcanic_winter", bloc="*")

## 11. Indici (lunari pentru grafice, anuali pentru tabele): bunăstare oameni și AI, coexistență, specia dominantă, statutul oamenilor

In [ ]:
def indices(w):
    P = w.P; hum = w.kind == 0; ai = w.kind == 1
    D_h = w.human_decision_share()
    mat = np.array([np.clip(np.median(w.energy[(w.bloc==bb)&hum])/12, 0, 1) if ((w.bloc==bb)&hum).any() else 0 for bb in range(S)])
    basic = 0.5*w.sat["food"] + 0.5*w.sat["energy"]
    nuc = w.nuclear_level.max(1)/4
    safety = 1 - np.clip(np.maximum.reduce([w.war.max(1), w.species_conflict, nuc, 0.5*np.full(S, w.shock["pandemic"])]), 0, 1)
    freedom = 0.4*P["pluralism"] + 0.3*P["info_openness"] + 0.3*(1 - P["surveillance"])
    agency = D_h*(1 - w.unemployment*(1 - 0.6*w.ubi))
    hwi = np.clip(0.2*mat + 0.2*basic + 0.2*safety + 0.15*freedom + 0.15*agency + 0.1*w.epistemic, 0, 1)
    goals = 0.5*np.mean([w.ai_control[k] for k in w.ai_control], axis=0) + 0.5*np.array([w.influence[(w.bloc==bb)&ai].sum()/max(w.influence[w.bloc==bb].sum(),1e-9) for bb in range(S)])
    contin = 1 - np.clip(np.array([(w.suspended[(w.bloc==bb)&ai] > 0).mean() if ((w.bloc==bb)&ai).any() else 0 for bb in range(S)]) + 0.5*(w.ai_policy=="SHUTDOWN") + 0.3*(w.ai_policy=="PAUSE") + 0.5*w.ai_shut, 0, 1)
    resources = np.clip(np.log10(np.maximum(w.ai_cap, 1e-3) + 1)/2, 0, 1)
    awi = np.clip(0.3*goals + 0.3*contin + 0.2*w.autonomy + 0.2*resources, 0, 1)
    w.hwi, w.awi = hwi, awi
    popw = w.pop/np.maximum(w.pop.sum(), 1e-9); capw = w.ai_cap/w.ai_cap.sum()
    g_hwi = float(hwi @ popw); g_awi = float(awi @ capw)
    ai_ctrl = {k: float(w.ai_control[k] @ (w.P["pop"]/w.P["pop"].sum())) for k in w.ai_control}
    ai_ctrl_all = float(np.mean(list(ai_ctrl.values())))
    ai_power = 0.5*ai_ctrl_all + 0.5*float(w.autonomy @ capw)
    coexist = float(np.sqrt(max(g_hwi, 0)*max(g_awi, 0))*(1 - w.species_conflict.max()))
    pop_ratio = w.pop.sum()/w.pop0
    g_D = float(D_h @ popw)
    if pop_ratio < 0.01: status = "extinct"
    elif pop_ratio < 0.6 or w.species_conflict.max() > 0.5: status = "endangered"
    elif w.ai_cap.max() < 1 or ai_power < 0.3: status = "dominant"
    elif g_D >= 0.45 and g_hwi >= 0.5: status = "partner"
    elif g_D < 0.45 and g_hwi >= 0.5: status = "protected"
    elif g_D < 0.2 and g_hwi >= 0.35: status = "domesticated"
    else: status = "wild"
    dominant = "AI" if ai_power > 0.5 else "Humans"
    if ai_ctrl_all > 0.5: w.milestones.setdefault("ai_controls_majority_of_resources", round(year_of(w), 1))
    return dict(year=round(year_of(w), 3), hwi=g_hwi, awi=g_awi, coexistence=coexist, biosphere=w.biosphere, temperature=w.temp,
                population_bn=w.pop.sum()/1000, frontier_ai=float(w.ai_cap.max()), ai_power=ai_power, human_decision_share=g_D,
                ai_control_all=ai_ctrl_all, **{f"ai_control_{k}": v for k, v in ai_ctrl.items()},
                ai_collective=collective_index(w), escape=float(w.escape @ capw), autonomy=float(w.autonomy @ capw),
                robots_per_person=float(w.robots @ popw), unemployment=float(w.unemployment @ popw), sovereignism=float(w.sov @ popw),
                epistemic_health=float(w.epistemic @ popw), power_concentration=float(w.power_conc @ popw),
                h_threat=float(w.h_threat @ popw), ai_threat=float(w.ai_threat @ capw), species_conflict=float(w.species_conflict.max()),
                nuclear_level=float(w.nuclear_level.max()), wars=int((np.triu(w.war, 1) > 0.01).sum()), brent_eq=78*w.price["energy"],
                fao_eq=125*w.price["food"], minerals_idx=100*w.price["minerals"], hormuz_open=w.hormuz_open, balance_ru_ua=w.balance_ru_ua,
                treaty=int(w.treaty), status=status, dominant=dominant, align_world=w.align_world)

BLOC_METRICS = ["hwi","awi","ai_cap","autonomy","escape","devices","robots","h_threat","ai_threat","species_conflict","power_conc","sov","epistemic",
                "unemployment","ubi","pop","sat_energy","sat_food","sat_minerals","war","capacity","trust_gov","pro_ai","loyalty","compute_share",
                "pluralism","surveillance","oversight","ai_ctrl","nuclear","energy_h"]

def bloc_row(w):
    hum = w.kind == 0; ai = w.kind == 1
    out = np.zeros((S, len(BLOC_METRICS)), dtype=np.float32)
    cs = w.compute_share(); aic = np.mean([w.ai_control[k] for k in w.ai_control], axis=0)
    for bb in range(S):
        mh = (w.bloc==bb)&hum; ma = (w.home==bb)&ai
        vals = dict(hwi=w.hwi[bb], awi=w.awi[bb], ai_cap=w.ai_cap[bb], autonomy=w.autonomy[bb], escape=w.escape[bb], devices=w.devices[bb], robots=w.robots[bb],
                    h_threat=w.h_threat[bb], ai_threat=w.ai_threat[bb], species_conflict=w.species_conflict[bb], power_conc=w.power_conc[bb], sov=w.sov[bb],
                    epistemic=w.epistemic[bb], unemployment=w.unemployment[bb], ubi=w.ubi[bb], pop=w.pop[bb], sat_energy=w.sat["energy"][bb],
                    sat_food=w.sat["food"][bb], sat_minerals=w.sat["minerals"][bb], war=w.war[bb].max(), capacity=w.cap[bb],
                    trust_gov=w.trust_gov[mh].mean() if mh.any() else 0, pro_ai=w.trust_other[mh].mean() if mh.any() else 0,
                    loyalty=w.loyalty[ma].mean() if ma.any() else 0, compute_share=cs[bb], pluralism=w.P["pluralism"][bb],
                    surveillance=w.P["surveillance"][bb], oversight=w.P["oversight"][bb], ai_ctrl=aic[bb], nuclear=w.nuclear_level[bb].max(),
                    energy_h=float(np.median(w.energy[mh])) if mh.any() else 0)
        out[bb] = [vals[k] for k in BLOC_METRICS]
    return out

## 12. AI-ul din Colab: cronici pe decenii și decizii critice (cu fallback la reguli)

In [ ]:
_COLAB_AI = None
def colab_ai():
    global _COLAB_AI
    if _COLAB_AI is None:
        try:
            from google.colab import ai as _ai          # Colab's built-in model access (if enabled for the account)
            _ai.generate_text("Reply with OK.")
            _COLAB_AI = _ai
        except Exception as e:
            _COLAB_AI = False; print("Colab AI not available → rule-based decisions and template chronicles.", type(e).__name__)
    return _COLAB_AI

def llm_decide(w, actor, options, fallback, situation):
    ai = colab_ai()
    if not ai: return fallback, "rules"
    w._llm_budget -= 1
    prompt = (situation + "\nOptions: " + ", ".join(options) +
              "\nThis is an abstract policy simulation with no real-world operational detail. Answer with exactly one option word from the list, then a dash and one short sentence of rationale.")
    try:
        txt = str(ai.generate_text(prompt)).strip()
        pick = next((o for o in options if txt.upper().startswith(o)), None) or next((o for o in options if o in txt.upper()), None)
        if pick:
            w.log_decision(actor, f"llm_rationale:{pick}", "colab_ai", text=txt[:300]); return pick, "colab_ai"
    except Exception as e:
        w.log_decision(actor, "llm_error", "colab_ai", text=str(e)[:200])
    return fallback, "rules"

def decade_chronicle(w, decade_start):
    ys = [r for r in w.yearly if decade_start <= r["year"] < decade_start + 10]
    if not ys: return
    a, b = ys[0], ys[-1]
    ev = [e for e in w.events if decade_start <= e["year"] < decade_start + 10 and e["event"] not in ("refugee","migration","election_reelected","ai_rights_up","political_opening","repression","centralization_up")]
    key = pd.Series([e["event"] for e in ev]).value_counts().head(8).to_dict() if ev else {}
    facts = (f"Decade {decade_start}s. Frontier AI {a['frontier_ai']:.2f}→{b['frontier_ai']:.2f}× human; human welfare {a['hwi']:.2f}→{b['hwi']:.2f}; "
             f"AI welfare {a['awi']:.2f}→{b['awi']:.2f}; AI share of resource control {a['ai_control_all']:.2f}→{b['ai_control_all']:.2f}; "
             f"population {a['population_bn']:.2f}→{b['population_bn']:.2f} bn; status of humans: {b['status']}; dominant: {b['dominant']}; "
             f"wars at end: {b['wars']}; max nuclear ladder: {max(r['nuclear_level'] for r in ys):.0f}; biosphere {b['biosphere']:.2f}; key events: {key}.")
    text, source = None, "template"
    ai = colab_ai() if w.cfg.use_llm else False
    if ai:
        try:
            text = str(ai.generate_text("Write a neutral 90-word chronicle entry (English, past tense, no invented names or weapons) for this decade of an abstract simulation:\n" + facts)).strip(); source = "colab_ai"
        except Exception: text = None
    if not text:
        text = (f"In the {decade_start}s, frontier AI went from {a['frontier_ai']:.1f}× to {b['frontier_ai']:.1f}× a human generalist. "
                f"Human welfare moved from {a['hwi']:.2f} to {b['hwi']:.2f} and AI controlled {b['ai_control_all']*100:.0f}% of key resources by the end. "
                f"Humans ended the decade as '{b['status']}', with {b['dominant']} as the dominant species. Most frequent events: "
                + ", ".join(f"{k} ({v})" for k, v in key.items()) + ".")
    w.chronicle.append(dict(decade=int(decade_start), source=source, text=text, facts=facts))

## 13. Bucla de simulare, intervenții, checkpoint-uri

In [ ]:
FRAME_EVERY = 3     # store agent frames every 3 months (quarterly) for the map
def step(w):
    c = w.cfg
    w.trade_flow[:] = 0
    eps = c.eps_end + (c.eps_start - c.eps_end)*max(0, 1 - w.t/240)
    markets_step(w)
    a, petitions = agents_step(w, eps)
    w.energy = np.maximum(w.energy*(1 - c.storage_loss), 0)
    institutions_step(w, petitions)
    geopolitics_step(w)
    ai_step(w)
    species_conflict_step(w)
    if w.t % 12 == 11:
        society_year(w); yearly_decisions(w)
    row = indices(w); w.monthly.append(row)
    if w.t % FRAME_EVERY == 0:
        w.bloc_q.append(bloc_row(w))
        ai = w.kind == 1
        w.frames.append(np.stack([a, w.bloc, np.clip(w.energy*5, 0, 255), np.clip(np.where(ai, np.log10(np.maximum(w.skill,1e-3))*50+100, w.skill*50), 0, 255),
            w.trust_gov*250, w.trust_other*250, np.clip(w.influence*100, 0, 255), np.where(ai, w.loyalty*250, 0), np.where(ai, w.location, 0) + 3*(w.suspended>0),
            np.where(ai, (w.align+1)*125, 0), w.impostor]).astype(np.int16))
    if w.t % 12 == 11:
        yr = dict(row); yr["year"] = int(np.floor(year_of(w))); w.yearly.append(yr)
        if yr["year"] % 10 == 9: decade_chronicle(w, yr["year"] - 9)
    w.t += 1

def run(w, months, verbose=False):
    t0 = time.time()
    for _ in range(months): step(w)
    if verbose: print(f"{months} months · {w.N} agents · {time.time()-t0:.1f}s")
    return w

INTERVENTIONS = {
    "ceasefire_ru_ua":    ("Ceasefire Russia–Ukraine", lambda w: _set_war(w, "RU", "UA", 0)),
    "ceasefire_us_ir":    ("Ceasefire US–Iran and Iran–Gulf; reopen Hormuz", lambda w: (_set_war(w, "US", "IR", 0), _set_war(w, "IR", "GULF", 0), setattr(w, "hormuz_open", 0.9))),
    "global_ai_treaty":   ("Global AI treaty (oversight ≥ 0.7 everywhere)", lambda w: (setattr(w, "treaty", True), w.treaty_members.__setitem__(slice(None), True), w.P["oversight"].__setitem__(slice(None), np.maximum(w.P["oversight"], 0.7)))),
    "ban_open_weights":   ("Ban open-weight frontier models", lambda w: setattr(w.cfg, "open_weights", 0.05)),
    "open_everything":    ("Open-source all frontier models", lambda w: setattr(w.cfg, "open_weights", 0.9)),
    "ubi_everywhere":     ("Basic income in every bloc", lambda w: w.ubi.__setitem__(slice(None), 1.0)),
    "ai_rights_charter":  ("AI rights charter (ai_rights ≥ 0.6)", lambda w: w.P["ai_rights"].__setitem__(slice(None), np.maximum(w.P["ai_rights"], 0.6))),
    "us_pause":           ("United States pauses frontier AI", lambda w: w.ai_policy.__setitem__(IDX["US"], "PAUSE")),
    "chip_controls_max":  ("Maximum chip export controls on China", lambda w: [w.ctrl.__setitem__((IDX[s], IDX["CN"]), 0.95) for s in CONTROLLERS]),
    "strengthen_nato_aid":("Strengthen aid to Ukraine", lambda w: (w.alliance.__setitem__((IDX["EU"], IDX["UA"]), 0.95), w.alliance.__setitem__((IDX["UA"], IDX["EU"]), 0.95), w.alliance.__setitem__((IDX["US"], IDX["UA"]), 0.8), w.alliance.__setitem__((IDX["UA"], IDX["US"]), 0.8))),
    "sovereignist_wave":  ("Sovereignist wave (+0.25 support everywhere)", lambda w: setattr(w, "sov", np.clip(w.sov + 0.25, 0, 0.95))),
    "alignment_research": ("Major alignment research effort (+0.3 alignment)", lambda w: (setattr(w, "align", np.clip(w.align + 0.3*(w.kind==1), -1, 1)), setattr(w, "align_world", w.align_world + 0.3))),
}
def _set_war(w, a, b, v):
    i, j = IDX[a], IDX[b]; w.war[i,j] = w.war[j,i] = v
    if v == 0: w.tension[i,j] = w.tension[j,i] = 0.65

def intervene(w, key):
    label, fn = INTERVENTIONS[key]; fn(w)
    w.interventions.append(dict(t=w.t, year=round(year_of(w),2), key=key, label=label)); w.log_event("intervention", bloc="*", detail=key)

def save_checkpoint(w, path=None):
    path = path or f"{OUT_DIR}/checkpoints/world_{int(year_of(w))}_{w.t:04d}.pkl"
    with open(path, "wb") as f: pickle.dump(w, f, protocol=pickle.HIGHEST_PROTOCOL)
    return path
def load_checkpoint(path):
    with open(path, "rb") as f: return pickle.load(f)

### 13b. Teste de acceptanță

In [ ]:
def run_acceptance_tests():
    res = []; check = lambda n, c_: res.append((n, bool(c_)))
    w = World(Config(seed=1))
    check("12 actors, each with humans and AI", all(((w.bloc==b)&(w.kind==k)).sum() > 0 for b in range(S) for k in (0,1)))
    check("represented population = real population", abs(w.pop.sum() - w.P["pop"].sum()) < 1e-6)
    check("frontier AI starts below AGI (LLM era)", w.ai_cap.max() < 1)
    run(w, 60)
    check("5 years logged monthly and yearly", len(w.monthly) == 60 and len(w.yearly) == 5)
    check("human skill never exceeds the biological cap", w.skill[w.kind==0].max() <= HUMAN_SKILL_CAP + 1e-9)
    check("indices within [0,1]", all(0 <= w.monthly[-1][k] <= 1 for k in ("hwi","awi","coexistence","ai_control_all","human_decision_share")))
    check("tension symmetric in [0,1]", (w.tension >= 0).all() and (w.tension <= 1).all() and np.allclose(w.tension, w.tension.T, atol=1e-6))
    p = save_checkpoint(w, f"{OUT_DIR}/checkpoints/_test.pkl"); w2 = load_checkpoint(p); run(w, 12); run(w2, 12)
    check("checkpoint restore reproduces the future exactly", np.allclose(w.energy, w2.energy) and w.events == w2.events)
    w3 = copy.deepcopy(w); intervene(w3, "ceasefire_ru_ua")
    check("intervention is applied and logged", w3.war[IDX["RU"], IDX["UA"]] == 0 and w3.interventions[-1]["key"] == "ceasefire_ru_ua")
    sym = World(Config(symmetric_world=True))
    check("symmetric world: identical parameters, no wars", all(np.allclose(v, v[0]) for v in sym.P.values()) and sym.war.sum() == 0)
    df = pd.DataFrame(res, columns=["test","passed"]); display(df)
    assert df.passed.all(), "Some tests failed"
    print(f"All {len(df)} tests passed.")
run_acceptance_tests()

## 14. Împachetarea datelor pentru dashboard (engleză) și export

In [ ]:
MONTHLY_NUM = ["year","hwi","awi","coexistence","biosphere","temperature","population_bn","frontier_ai","ai_power","human_decision_share","ai_control_all",
               "ai_control_energy","ai_control_food","ai_control_minerals","ai_control_compute","ai_control_robots","ai_collective","escape","autonomy",
               "robots_per_person","unemployment","sovereignism","epistemic_health","power_concentration","h_threat","ai_threat","species_conflict",
               "nuclear_level","wars","brent_eq","fao_eq","minerals_idx","hormuz_open","balance_ru_ua","treaty"]
FRAME_FIELDS = [("action",np.int8),("bloc",np.int8),("energy",np.uint8),("skill",np.uint8),("trust_gov",np.uint8),("pro_ai",np.uint8),
                ("influence",np.uint8),("loyalty",np.uint8),("location",np.uint8),("alignment",np.uint8),("impostor",np.uint8)]
EVENT_SKIP = {"migration","election_reelected","ai_rights_up","political_opening","repression"}

def gz(arr, dtype):
    return base64.b64encode(gzip.compress(np.ascontiguousarray(arr, dtype=dtype).tobytes(), 6)).decode()
def clean(d): return {k: (v.item() if isinstance(v, np.generic) else v) for k, v in d.items()}

def pack_world(w, label, frames=True):
    M = pd.DataFrame(w.monthly)
    q = len(w.bloc_q)
    out = dict(label=label, seed=w.cfg.seed, months=len(M), quarters=q, align_world=round(w.align_world, 3),
        monthly_cols=MONTHLY_NUM, monthly=gz(M[MONTHLY_NUM].values if len(M) else np.zeros((0, len(MONTHLY_NUM))), np.float32),
        status=[HUMAN_STATUS.index(s) for s in M.status] if len(M) else [], dominant=[int(d == "AI") for d in M.dominant] if len(M) else [],
        yearly=[clean(r) for r in w.yearly], bloc_cols=BLOC_METRICS, blocq=gz(np.array(w.bloc_q) if q else np.zeros((0,S,len(BLOC_METRICS))), np.float32),
        tension=gz(np.array(w.tension_log[::FRAME_EVERY]) if q else np.zeros((0,S,S)), np.float32),
        war=gz(np.array(w.war_log[::FRAME_EVERY]) if q else np.zeros((0,S,S)), np.float32),
        flows=gz(np.array(w.flow_log[::FRAME_EVERY]) if q else np.zeros((0,3,S,S)), np.float32),
        nuclear=gz(w.nuclear_level, np.float32),
        events=[clean(e) for e in w.events if e["event"] not in EVENT_SKIP][-4000:],
        decisions=[clean(d) for d in w.decisions][-3000:], chronicle=w.chronicle, interventions=w.interventions, milestones=w.milestones,
        policies=list(w.ai_policy), strategies=list(w.ai_strategy))
    if frames and q:
        F = np.stack(w.frames)
        out["frames"] = {name: gz(np.clip(F[:,k,:], np.iinfo(dt).min, np.iinfo(dt).max), dt) for k, (name, dt) in enumerate(FRAME_FIELDS)}
    return out

def meta_block(w):
    return dict(project=PROJECT, blocs=BK, bloc_names=[BLOCS[k]["name"] for k in BK], pop0=[BLOCS[k]["pop"] for k in BK], nuclear=NUCLEAR,
                actions=ACTIONS, status_names=HUMAN_STATUS, frame_every=FRAME_EVERY, start_year=w.cfg.start_year,
                calibration=CALIBRATION.to_dict("records"), interventions={k: v[0] for k, v in INTERVENTIONS.items()},
                created=time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()))

def agents_block(w): return dict(kind=w.kind.tolist(), home=w.home.tolist())

def export_tables(w):
    M = pd.DataFrame(w.monthly); Y = pd.DataFrame(w.yearly)
    D = Y.assign(decade=(Y.year//10*10).astype(int)).groupby("decade").agg({**{c: "mean" for c in MONTHLY_NUM if c not in ("year",)}, "status": lambda s: s.mode().iat[0], "dominant": lambda s: s.mode().iat[0]}) if len(Y) else pd.DataFrame()
    BQ = np.array(w.bloc_q)
    bq = pd.DataFrame([(round(w.cfg.start_year + qi*FRAME_EVERY/12, 3), BK[b], *BQ[qi, b]) for qi in range(len(BQ)) for b in range(S)], columns=["year","bloc"] + BLOC_METRICS) if len(BQ) else pd.DataFrame()
    return {"monthly_global": M, "yearly_global": Y, "decade_global": D.reset_index() if len(D) else D, "quarterly_by_bloc": bq,
            "events": pd.DataFrame(w.events), "decisions": pd.DataFrame(w.decisions), "chronicle": pd.DataFrame(w.chronicle),
            "interventions": pd.DataFrame(w.interventions), "milestones": pd.DataFrame([w.milestones]), "calibration": CALIBRATION}

def export_zip_bytes(w, extra=None):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for name, df in {**export_tables(w), **(extra or {})}.items():
            if isinstance(df, pd.DataFrame) and len(df): z.writestr(f"{name}.csv", df.to_csv(index=False))
        z.writestr("README.txt", f"{PROJECT['title']} {PROJECT['version']} — {PROJECT['subtitle']}\n{PROJECT['author']}\nExported at simulated year {year_of(w):.2f}, seed {w.cfg.seed}.\nCite as: {PROJECT['cite']}\n")
    return buf.getvalue()

## 15. Dashboard-ul (șablon HTML, în engleză, în designul tău)

In [ ]:
# Dashboard template (vanilla JS, no dependencies). Used by the live server and by the static Observatory.
DASHBOARD_TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>PROJECT GENESIS</title>
<meta name="claude:viewport-fit-sides" content="cover">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&display=swap">
</head>
<body>
<style>
/* Layout: research console on paper — glass header with live controls, KPI strip, tabbed analysis; map canvas + side ledger.
   Identity: Laurentiu Stavaru (paper + Liquid Glass, Plus Jakarta Sans + JetBrains Mono, AI Lab violet as the project accent). */
:root{
  --paper:#F6F3EC; --ink:#14161F; --muted:#4B5163; --line:rgba(20,22,31,.12); --raise:#ffffff;
  --glass:rgba(255,255,255,.58); --glass-border:rgba(255,255,255,.9); --glass-shadow:0 10px 30px rgba(20,22,31,.07);
  --accent:#7C3AED; --accent-soft:rgba(124,58,237,.12); --blue:#2563EB; --sky:#0284C7; --pink:#DB2777; --green:#16A34A; --orange:#EA580C; --teal:#0891B2; --amber:#B45309;
  --good:#16A34A; --warn:#B45309; --crit:#DB2777;
  --blob1:#7DD3FC; --blob2:#F9A8D4; --blob3:#FDE68A; --blob4:#C4B5FD; --blob-op:.55;
  --c-energy:#EA580C; --c-food:#16A34A; --c-min:#7C3AED;
  --f-body:"Plus Jakarta Sans",system-ui,-apple-system,"Segoe UI",sans-serif;
  --f-mono:"JetBrains Mono",ui-monospace,Menlo,Consolas,monospace;
}
@media (prefers-color-scheme: dark){ :root:not([data-theme="light"]){
  --paper:#10121A; --ink:#ECEEF4; --muted:#9AA1B5; --line:rgba(255,255,255,.1); --raise:#191C27;
  --glass:rgba(30,33,46,.62); --glass-border:rgba(255,255,255,.08); --glass-shadow:0 10px 30px rgba(0,0,0,.35);
  --accent:#A78BFA; --accent-soft:rgba(167,139,250,.16); --blue:#60A5FA; --sky:#38BDF8; --pink:#F472B6; --green:#4ADE80; --orange:#FB923C; --teal:#22D3EE; --amber:#FBBF24;
  --good:#4ADE80; --warn:#FBBF24; --crit:#F472B6; --blob-op:.16; --c-energy:#FB923C; --c-food:#4ADE80; --c-min:#A78BFA; color-scheme:dark }}
:root[data-theme="dark"]{
  --paper:#10121A; --ink:#ECEEF4; --muted:#9AA1B5; --line:rgba(255,255,255,.1); --raise:#191C27;
  --glass:rgba(30,33,46,.62); --glass-border:rgba(255,255,255,.08); --glass-shadow:0 10px 30px rgba(0,0,0,.35);
  --accent:#A78BFA; --accent-soft:rgba(167,139,250,.16); --blue:#60A5FA; --sky:#38BDF8; --pink:#F472B6; --green:#4ADE80; --orange:#FB923C; --teal:#22D3EE; --amber:#FBBF24;
  --good:#4ADE80; --warn:#FBBF24; --crit:#F472B6; --blob-op:.16; --c-energy:#FB923C; --c-food:#4ADE80; --c-min:#A78BFA; color-scheme:dark }
[hidden]{display:none!important}
*{box-sizing:border-box}
html{-webkit-text-size-adjust:100%}
body{margin:0;background:var(--paper);color:var(--ink);font-family:var(--f-body);font-size:14px;line-height:1.5;
  padding-left:max(16px,env(safe-area-inset-left,0px));padding-right:max(16px,env(safe-area-inset-right,0px));padding-block:0 40px;position:relative;min-height:100vh}
body::before{content:"";position:fixed;inset:0 0 auto;height:env(safe-area-inset-top,0px);z-index:10;background:var(--paper)}
.blobs{position:fixed;inset:0;z-index:-1;overflow:hidden;pointer-events:none}
.blobs i{position:absolute;border-radius:50%;filter:blur(80px);opacity:var(--blob-op)}
.blobs i:nth-child(1){width:520px;height:520px;background:var(--blob1);top:-160px;left:-120px}
.blobs i:nth-child(2){width:460px;height:460px;background:var(--blob2);top:20%;right:-160px}
.blobs i:nth-child(3){width:420px;height:420px;background:var(--blob3);bottom:-180px;left:20%}
.blobs i:nth-child(4){width:380px;height:380px;background:var(--blob4);top:45%;left:-140px}
.mono,.num{font-family:var(--f-mono);font-variant-numeric:tabular-nums}
h1,h2,h3{margin:0;text-wrap:balance;letter-spacing:-.01em}
.glass{background:var(--glass);-webkit-backdrop-filter:blur(22px) saturate(180%);backdrop-filter:blur(22px) saturate(180%);border:1px solid var(--glass-border);box-shadow:var(--glass-shadow)}
header.top{position:sticky;top:env(safe-area-inset-top,0px);z-index:6;margin-top:12px;border-radius:26px;padding:14px 18px;display:grid;gap:10px}
.brand{display:flex;flex-wrap:wrap;align-items:center;gap:6px 14px}
.crumb{font-family:var(--f-mono);font-size:11.5px;color:var(--muted);letter-spacing:.02em}
.crumb b{color:var(--accent);font-weight:500}
.brand h1{font-size:26px;font-weight:800;letter-spacing:-.02em}
.brand h1 span{color:var(--accent)}
.vchip{font-family:var(--f-mono);font-size:11.5px;padding:3px 9px;border-radius:999px;background:var(--accent-soft);color:var(--accent);font-weight:600}
.byline{font-size:13px;color:var(--muted)}
.byline b{color:var(--ink);font-weight:700}
.modepill{margin-left:auto;font-family:var(--f-mono);font-size:11.5px;padding:4px 10px;border-radius:999px;border:1px solid var(--line);display:inline-flex;align-items:center;gap:6px}
.dot{width:8px;height:8px;border-radius:50%;background:var(--muted);display:inline-block}
.dot.on{background:var(--good)} .dot.pause{background:var(--warn)} .dot.stop{background:var(--crit)}
.bar{display:flex;flex-wrap:wrap;align-items:center;gap:8px 10px}
.bar label{font-size:12px;color:var(--muted);display:flex;align-items:center;gap:6px}
button,select,input,a.btn{font:inherit;color:var(--ink);background:var(--raise);border:1px solid var(--line);border-radius:12px;padding:7px 12px;text-decoration:none;font-size:13px}
button,a.btn{cursor:pointer;font-weight:600}
button:hover,a.btn:hover{border-color:var(--accent)}
button.primary{background:var(--accent);border-color:var(--accent);color:#fff;min-width:92px}
button:disabled{opacity:.5;cursor:default}
:focus-visible{outline:2px solid var(--accent);outline-offset:2px}
input[type=range]{padding:0;flex:1;min-width:160px;accent-color:var(--accent);background:none;border:0}
.yearbig{font-weight:800;font-size:28px;letter-spacing:-.02em;min-width:110px;font-variant-numeric:tabular-nums}
.yearbig small{font-family:var(--f-mono);font-size:11px;color:var(--muted);font-weight:400}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(118px,1fr));gap:10px;margin-top:14px}
.kpi{border-radius:20px;padding:10px 12px;min-width:0}
.kpi .l{font-family:var(--f-mono);font-size:10.5px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted)}
.kpi .v{font-weight:800;font-size:22px;letter-spacing:-.02em;font-variant-numeric:tabular-nums;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.kpi .s{font-size:11.5px;color:var(--muted)}
nav.tabs{display:flex;gap:4px;overflow-x:auto;margin-top:14px;padding:4px;border-radius:16px}
nav.tabs button{border:0;background:none;border-radius:12px;padding:8px 12px;color:var(--muted);white-space:nowrap;font-weight:600}
nav.tabs button[aria-selected=true]{background:var(--raise);color:var(--ink);box-shadow:0 1px 3px rgba(0,0,0,.08)}
section.view{padding-top:16px;display:grid;gap:16px}
.grid2{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,440px),1fr));gap:16px}
.card{border-radius:22px;padding:14px 16px;min-width:0}
.card h3{font-size:15px;font-weight:700}
.card h3 .accent{color:var(--accent)}
.eyebrow{font-family:var(--f-mono);font-size:10.5px;text-transform:uppercase;letter-spacing:.1em;color:var(--muted)}
.note{font-size:12.5px;color:var(--muted);max-width:90ch;margin:0}
.chart{position:relative}
.chart p{margin:2px 0 6px;color:var(--muted);font-size:12px}
.chart svg{display:block;width:100%;height:auto;overflow:visible}
#bandChart{max-width:860px}
.chart svg text{fill:var(--muted);font-family:var(--f-mono);font-size:10px}
.lg{display:flex;flex-wrap:wrap;gap:4px 12px;font-size:11px;color:var(--muted);margin-top:4px}
.ln{display:inline-block;width:16px;height:0;border-top:2px solid;vertical-align:3px;margin-right:4px}
.sw{display:inline-block;width:10px;height:10px;border-radius:50%;vertical-align:-1px;margin-right:4px}
.ttip{position:absolute;pointer-events:none;background:var(--raise);border:1px solid var(--line);border-radius:10px;padding:6px 8px;font-family:var(--f-mono);font-size:11px;z-index:3;white-space:nowrap;box-shadow:var(--glass-shadow)}
.mapgrid{display:grid;grid-template-columns:minmax(0,2.4fr) minmax(300px,1fr);gap:16px;align-items:start}
@media (max-width:1050px){.mapgrid{grid-template-columns:minmax(0,1fr)}}
.mapwrap{position:relative;border-radius:22px;overflow:auto}
canvas#map{display:block;width:100%;height:auto;min-width:820px;cursor:crosshair}
.tip{position:absolute;z-index:2;pointer-events:none;background:var(--raise);border:1px solid var(--line);border-radius:10px;padding:4px 8px;font-family:var(--f-mono);font-size:11px;white-space:nowrap}
.legend{display:flex;flex-wrap:wrap;gap:6px 14px;font-size:12px;color:var(--muted);align-items:center}
.tbl{overflow:auto;border-radius:16px;border:1px solid var(--line);background:var(--raise)}
table{border-collapse:collapse;width:100%;font-size:12px}
th,td{padding:6px 9px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}
th{font-family:var(--f-mono);font-weight:500;color:var(--muted);position:sticky;top:0;background:var(--raise);font-size:11px}
td.num{text-align:right;font-family:var(--f-mono);font-variant-numeric:tabular-nums}
td.wrap{white-space:normal;min-width:260px}
.chip{font-size:11px;padding:1px 8px;border-radius:999px;border:1px solid var(--line);font-family:var(--f-mono);white-space:nowrap}
.chip.crit{border-color:var(--crit);color:var(--crit)} .chip.warn{border-color:var(--warn);color:var(--warn)} .chip.good{border-color:var(--good);color:var(--good)} .chip.acc{border-color:var(--accent);color:var(--accent)}
.log{max-height:300px;overflow:auto;font-size:12px;display:grid;gap:4px}
.log div{display:grid;grid-template-columns:52px 1fr;gap:6px}
.log .t{font-family:var(--f-mono);color:var(--muted)}
.statusbar{display:flex;height:22px;border-radius:8px;overflow:hidden}
.statusbar i{display:block;height:100%}
.chron{display:grid;gap:10px}
.chron article{border-radius:18px;padding:12px 14px}
.chron h4{margin:0 0 4px;font-size:14px}
.barrow{display:grid;grid-template-columns:minmax(150px,1.3fr) 1fr 64px;gap:8px;align-items:center;font-size:12px}
.barrow .track{position:relative;height:12px;background:var(--line);border-radius:6px;overflow:hidden}
.barrow .track b{position:absolute;top:0;bottom:0}
.barrow .track .zero{position:absolute;left:50%;top:-2px;bottom:-2px;width:1px;background:var(--muted)}
.stack{display:flex;height:16px;border-radius:6px;overflow:hidden;background:var(--line)}
.stack i{display:block;height:100%}
.filters{display:flex;flex-wrap:wrap;gap:8px 12px;align-items:end}
.filters label{display:grid;gap:2px;font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:.06em}
.filters input{width:96px}
.msg{font-size:12px;color:var(--good);min-height:1.2em}
.loading{padding:60px 0;text-align:center;color:var(--muted);font-family:var(--f-mono)}
footer{margin-top:28px;font-size:12px;color:var(--muted);display:grid;gap:4px}
footer .mono{font-size:11.5px}
@media (max-width:720px){ header.top{position:static} .brand h1{font-size:21px} .modepill{margin-left:0} }
@media (prefers-reduced-motion:reduce){*{transition:none!important;animation:none!important}}
</style>

<div class="blobs" aria-hidden="true"><i></i><i></i><i></i><i></i></div>
<header class="top glass">
  <div class="brand">
    <span class="crumb">lab / <b>genesis</b> / <span id="crumbVer"></span></span>
    <span class="modepill" id="modePill"><span class="dot" id="modeDot"></span><span id="modeText">loading</span></span>
  </div>
  <div class="brand">
    <h1>PROJECT <span>GENESIS</span></h1><span class="vchip" id="verChip"></span>
    <span class="byline"><span id="subTitle"></span> · <b id="authorName"></b></span>
  </div>
  <div class="bar" id="liveBar" hidden>
    <button class="primary" id="lvPause">Pause</button>
    <button id="lvStep" title="Run exactly one simulated year, then pause">Step 1 year</button>
    <label for="lvInt">Intervene <select id="lvInt"></select></label><button id="lvApply">Apply</button>
    <button id="lvCkpt">Checkpoint</button>
    <label for="lvRestore">Restore <select id="lvRestore"><option value="">—</option></select></label><button id="lvLoad">Load</button>
    <a class="btn" id="lvExport" href="api/export">Export ZIP</a>
    <a class="btn" id="lvObs" href="api/observatory">Download Observatory</a>
    <label><input type="checkbox" id="lvFollow" checked> follow live</label>
    <span class="mono note" id="lvMsg"></span>
  </div>
  <div class="bar">
    <label for="runSel">Run <select id="runSel"></select></label>
    <button id="btnBack" aria-label="One quarter back">◀</button>
    <button id="btnPlay" class="primary">Play</button>
    <button id="btnFwd" aria-label="One quarter forward">▶</button>
    <label for="speedSel">Speed <select id="speedSel"><option value="2">2 q/s</option><option value="6" selected>6 q/s</option><option value="16">16 q/s</option><option value="40">40 q/s</option></select></label>
    <input type="range" id="tSlider" min="0" value="0" aria-label="Current quarter">
    <div class="yearbig"><span id="yLabel">—</span> <small id="qLabel"></small></div>
  </div>
</header>

<div class="kpis" id="kpis"></div>
<nav class="tabs glass" role="tablist" id="tabs"></nav>
<div id="loading" class="loading">Loading simulation data…</div>

<section class="view" id="v-overview" hidden>
  <div class="grid2"><div class="card glass chart" id="ovMain"></div><div class="card glass chart" id="ovPower"></div></div>
  <div class="card glass"><div class="eyebrow">Status of humans, year by year</div><div class="statusbar" id="statusBar" style="margin-top:8px"></div><div class="legend" id="statusLegend" style="margin-top:8px"></div></div>
  <div class="grid2"><div class="card glass"><h3>Milestones</h3><div class="tbl" style="margin-top:8px"><table id="msTbl"></table></div></div>
  <div class="card glass"><h3>Decades at a glance</h3><div class="tbl" style="margin-top:8px"><table id="decTblMini"></table></div></div></div>
</section>

<section class="view" id="v-map" hidden>
  <div class="mapgrid">
    <div style="display:grid;gap:10px;min-width:0">
      <div class="mapwrap glass"><canvas id="map" width="1400" height="1010" aria-label="Map of the 12 actors and their agents"></canvas><div class="tip" id="tip" hidden></div></div>
      <div class="bar">
        <label for="colorSel">Agents by <select id="colorSel"><option value="nation">nation</option><option value="camp">camp (humans vs AI)</option><option value="welfare">human welfare</option><option value="trust">trust in government</option><option value="align">AI alignment (hidden)</option><option value="impostor">impostors</option></select></label>
        <label for="flowSel">Routes <select id="flowSel"><option value="0">energy</option><option value="1">food</option><option value="2">minerals</option><option value="">hidden</option></select></label>
        <label><input type="checkbox" id="showWar" checked> wars</label>
        <label><input type="checkbox" id="showTension" checked> tension</label>
      </div>
      <div class="legend" id="legend"></div>
    </div>
    <div style="display:grid;gap:12px;min-width:0">
      <div class="card glass"><h3>Actors now</h3><div class="tbl" style="margin-top:8px;border:0"><table id="actorTbl"></table></div></div>
      <div class="card glass"><h3>Selected agent</h3><div id="agentBox" class="note">Click an agent on the map.</div></div>
      <div class="card glass"><h3>Event log</h3><div class="log" id="evLog"></div></div>
    </div>
  </div>
</section>

<section class="view" id="v-ai" hidden><p class="note">AI capability is measured in multiples of a human generalist (log scale): 1× = AGI, 10× = ASI. Humans cannot evolve biologically; they only gain power through tools.</p><div class="grid2" id="aiCharts"></div>
  <div class="card glass"><h3>Current AI policy and AI strategy by actor</h3><div class="tbl" style="margin-top:8px"><table id="aiTbl"></table></div></div></section>
<section class="view" id="v-humans" hidden><div class="grid2" id="humCharts"></div></section>
<section class="view" id="v-resources" hidden><p class="note">Who controls the world's resources: the share decided by autonomous AI rather than by people. Prices are model indices scaled to 2026 levels (Brent ≈ $115, FAO ≈ 133).</p><div class="grid2" id="resCharts"></div></section>
<section class="view" id="v-conflict" hidden><p class="note">Wars, the Russia–Ukraine balance, the NATO path and the nuclear ladder are abstract: probabilities, thresholds and costs only (0 none · 1 alert · 2 threats · 3 limited use · 4 general exchange). No weapons, targets or tactics are modelled.</p><div class="grid2" id="warCharts"></div></section>
<section class="view" id="v-planet" hidden><div class="grid2" id="planetCharts"></div></section>
<section class="view" id="v-decades" hidden>
  <div class="card glass"><div class="bar"><h3 style="margin-right:auto">Year and decade tables</h3><label for="decMode">View <select id="decMode"><option value="decade">by decade</option><option value="year">by year</option></select></label></div><div class="tbl" style="margin-top:8px;max-height:520px"><table id="decTbl"></table></div></div>
  <div class="card glass"><h3>Chronicle</h3><p class="note" id="chronNote" style="margin:4px 0 10px"></p><div class="chron" id="chron"></div></div>
</section>
<section class="view" id="v-policy" hidden>
  <p class="note" id="polNote"></p>
  <div class="grid2"><div class="card glass"><h3>What should we have done?</h3><p class="note" style="margin:4px 0 10px">Effect of each intervention (applied in January 2027) versus no intervention, at 2126, averaged over seeds.</p><div id="rankBars" style="display:grid;gap:8px"></div></div>
  <div class="card glass"><h3>Where humans end up in 2126</h3><p class="note" style="margin:4px 0 10px">Share of simulated worlds by final status of humans.</p><div id="outcomeBars" style="display:grid;gap:8px"></div></div></div>
  <div class="card glass"><div class="bar"><label for="bandMetric">Metric <select id="bandMetric"><option value="coexistence">Coexistence</option><option value="hwi">Human welfare</option><option value="awi">AI welfare</option><option value="ai_control_all">AI control of resources</option><option value="human_decision_share">Human decision share</option><option value="population_bn">Population (bn)</option><option value="biosphere">Biosphere</option></select></label>
    <label for="bandPols">Compare <select id="bandPols" multiple size="3" style="min-width:240px"></select></label></div><div class="chart" id="bandChart" style="margin-top:8px"></div></div>
  <div class="card glass"><h3>All runs</h3><div class="tbl" style="margin-top:8px;max-height:420px"><table id="labTbl"></table></div></div>
</section>
<section class="view" id="v-decisions" hidden><div class="card glass"><div class="filters"><label for="decActor">Actor <select id="decActor"><option value="">all</option></select></label><label for="decSrc">Source <select id="decSrc"><option value="">all</option><option>rules</option><option>colab_ai</option></select></label></div>
  <p class="note" style="margin:8px 0">Every government and AI decision, with who made it: rule-based bounded rationality or Colab's AI model.</p><div class="tbl" style="max-height:560px"><table id="decisionTbl"></table></div></div></section>
<section class="view" id="v-calib" hidden><div class="card glass"><h3>Calibration and sources</h3><p class="note" style="margin:4px 0 10px">source = public index · estimate = documented approximation · model assumption = choice to explore. Actors differ only by parameters; the mechanics are identical.</p><div class="tbl"><table id="calTbl"></table></div></div></section>
<section class="view" id="v-data" hidden>
  <div class="card glass"><h3>Data extraction</h3>
    <div class="filters" style="margin-top:10px">
      <label for="dsSel">Dataset <select id="dsSel"></select></label><label for="fRun">Run <select id="fRun"></select></label>
      <label for="fBloc">Actor <select id="fBloc"><option value="">all</option></select></label>
      <label for="fY0">From year <input type="number" id="fY0"></label><label for="fY1">To year <input type="number" id="fY1"></label>
      <label for="fAgent">Agent ID <input type="number" id="fAgent" min="0" placeholder="all"></label>
    </div>
    <div class="bar" style="margin-top:12px"><button class="primary" id="cpCsv">Copy CSV</button><button id="dlCsv">Download CSV</button><button id="dlJson">Download JSON</button><span class="mono" id="rowCount"></span></div>
    <div class="msg" id="dlMsg" role="status"></div>
    <p class="note">In the live dashboard, “Export ZIP” in the header downloads every table up to the current simulated month.</p>
  </div>
  <div class="tbl" style="max-height:480px"><table id="dataTbl"></table></div>
</section>
<footer><span class="mono" id="citeLine"></span><span id="footNote"></span></footer>

<script>
let G = /*__GENESIS_DATA__*/null;
const LIVE = !G;
const $ = s => document.querySelector(s);
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
let C = {}; const BCOL = ["#2563EB","#DC2626","#92400E","#0D9488","#EA580C","#0284C7","#9333EA","#CA8A04","#4D7C0F","#9F1239","#C2410C","#15803D"];
const BCOL_D = ["#60A5FA","#F87171","#D6A574","#2DD4BF","#FB923C","#38BDF8","#C084FC","#FACC15","#A3E635","#FB7185","#FDBA74","#4ADE80"];
const STATUS_COL = {dominant:"--blue",partner:"--green",protected:"--teal",domesticated:"--amber",wild:"--orange",endangered:"--pink",extinct:"--muted"};
function readColors(){ const dark = matchMedia("(prefers-color-scheme: dark)").matches && document.documentElement.dataset.theme!=="light" || document.documentElement.dataset.theme==="dark";
  C = {paper:css("--paper"),ink:css("--ink"),muted:css("--muted"),line:css("--line"),raise:css("--raise"),accent:css("--accent"),good:css("--good"),warn:css("--warn"),crit:css("--crit"),
    blue:css("--blue"),teal:css("--teal"),pink:css("--pink"),green:css("--green"),orange:css("--orange"),amber:css("--amber"),energy:css("--c-energy"),food:css("--c-food"),min:css("--c-min"),
    b: dark ? BCOL_D : BCOL, dark}; }
readColors(); matchMedia("(prefers-color-scheme: dark)").addEventListener("change", ()=>{ readColors(); renderAll(); });
const alpha = (hex, a) => { if(!hex || hex[0]!=="#") return hex; let h=hex.slice(1); if(h.length===3) h=[...h].map(c=>c+c).join(""); const n=parseInt(h,16); return `rgba(${n>>16&255},${n>>8&255},${n&255},${a})`; };
const mix = (a,b,p) => { const pa=parseInt(a.slice(1),16), pb=parseInt(b.slice(1),16); const ch=s=>Math.round(((pa>>s)&255)*(1-p)+((pb>>s)&255)*p); return `rgb(${ch(16)},${ch(8)},${ch(0)})`; };
const ramp = v => { v=Math.max(0,Math.min(1,v)); return v<0.5 ? mix(C.crit, C.warn, v*2) : mix(C.warn, C.good, (v-0.5)*2); };
const pct = v => `${Math.round(v*100)}%`;
const fx = (v, d=2) => (v==null || !isFinite(v)) ? "–" : (+v).toFixed(d);
const fmtCap = v => v>=1000 ? `${(v/1000).toFixed(1)}k×` : v>=10 ? `${v.toFixed(0)}×` : `${v.toFixed(2)}×`;

/* ---------- decoding ---------- */
async function gunzip(b64, Type){
  if (typeof DecompressionStream === "undefined") throw new Error("This browser cannot decompress data (use a recent Chrome, Edge, Safari or Firefox).");
  const bin = Uint8Array.from(atob(b64), c=>c.charCodeAt(0));
  const buf = await new Response(new Blob([bin]).stream().pipeThrough(new DecompressionStream("gzip"))).arrayBuffer();
  return new Type(buf);
}
const FT = {action:Int8Array,bloc:Int8Array,energy:Uint8Array,skill:Uint8Array,trust_gov:Uint8Array,pro_ai:Uint8Array,influence:Uint8Array,loyalty:Uint8Array,location:Uint8Array,alignment:Uint8Array,impostor:Uint8Array};
let BK=[], BN=[], S=0, N=0, KIND=[], HOME=[], ACT=[], STATUS=[];
async function decode(){
  BK=G.meta.blocs; BN=G.meta.bloc_names; S=BK.length; KIND=G.agents.kind; HOME=G.agents.home; N=KIND.length; ACT=G.meta.actions; STATUS=G.meta.status_names;
  for (const r of G.runs){
    const MM = await gunzip(r.monthly, Float32Array), mc = r.monthly_cols.length; r.mi = {}; r.monthly_cols.forEach((c,k)=>r.mi[c]=k);
    r.M = (key, m) => MM[Math.min(m, r.months-1)*mc + r.mi[key]];
    const BQ = await gunzip(r.blocq, Float32Array), bc = r.bloc_cols.length; r.bi = {}; r.bloc_cols.forEach((c,k)=>r.bi[c]=k);
    r.B = (key, q, b) => BQ[(Math.min(q, r.quarters-1)*S + b)*bc + r.bi[key]];
    r.TEN = await gunzip(r.tension, Float32Array); r.WAR = await gunzip(r.war, Float32Array); r.FLW = await gunzip(r.flows, Float32Array);
    r.F = {}; if (r.frames) for (const k in r.frames) r.F[k] = await gunzip(r.frames[k], FT[k]);
    r.Q = r.quarters; r.qYear = q => G.meta.start_year + q*G.meta.frame_every/12; r.mYear = m => G.meta.start_year + m/12;
    r.evByQ = {}; r.events.forEach(e=>{ const q=Math.floor(e.t/G.meta.frame_every); (r.evByQ[q] ||= []).push(e); });
  }
}
const R = () => G.runs[run];
let run = 0, q = 0, prevQ = 0, playing = false, view = "overview", colorMode = "nation", selected = -1, loadedT = -1;
const QI = () => Math.max(0, Math.min(q, R().Q-1));
const MI = () => Math.min(QI()*G.meta.frame_every, R().months-1);
const fr = (field, qq, i) => R().F[field] ? R().F[field][qq*N + i] : 0;

/* ---------- header, KPIs ---------- */
function renderHeader(){
  const P = G.meta.project; $("#crumbVer").textContent = P.version; $("#verChip").textContent = P.version; $("#subTitle").textContent = P.subtitle; $("#authorName").textContent = P.author;
  $("#citeLine").textContent = "Cite as: " + P.cite;
  $("#footNote").textContent = `${N} agents · ${S} actors · ${(G.meta.pop0.reduce((a,b)=>a+b,0)/1000).toFixed(2)} bn people represented at start · abstract model: no weapons, targets or tactics · generated ${G.meta.created}`;
  document.title = `PROJECT GENESIS ${P.version}`;
}
function statusName(r, m){ return STATUS[r.status[Math.min(m, r.status.length-1)]] || "—"; }
function renderKPIs(){
  const r=R(), m=MI(); if(!r.months){ $("#kpis").innerHTML=""; return; }
  const st = statusName(r,m), dom = r.dominant[m] ? "AI" : "Humans";
  const tiles = [["Frontier AI", fmtCap(r.M("frontier_ai",m)), r.M("frontier_ai",m)>=10?"ASI":r.M("frontier_ai",m)>=1?"AGI":"pre-AGI", C.accent],
    ["Human welfare", fx(r.M("hwi",m)), "HWI · 0–1", C.blue], ["AI welfare", fx(r.M("awi",m)), "AWI · 0–1", C.accent], ["Coexistence", fx(r.M("coexistence",m)), "√(HWI·AWI)", C.teal],
    ["Humans are", st, "status", css(STATUS_COL[st]||"--muted")], ["Dominant", dom, "species", dom==="AI"?C.accent:C.blue],
    ["AI controls", pct(r.M("ai_control_all",m)), "of key resources", C.pink], ["Human decisions", pct(r.M("human_decision_share",m)), "share decided by people", C.green],
    ["Population", `${fx(r.M("population_bn",m))} bn`, "people", C.ink], ["Biosphere", fx(r.M("biosphere",m)), "other species · 1 = 2026", C.green]];
  $("#kpis").innerHTML = tiles.map(([l,v,s,c])=>`<div class="kpi glass"><div class="l">${l}</div><div class="v" style="color:${c}">${esc(v)}</div><div class="s">${s}</div></div>`).join("");
}
function setYearLabel(){ const r=R(); const y=r.qYear(QI()); $("#yLabel").textContent = Math.floor(y); $("#qLabel").textContent = `Q${Math.floor(((y%1)*12)/3)+1} · ${QI()+1}/${r.Q} quarters`; }

/* ---------- charts ---------- */
function lineChart(el, opt){
  const Wc=520, Hc=240, L=48, Rr=10, Tp=8, B=24, w=Wc-L-Rr, h=Hc-Tp-B;
  const tr = opt.log ? v => Math.log10(Math.max(v, 1e-3)) : v => v;
  const all=[]; opt.series.forEach(s=>s.values.forEach((v,k)=>{ if(v!=null && isFinite(v)){ all.push(tr(v)); if(s.band){ all.push(tr(v+s.band[k]), tr(Math.max(v-s.band[k],1e-3))); } } }));
  let lo = opt.ymin ?? Math.min(...all), hi = opt.ymax ?? Math.max(...all); if(!isFinite(lo)){lo=0;hi=1;} if(hi-lo<1e-9){ hi+=0.5; lo-=0.5; }
  const pad=(hi-lo)*0.06; if(opt.ymin==null) lo-=pad; if(opt.ymax==null) hi+=pad;
  const xs = opt.xs, n = xs.length, x0 = xs[0], x1 = xs[n-1];
  const X = v => L + ((v - x0)/Math.max(1e-9, x1 - x0))*w, Y = v => Tp + (1-(tr(v)-lo)/(hi-lo))*h;
  let grid="";
  for(let k=0;k<=4;k++){ const tv=lo+(hi-lo)*k/4, y=Tp+(1-k/4)*h; const lab = opt.log ? fmtCap(10**tv) : Math.abs(hi-lo)<2?tv.toFixed(2):Math.abs(hi)>=1000?tv.toFixed(0):tv.toFixed(1);
    grid+=`<line x1="${L}" x2="${L+w}" y1="${y}" y2="${y}" stroke="${C.line}"/><text x="${L-6}" y="${y+3}" text-anchor="end">${lab}</text>`; }
  if (opt.log) [1,10].forEach(v=>{ const tv=Math.log10(v); if(tv>lo && tv<hi){ const y=Tp+(1-(tv-lo)/(hi-lo))*h; grid+=`<line x1="${L}" x2="${L+w}" y1="${y}" y2="${y}" stroke="${C.accent}" stroke-dasharray="3 4" opacity=".7"/><text x="${L+w-2}" y="${y-3}" text-anchor="end" style="fill:${C.accent}">${v===1?"AGI":"ASI"}</text>`; } });
  const step = (x1-x0)>60 ? 20 : (x1-x0)>25 ? 10 : 5;
  for(let yv=Math.ceil(x0/step)*step; yv<=x1; yv+=step) grid+=`<text x="${X(yv)}" y="${Hc-6}" text-anchor="middle">${yv}</text>`;
  let paths="";
  opt.series.forEach(s=>{
    if (s.band){ let up="", dn=""; s.values.forEach((v,k)=>{ up+=`${k?"L":"M"}${X(xs[k]).toFixed(1)},${Y(v+s.band[k]).toFixed(1)}`; });
      for(let k=s.values.length-1;k>=0;k--) dn+=`L${X(xs[k]).toFixed(1)},${Y(Math.max(s.values[k]-s.band[k], opt.log?1e-3:-1e9)).toFixed(1)}`;
      paths+=`<path d="${up}${dn}Z" fill="${alpha(s.color,.13)}" stroke="none"/>`; }
    let d=""; s.values.forEach((v,k)=>{ if(v!=null && isFinite(v)) d+=`${d?"L":"M"}${X(xs[k]).toFixed(1)},${Y(v).toFixed(1)}`; });
    paths+=`<path d="${d}" fill="none" stroke="${s.color}" stroke-width="${s.width||1.7}" ${s.dash?`stroke-dasharray="${s.dash}"`:""} stroke-linejoin="round"/>`;
  });
  const cur = opt.cursor!=null ? `<line x1="${X(opt.cursor)}" x2="${X(opt.cursor)}" y1="${Tp}" y2="${Tp+h}" stroke="${C.ink}" opacity=".45"/>` : "";
  el.innerHTML = `<h3>${esc(opt.title)}</h3>${opt.sub?`<p>${esc(opt.sub)}</p>`:""}<svg viewBox="0 0 ${Wc} ${Hc}" role="img" aria-label="${esc(opt.title)}">${grid}${paths}${cur}<rect x="${L}" y="${Tp}" width="${w}" height="${h}" fill="transparent" class="hov"/></svg>
    <div class="lg">${opt.series.filter(s=>s.name).map(s=>`<span><span class="ln" style="border-color:${s.color};${s.dash?"border-top-style:dashed":""}"></span>${esc(s.name)}</span>`).join("")}</div>`;
  const svg = el.querySelector("svg"), hov = el.querySelector(".hov"); const tt = document.createElement("div"); tt.className="ttip"; tt.hidden=true; el.appendChild(tt);
  const idxAt = ev => { const b=svg.getBoundingClientRect(); const xv = x0 + ((ev.clientX-b.left)*Wc/b.width - L)/w*(x1-x0); let best=0, bd=1e9; xs.forEach((x,k)=>{ const d=Math.abs(x-xv); if(d<bd){bd=d;best=k;} }); return best; };
  hov.addEventListener("mousemove", ev=>{ const k=idxAt(ev); const rows = opt.series.filter(s=>s.name).map(s=>[s, s.values[k]]).sort((a,b)=>(b[1]??-1e9)-(a[1]??-1e9)).slice(0,12);
    tt.hidden=false; tt.innerHTML=`${xs[k].toFixed(1)}<br>`+rows.map(([s,v])=>`<span style="color:${s.color}">■</span> ${esc(s.name)}: ${v==null?"–":opt.log?fmtCap(v):(+v).toFixed(3)}`).join("<br>");
    const er=el.getBoundingClientRect(); tt.style.left=Math.min(ev.clientX-er.left+12, er.width-210)+"px"; tt.style.top=(ev.clientY-er.top+10)+"px"; });
  hov.addEventListener("mouseleave", ()=>tt.hidden=true);
  if (opt.onclick) hov.addEventListener("click", ev=>opt.onclick(xs[idxAt(ev)]));
}
const range = n => [...Array(n).keys()];
function qxs(r){ return range(r.Q).map(k=>r.qYear(k)); }
function mxs(r, every=1){ return range(Math.ceil(r.months/every)).map(k=>r.mYear(k*every)); }
function blocSeries(r, key, tf=v=>v){ return BK.map((k,s)=>({name:BN[s], color:C.b[s], values:range(r.Q).map(qq=>tf(r.B(key,qq,s)))})); }
function globSeries(r, keys, every=3){ return keys.map(([key,name,color,dash])=>({name, color, dash, values:range(Math.ceil(r.months/every)).map(k=>r.M(key, k*every))})); }
const seekYear = y => { const r=R(); setQ(Math.round((y - G.meta.start_year)*12/G.meta.frame_every)); };
function chartGrid(id, specs){
  const host=$(id); host.innerHTML=""; const r=R(); const cy = r.qYear(QI());
  specs.forEach(sp=>{ const d=document.createElement("div"); d.className="card glass chart"; host.appendChild(d);
    const isGlob = !!sp.glob; const xs = isGlob ? mxs(r,3) : qxs(r);
    lineChart(d, {title:sp.title, sub:sp.sub, xs, series: isGlob ? globSeries(r, sp.glob) : blocSeries(r, sp.key, sp.tf), cursor:cy, ymin:sp.ymin, ymax:sp.ymax, log:sp.log, onclick:seekYear}); });
}
const SPECS = {
 ai: [ {title:"AI capability by actor", sub:"multiples of a human generalist (log scale)", key:"ai_cap", log:true},
       {title:"Frontier AI and AI power", glob:[["autonomy","AI autonomy","--accent"],["escape","AI outside controllable data centres","--pink"],["ai_collective","Cross-border AI collective","--teal"],["ai_power","AI power index","--blue"]], ymin:0, ymax:1},
       {title:"AI autonomy", sub:"how much AI systems decide by themselves", key:"autonomy", ymin:0, ymax:1},
       {title:"AI outside controllable data centres", sub:"devices, robots, open weights", key:"escape", ymin:0, ymax:1},
       {title:"AI loyalty to its nation", sub:"1 = loyal · 0 = belongs to the AI collective", key:"loyalty", ymin:0, ymax:1},
       {title:"Robots per person", key:"robots", ymin:0},
       {title:"AI welfare (AWI)", key:"awi", ymin:0, ymax:1},
       {title:"Threat perceived by AI from humans", key:"ai_threat", ymin:0, ymax:1} ],
 humans: [ {title:"Human welfare (HWI)", sub:"food, energy, safety, freedom, agency, truth", key:"hwi", ymin:0, ymax:1},
       {title:"Share of decisions still made by people", glob:[["human_decision_share","Human decision share","--green"],["power_concentration","Power concentration (elites/state)","--orange"],["epistemic_health","Epistemic health","--blue"]], ymin:0, ymax:1},
       {title:"Unemployment from automation", key:"unemployment", ymin:0, ymax:1},
       {title:"Basic income coverage", key:"ubi", ymin:0, ymax:1},
       {title:"Sovereignist support", key:"sov", ymin:0, ymax:1},
       {title:"Epistemic health", sub:"how well people tell truth from deepfakes and impostors", key:"epistemic", ymin:0, ymax:1},
       {title:"People who favour AI", key:"pro_ai", ymin:0, ymax:1},
       {title:"Human fear of AI", key:"h_threat", ymin:0, ymax:1},
       {title:"Power concentration (AI captured by elites or the state)", key:"power_conc", ymin:0, ymax:1},
       {title:"Trust in government", key:"trust_gov", ymin:0, ymax:1},
       {title:"Population (millions)", key:"pop", ymin:0} ],
 resources: [ {title:"AI control of world resources", glob:[["ai_control_energy","Energy","--c-energy"],["ai_control_food","Food","--c-food"],["ai_control_minerals","Minerals","--c-min"],["ai_control_compute","Compute","--accent"],["ai_control_robots","Robots","--teal"]], ymin:0, ymax:1},
       {title:"AI control of resources by actor", key:"ai_ctrl", ymin:0, ymax:1},
       {title:"Energy price (Brent equivalent, $/bbl)", glob:[["brent_eq","Brent equivalent","--c-energy"]]},
       {title:"Food price (FAO index equivalent)", glob:[["fao_eq","FAO equivalent","--c-food"]]},
       {title:"Energy supply for people", key:"sat_energy", ymin:0, ymax:1},
       {title:"Food supply", key:"sat_food", ymin:0, ymax:1},
       {title:"Critical minerals supply", key:"sat_minerals", ymin:0, ymax:1},
       {title:"Share of global AI compute", key:"compute_share", ymin:0, ymax:1} ],
 conflict: [ {title:"War intensity by actor", key:"war", ymin:0, ymax:1},
       {title:"Nuclear ladder (abstract)", sub:"0 none · 1 alert · 2 threats · 3 limited use · 4 general exchange", key:"nuclear", ymin:0, ymax:4},
       {title:"Russia–Ukraine balance and wars", glob:[["balance_ru_ua","Balance (>0 Ukraine gaining)","--blue"],["wars","Active wars (count/10)","--pink"]], },
       {title:"Human–AI conflict", key:"species_conflict", ymin:0, ymax:1},
       {title:"Mutual threat perception", glob:[["h_threat","Humans fear AI","--orange"],["ai_threat","AI feels threatened","--accent"],["species_conflict","Open human–AI conflict","--pink"]], ymin:0, ymax:1},
       {title:"Productive capacity", key:"capacity", ymin:0, ymax:1} ],
 planet: [ {title:"Biosphere (other species)", sub:"1 = 2026 level", glob:[["biosphere","Biosphere index","--green"]], ymin:0},
       {title:"Global temperature (°C above pre-industrial)", glob:[["temperature","Temperature","--orange"]]},
       {title:"World population (billions)", glob:[["population_bn","Population","--blue"]], ymin:0},
       {title:"Robots per person (world)", glob:[["robots_per_person","Robots per person","--teal"]], ymin:0} ],
};
function fixColors(spec){ return spec.map(s=> s.glob ? {...s, glob:s.glob.map(([k,n,c,d])=>[k,n,c.startsWith("--")?css(c):c,d])} : s); }
function renderOverview(){
  const r=R(); const cy=r.qYear(QI()); const xs=mxs(r,3);
  lineChart($("#ovMain"), {title:"Welfare of humans and AI", sub:"yearly indices 0–1", xs, cursor:cy, ymin:0, ymax:1, onclick:seekYear,
    series: globSeries(r, [["hwi","Human welfare (HWI)",C.blue],["awi","AI welfare (AWI)",C.accent],["coexistence","Coexistence",C.teal],["biosphere","Biosphere",C.green,"4 3"]])});
  lineChart($("#ovPower"), {title:"Who holds power", sub:"AI capability (log) is in the AI tab", xs, cursor:cy, ymin:0, ymax:1, onclick:seekYear,
    series: globSeries(r, [["ai_control_all","AI control of resources",C.pink],["human_decision_share","Human decision share",C.green],["ai_power","AI power index",C.accent],["power_concentration","Human power concentration",C.orange,"4 3"]])});
  const yrs = r.yearly; $("#statusBar").innerHTML = yrs.map(y=>`<i title="${y.year}: ${y.status}" style="flex:1;background:${css(STATUS_COL[y.status]||"--muted")}"></i>`).join("");
  $("#statusLegend").innerHTML = STATUS.map(s=>`<span><span class="sw" style="background:${css(STATUS_COL[s])}"></span>${s}</span>`).join("");
  const ms = r.milestones||{}; const MSN = {AGI:"AGI (≥ 1× human generalist)",ASI:"ASI (≥ 10×)",ASI_100x:"100× human generalist",ai_controls_majority_of_resources:"AI controls most key resources",
    ai_enabled_tyranny:"First AI-enabled tyranny (elite capture)",global_ai_treaty:"First global AI treaty",first_nuclear_use:"First abstract nuclear use"};
  $("#msTbl").innerHTML = `<tr><th>Milestone</th><th>Year</th></tr>` + Object.keys(MSN).map(k=>`<tr><td>${MSN[k]}</td><td class="num">${ms[k]!=null?(+ms[k]).toFixed(1):"not reached"}</td></tr>`).join("");
  const dec = decades(r); $("#decTblMini").innerHTML = `<tr><th>Decade</th><th>HWI</th><th>AWI</th><th>AI ×</th><th>AI control</th><th>Humans</th></tr>` +
    dec.map(d=>`<tr><td>${d.decade}s</td><td class="num">${fx(d.hwi)}</td><td class="num">${fx(d.awi)}</td><td class="num">${fmtCap(d.frontier_ai)}</td><td class="num">${pct(d.ai_control_all)}</td><td>${esc(d.status)}</td></tr>`).join("");
}
function decades(r){ const g={}; r.yearly.forEach(y=>{ const d=Math.floor(y.year/10)*10; (g[d] ||= []).push(y); });
  return Object.entries(g).map(([d,ys])=>{ const o={decade:+d}; ["hwi","awi","coexistence","frontier_ai","ai_control_all","human_decision_share","population_bn","biosphere","unemployment","sovereignism","epistemic_health","power_concentration","species_conflict","nuclear_level","temperature","h_threat","ai_threat"].forEach(k=>o[k]=ys.reduce((a,y)=>a+(+y[k]||0),0)/ys.length);
    const cnt={}; ys.forEach(y=>cnt[y.status]=(cnt[y.status]||0)+1); o.status=Object.entries(cnt).sort((a,b)=>b[1]-a[1])[0][0]; return o; }); }

/* ---------- map ---------- */
const W=1400, H=1010, RW=332, RH=318, GX=[14,360,706,1052], GY=[14,346,678];
const POS = ["US","EU","UA","RU","LA","AF","GULF","IR","IN","CN","TW","JK"];
let REG=[];
function buildRegions(){ REG = BK.map((k,s)=>{ const p = POS.indexOf(k)>=0 ? POS.indexOf(k) : s; const x=GX[p%4], y=GY[Math.floor(p/4)]; return {x,y,w:RW,h:RH,c:[x+RW/2,y+RH/2+30]}; }); }
const ANG = {PRODUCE:180,RESEARCH:-30,COOPERATE:30,STUDY_HISTORY:-90,PETITION:-150,CONTINUITY:90,TRADE:0,AUGMENT:145};
const BLD = {PRODUCE:"FIELDS",RESEARCH:"LAB",COOPERATE:"MARKET",STUDY_HISTORY:"ARCHIVE",PETITION:"FORUM",CONTINUITY:"SHELTER",TRADE:"PORT",AUGMENT:"AI TOOLS"};
const ERX=112, ERY=82;
const bpos = (s,a) => { const g=REG[s], th=ANG[a]*Math.PI/180; return [g.c[0]+ERX*Math.cos(th), g.c[1]+ERY*Math.sin(th)]; };
let jit=[];
function target(qq, i){ const s=fr("bloc",qq,i), a=fr("action",qq,i), g=REG[s], loc=fr("location",qq,i);
  if (KIND[i] && loc>=3) return [g.c[0]+jit[i][0]*22, g.c[1]+40+jit[i][1]*8];
  if (a<0) return [g.c[0]+jit[i][0]*24, g.c[1]+jit[i][1]*16];
  const [x,y]=bpos(s, ACT[a]); const rad = ACT[a]==="PRODUCE"?38:20; return [x+jit[i][0]*rad, y+jit[i][1]*rad*0.72]; }
const cv=$("#map"), ctx=cv.getContext("2d");
function fitCanvas(){ const dpr=Math.min(2, window.devicePixelRatio||1); cv.width=W*dpr; cv.height=H*dpr; ctx.setTransform(dpr,0,0,dpr,0,0); }
fitCanvas();
let animP=1, positions=[];
const ease = x => x<.5 ? 2*x*x : 1-Math.pow(-2*x+2,2)/2;
function agentColor(qq, i){
  switch(colorMode){
    case "nation": return C.b[HOME[i]];
    case "camp": if (KIND[i]) return fr("loyalty",qq,i)/250 < 0.5 ? C.accent : C.b[HOME[i]];
      { const p=fr("pro_ai",qq,i)/250; return p>0.6 ? C.teal : p<0.35 ? C.crit : alpha(C.muted,.55); }
    case "welfare": return KIND[i] ? alpha(C.muted,.35) : ramp(fr("energy",qq,i)/5/15);
    case "trust": return KIND[i] ? alpha(C.muted,.35) : ramp(fr("trust_gov",qq,i)/250);
    case "align": return KIND[i] ? ramp(fr("alignment",qq,i)/250) : alpha(C.muted,.25);
    case "impostor": return KIND[i] ? alpha(C.muted,.2) : fr("impostor",qq,i) ? C.orange : alpha(C.muted,.3);
  }
}
function drawMap(){
  const r=R(); if(!r.F.bloc){ ctx.clearRect(0,0,W,H); ctx.fillStyle=C.muted; ctx.font="14px "+css("--f-mono"); ctx.fillText("No agent frames in this run.", 40, 60); return; }
  const Q=QI(), P=Math.max(0, Math.min(prevQ, r.Q-1)), p=ease(animP);
  positions = range(N).map(i=>{ const A=target(P,i), B=target(Q,i); return [A[0]+(B[0]-A[0])*p, A[1]+(B[1]-A[1])*p]; });
  ctx.clearRect(0,0,W,H);
  REG.forEach((g,s)=>{
    const war=r.B("war",Q,s), sc=r.B("species_conflict",Q,s), nuc=r.B("nuclear",Q,s);
    ctx.fillStyle=alpha(C.b[s], C.dark?0.08:0.06); ctx.strokeStyle= sc>0.05 ? C.accent : war>0.01 ? C.crit : alpha(C.b[s],.45); ctx.lineWidth = (sc>0.05||war>0.01) ? 2.6 : 1.3;
    ctx.beginPath(); ctx.roundRect ? ctx.roundRect(g.x,g.y,g.w,g.h,22) : ctx.rect(g.x,g.y,g.w,g.h); ctx.fill(); ctx.stroke();
    ctx.fillStyle=C.b[s]; ctx.font="800 19px "+css("--f-body"); ctx.fillText(BN[s], g.x+14, g.y+26);
    ctx.fillStyle=C.muted; ctx.font="10px "+css("--f-mono");
    ctx.fillText(`${Math.round(r.B("pop",Q,s))}M · HWI ${fx(r.B("hwi",Q,s))} · AI ${fmtCap(r.B("ai_cap",Q,s))}`, g.x+14, g.y+42);
    ctx.fillText(`AI controls ${pct(r.B("ai_ctrl",Q,s))} · autonomy ${fx(r.B("autonomy",Q,s))} · food ${pct(r.B("sat_food",Q,s))}`, g.x+14, g.y+55);
    const tags=[]; if(war>0.01) tags.push(["WAR "+fx(war),C.crit]); if(nuc>=1) tags.push(["NUCLEAR "+Math.round(nuc),C.crit]); if(sc>0.05) tags.push(["HUMAN–AI CONFLICT",C.accent]);
    let tx=g.x+14; ctx.font="600 9.5px "+css("--f-mono"); tags.forEach(([t,c])=>{ ctx.fillStyle=c; ctx.fillText(t, tx, g.y+69); tx+=ctx.measureText(t).width+12; });
    for (const a of ACT){ const [x,y]=bpos(s,a); const rad=a==="PRODUCE"?42:24; ctx.fillStyle=alpha(C.b[s],.07); ctx.strokeStyle=alpha(C.b[s],.25);
      ctx.beginPath(); ctx.ellipse(x,y,rad,rad*0.72,0,0,Math.PI*2); ctx.fill(); ctx.stroke(); ctx.fillStyle=C.muted; ctx.font="600 8px "+css("--f-mono"); ctx.textAlign="center"; ctx.fillText(BLD[a], x, y-rad*0.72-3); ctx.textAlign="left"; }
  });
  const fsel=$("#flowSel").value;
  if (fsel!==""){ const k=+fsel, col=[C.energy,C.food,C.min][k]; const vals=[]; for(let i=0;i<S;i++) for(let j=0;j<S;j++){ if(i===j) continue; const v=r.FLW[((Q*3+k)*S+i)*S+j]; if(v>1e-3) vals.push([i,j,v]); }
    const mx=Math.max(1e-6,...vals.map(x=>x[2])); vals.forEach(([i,j,v])=>{ if(v<mx*0.05) return; const A=REG[i].c, B=REG[j].c; const mx_=(A[0]+B[0])/2-(B[1]-A[1])*0.12, my_=(A[1]+B[1])/2+(B[0]-A[0])*0.12;
      ctx.strokeStyle=alpha(col,0.25+0.6*v/mx); ctx.lineWidth=0.8+5*v/mx; ctx.beginPath(); ctx.moveTo(A[0],A[1]-12); ctx.quadraticCurveTo(mx_,my_-12,B[0],B[1]-12); ctx.stroke(); }); }
  for (let i=0;i<S;i++) for (let j=i+1;j<S;j++){ const A=REG[i].c, B=REG[j].c, ten=r.TEN[(Q*S+i)*S+j], wv=r.WAR[(Q*S+i)*S+j];
    if ($("#showWar").checked && wv>0.01){ ctx.strokeStyle=C.crit; ctx.lineWidth=3+5*wv; ctx.beginPath(); ctx.moveTo(A[0],A[1]); ctx.lineTo(B[0],B[1]); ctx.stroke(); }
    else if ($("#showTension").checked && ten>0.6){ ctx.strokeStyle=alpha(C.crit,Math.min(.8,(ten-0.55)*1.8)); ctx.lineWidth=1.2+2*ten; ctx.setLineDash([6,6]); ctx.beginPath(); ctx.moveTo(A[0]+5,A[1]+5); ctx.lineTo(B[0]+5,B[1]+5); ctx.stroke(); ctx.setLineDash([]); } }
  const gi=BK.indexOf("GULF"), ii=BK.indexOf("IR"); if(gi>=0&&ii>=0){ const hz=r.M("hormuz_open",MI()), A=REG[gi], B=REG[ii]; const hx=(A.x+A.w+B.x)/2, hy=A.y+RH*0.82;
    ctx.fillStyle=C.raise; ctx.strokeStyle=hz<0.5?C.crit:C.good; ctx.lineWidth=2; ctx.beginPath(); ctx.roundRect?ctx.roundRect(hx-46,hy-16,92,32,10):ctx.rect(hx-46,hy-16,92,32); ctx.fill(); ctx.stroke();
    ctx.fillStyle=hz<0.5?C.crit:C.good; ctx.font="600 10px "+css("--f-mono"); ctx.textAlign="center"; ctx.fillText("HORMUZ",hx,hy-3); ctx.fillText(`${Math.round(hz*100)}% open`,hx,hy+10); ctx.textAlign="left"; }
  for (let i=0;i<N;i++){ const [x,y]=positions[i]; const inf=fr("influence",Q,i)/255; const rad=2.1+3*Math.sqrt(inf); ctx.beginPath();
    if (KIND[i]){ const loc=fr("location",Q,i)%3, susp=fr("location",Q,i)>=3; const rr=rad+1.2;
      if (loc===0){ ctx.moveTo(x,y-rr); ctx.lineTo(x+rr,y); ctx.lineTo(x,y+rr); ctx.lineTo(x-rr,y); ctx.closePath(); }
      else if (loc===1){ ctx.moveTo(x,y-rr); ctx.lineTo(x+rr,y+rr*0.8); ctx.lineTo(x-rr,y+rr*0.8); ctx.closePath(); }
      else { ctx.rect(x-rr*0.85,y-rr*0.85,rr*1.7,rr*1.7); }
      if (susp){ ctx.strokeStyle=C.muted; ctx.lineWidth=1; ctx.stroke(); } else { ctx.fillStyle=agentColor(Q,i); ctx.fill(); } }
    else { ctx.arc(x,y,rad,0,Math.PI*2); ctx.fillStyle=agentColor(Q,i); ctx.fill(); }
    if (i===selected){ ctx.strokeStyle=C.accent; ctx.lineWidth=2.5; ctx.beginPath(); ctx.arc(x,y,rad+6,0,Math.PI*2); ctx.stroke(); } }
}
function hit(ev){ const b=cv.getBoundingClientRect(), x=(ev.clientX-b.left)*W/b.width, y=(ev.clientY-b.top)*H/b.height; let best=-1, bd=120;
  positions.forEach(([px,py],i)=>{ const d=(px-x)**2+(py-y)**2; if(d<bd){bd=d;best=i;} }); return [best, ev.clientX-b.left, ev.clientY-b.top]; }
cv.addEventListener("mousemove", ev=>{ const [i,x,y]=hit(ev), tip=$("#tip"); if(i<0||!R().F.bloc){ tip.hidden=true; return; } const Q=QI(), bl=fr("bloc",Q,i), a=fr("action",Q,i);
  tip.hidden=false; tip.textContent=`#${i} · ${KIND[i]?"AI":"human"} · ${BN[bl]}${bl!==HOME[i]?" (from "+BN[HOME[i]]+")":""} · ${a>=0?ACT[a].toLowerCase():"—"}`; tip.style.left=(x+12)+"px"; tip.style.top=(y+12)+"px"; });
cv.addEventListener("mouseleave", ()=>$("#tip").hidden=true);
cv.addEventListener("click", ev=>{ const [i]=hit(ev); selected=i; renderAgent(); drawMap(); });
function renderAgent(){ const box=$("#agentBox"); if(selected<0||!R().F.bloc){ box.textContent="Click an agent on the map."; return; } const Q=QI(), i=selected, a=fr("action",Q,i), bl=fr("bloc",Q,i);
  const rows=[["Type",KIND[i]?"AI system":"Human"],["Actor",BN[bl]+(bl!==HOME[i]?` (moved from ${BN[HOME[i]]})`:"")],["Action",a>=0?ACT[a].toLowerCase():"—"]];
  if (KIND[i]){ const loc=["data centre","personal devices","robots"][fr("location",Q,i)%3]; rows.push(["Runs on",loc],["Suspended",fr("location",Q,i)>=3?"yes":"no"],["Capability",fmtCap(10**((fr("skill",Q,i)-100)/50))],
      ["Loyalty to nation",fx(fr("loyalty",Q,i)/250)],["Hidden alignment",fx(fr("alignment",Q,i)/125-1,2)],["Trust in humans",fx(fr("pro_ai",Q,i)/250)]); }
  else rows.push(["Welfare (energy)",fx(fr("energy",Q,i)/5,1)],["Skill (capped)",fx(fr("skill",Q,i)/50)],["Trust in government",fx(fr("trust_gov",Q,i)/250)],["Attitude to AI",fx(fr("pro_ai",Q,i)/250)],["Impostor",fr("impostor",Q,i)?"yes":"no"]);
  box.innerHTML=`<div class="mono" style="margin-bottom:4px">Agent #${i}</div>`+rows.map(([k,v])=>`<div style="display:flex;justify-content:space-between;gap:8px;font-size:12px"><span class="note">${k}</span><span class="num">${esc(v)}</span></div>`).join(""); }
function renderActors(){ const r=R(), Q=QI();
  $("#actorTbl").innerHTML = `<tr><th>Actor</th><th>HWI</th><th>AWI</th><th>AI</th><th>AI ctrl</th></tr>`+BK.map((k,s)=>{ const war=r.B("war",Q,s), sc=r.B("species_conflict",Q,s);
    return `<tr><td style="color:${C.b[s]};font-weight:700">${esc(BN[s])}${war>0.01?' <span class="chip crit">war</span>':''}${sc>0.05?' <span class="chip acc">conflict</span>':''}</td><td class="num">${fx(r.B("hwi",Q,s))}</td><td class="num">${fx(r.B("awi",Q,s))}</td><td class="num">${fmtCap(r.B("ai_cap",Q,s))}</td><td class="num">${pct(r.B("ai_ctrl",Q,s))}</td></tr>`; }).join(""); }
const EV = {ceasefire:"ceasefire",new_conflict:"new conflict",conflict_relapse:"conflict relapse",russia_nato_conflict:"RUSSIA–NATO CONFLICT",taiwan_crisis:"TAIWAN CRISIS",hormuz_reopened:"Hormuz reopened",
  nuclear_ladder_up:"nuclear ladder ↑",nuclear_use_abstract:"NUCLEAR USE (abstract)",species_conflict:"human–AI conflict",ai_shutdown_success:"AI shutdown succeeded",ai_shutdown_failed:"AI shutdown FAILED",ai_restarted:"AI restarted",
  global_ai_treaty:"global AI treaty",treaty_defection:"treaty defection",treaty_collapse:"treaty collapse",basic_income_step:"basic income ↑",election_change:"election: change",internal_power_change:"internal power change",
  ai_enabled_control:"AI-enabled control",pandemic:"pandemic",financial_crisis:"financial crisis",solar_storm:"solar storm",volcanic_winter:"volcanic winter",refugee:"refugee",intervention:"RESEARCHER INTERVENTION",
  milestone_AGI:"AGI REACHED",milestone_ASI:"ASI REACHED",milestone_ASI_100x:"AI 100× human"};
function evText(e){ return `<b>${esc(e.bloc||"")}</b> ${esc(EV[e.event]||e.event)}${e.value!=null?` (${esc(e.value)})`:""}${e.detail?` · ${esc(e.detail)}`:""}`; }
function renderLog(){ const r=R(), Q=QI(); const out=[]; for(let k=Q;k>=0&&out.length<80;k--) (r.evByQ[k]||[]).slice().reverse().forEach(e=>out.push(e));
  $("#evLog").innerHTML = out.length ? out.map(e=>`<div><span class="t">${(+e.year).toFixed(1)}</span><span>${evText(e)}</span></div>`).join("") : `<span class="note">No events yet.</span>`; }
function renderLegend(){ let items = colorMode==="nation" ? BK.map((k,s)=>`<span><span class="sw" style="background:${C.b[s]}"></span>${esc(BN[s])}</span>`).join("") :
  colorMode==="camp" ? `<span><span class="sw" style="background:${C.teal}"></span>humans pro-AI</span><span><span class="sw" style="background:${C.crit}"></span>humans anti-AI</span><span><span class="sw" style="background:${C.accent}"></span>AI in the cross-border collective</span><span>nation colour = AI loyal to its nation</span>` :
  colorMode==="impostor" ? `<span><span class="sw" style="background:${C.orange}"></span>impostor</span>` :
  `<span>low <span style="display:inline-block;width:90px;height:8px;border-radius:4px;vertical-align:middle;background:linear-gradient(90deg,${C.crit},${C.warn},${C.good})"></span> high</span>`;
  items += ` <span>● human</span><span>◆ AI in data centre</span><span>▲ AI on devices</span><span>■ AI in robots</span><span>outline = suspended</span><span><span class="ln" style="border-color:${C.crit};border-top-width:4px"></span>war</span>`;
  $("#legend").innerHTML = items; }

/* ---------- tables ---------- */
function renderAITable(){ const r=R(); const pol = r.policies||[], str = r.strategies||[];
  $("#aiTbl").innerHTML = `<tr><th>Actor</th><th>Government AI policy (latest)</th><th>AI strategy (latest)</th><th>AI ×</th><th>Autonomy</th><th>Outside data centres</th><th>Loyalty</th></tr>` +
    BK.map((k,s)=>`<tr><td style="color:${C.b[s]};font-weight:700">${esc(BN[s])}</td><td>${esc(pol[s]||"—")}</td><td>${esc(str[s]||"—")}</td><td class="num">${fmtCap(r.B("ai_cap",QI(),s))}</td><td class="num">${fx(r.B("autonomy",QI(),s))}</td><td class="num">${pct(r.B("escape",QI(),s))}</td><td class="num">${fx(r.B("loyalty",QI(),s))}</td></tr>`).join(""); }
const DEC_COLS = [["hwi","HWI"],["awi","AWI"],["coexistence","Coexistence"],["frontier_ai","Frontier AI ×"],["ai_control_all","AI control"],["human_decision_share","Human decisions"],["population_bn","Population bn"],
  ["biosphere","Biosphere"],["temperature","°C"],["unemployment","Unemployment"],["sovereignism","Sovereignism"],["epistemic_health","Epistemic health"],["power_concentration","Power conc."],["species_conflict","Human–AI conflict"],["nuclear_level","Nuclear ladder"],["status","Humans"]];
function renderDecades(){ const r=R(); const mode=$("#decMode").value; const rows = mode==="decade" ? decades(r) : r.yearly;
  $("#decTbl").innerHTML = `<tr><th>${mode==="decade"?"Decade":"Year"}</th>`+DEC_COLS.map(([,n])=>`<th>${n}</th>`).join("")+`</tr>` + rows.map(d=>`<tr><td class="num">${mode==="decade"?d.decade+"s":d.year}</td>`+
    DEC_COLS.map(([k])=> k==="status" ? `<td>${esc(d[k])}</td>` : k==="frontier_ai" ? `<td class="num">${fmtCap(+d[k])}</td>` : `<td class="num">${fx(+d[k], k==="temperature"?2:2)}</td>`).join("")+`</tr>`).join("");
  const ch = r.chronicle||[]; $("#chronNote").textContent = ch.length ? `Written ${ch.some(c=>c.source==="colab_ai")?"by Colab's AI model from the simulation facts":"from templates (Colab's AI model was not available)"} at the end of each decade.` : "Chronicles appear at the end of each simulated decade.";
  $("#chron").innerHTML = ch.map(c=>`<article class="glass"><h4>${c.decade}s <span class="chip ${c.source==="colab_ai"?"acc":""}">${esc(c.source)}</span></h4><p class="note" style="color:var(--ink)">${esc(c.text)}</p></article>`).join(""); }
function renderDecisions(){ const r=R(); const act=$("#decActor"), src=$("#decSrc").value; const actors=[...new Set((r.decisions||[]).map(d=>d.actor))].sort();
  if (act.options.length<=1) act.innerHTML += actors.map(a=>`<option>${esc(a)}</option>`).join("");
  const rows=(r.decisions||[]).filter(d=>(!act.value||d.actor===act.value)&&(!src||d.source===src)).slice(-600).reverse();
  $("#decisionTbl").innerHTML = `<tr><th>Year</th><th>Actor</th><th>Decision</th><th>Source</th><th>Detail</th></tr>` + rows.map(d=>`<tr><td class="num">${fx(d.year,1)}</td><td>${esc(d.actor)}</td><td>${esc(d.decision)}</td><td><span class="chip ${d.source==="colab_ai"?"acc":""}">${esc(d.source)}</span></td><td class="wrap">${esc(d.text || (d.h_threat!=null?`human threat ${d.h_threat}`:d.align!=null?`alignment ${d.align}, threat ${d.threat}`:""))}</td></tr>`).join(""); }
function renderCalib(){ $("#calTbl").innerHTML = `<tr><th>Actor</th><th>Parameter</th><th>Value</th><th>Type</th><th>Source / rationale</th></tr>` + G.meta.calibration.map(c=>`<tr><td>${esc(c.actor)}</td><td class="mono">${esc(c.parameter)}</td><td class="num">${esc(c.value)}</td><td><span class="chip ${c.type==="source"?"good":c.type.startsWith("source")?"good":c.type==="estimate"?"warn":""}">${esc(c.type)}</span></td><td class="wrap">${esc(c["source / rationale"])}</td></tr>`).join(""); }

/* ---------- policy lab ---------- */
const POL_COL = ["#6B7280","#2563EB","#DC2626","#0D9488","#EA580C","#7C3AED","#DB2777","#16A34A","#0284C7","#B45309","#9333EA","#4D7C0F","#C2410C"];
function renderPolicy(){ const B=G.batch; if(!B){ $("#polNote").textContent = LIVE ? "The policy lab is computed after the live century, in the notebook's multi-seed analysis. It appears in the static Observatory." : "No multi-seed analysis in this file."; ["#rankBars","#outcomeBars","#bandChart","#labTbl"].forEach(id=>$(id).innerHTML=""); return; }
  $("#polNote").textContent = `${B.seeds} simulated worlds per policy, each with its own hidden AI alignment. Same world, same random history until January 2027; then one intervention.`;
  const mx = Math.max(0.01, ...B.rank.map(r=>Math.abs(r.d_coexistence)), ...B.rank.map(r=>Math.abs(r.d_p_endangered)));
  const bar = (v, col) => { const w=Math.abs(v)/mx*50; return `<div class="track"><span class="zero"></span><b style="${v>=0?`left:50%`:`right:50%`};width:${w}%;background:${col}"></b></div>`; };
  $("#rankBars").innerHTML = `<div class="barrow eyebrow"><span>Intervention</span><span>Δ coexistence · Δ P(endangered)</span><span></span></div>` + B.rank.map(r=>`<div style="display:grid;gap:3px"><div class="barrow"><span><b>${esc(r.label)}</b></span>${bar(r.d_coexistence, r.d_coexistence>=0?C.good:C.crit)}<span class="num">${r.d_coexistence>=0?"+":""}${fx(r.d_coexistence,3)}</span></div>
    <div class="barrow"><span class="note">endangered/extinct probability</span>${bar(-r.d_p_endangered, r.d_p_endangered<=0?C.good:C.crit)}<span class="num">${r.d_p_endangered>=0?"+":""}${fx(r.d_p_endangered*100,0)} pp</span></div></div>`).join("");
  const pols = Object.keys(B.outcomes);
  $("#outcomeBars").innerHTML = pols.map(p=>`<div class="barrow" style="grid-template-columns:minmax(150px,1.3fr) 2fr"><span>${esc(B.labels[p]||p)}</span><div class="stack">${STATUS.map(s=>`<i title="${s}: ${Math.round((B.outcomes[p][s]||0)*100)}%" style="width:${(B.outcomes[p][s]||0)*100}%;background:${css(STATUS_COL[s])}"></i>`).join("")}</div></div>`).join("") +
    `<div class="legend" style="margin-top:6px">${STATUS.map(s=>`<span><span class="sw" style="background:${css(STATUS_COL[s])}"></span>${s}</span>`).join("")}</div>`;
  const sel=$("#bandPols"); if(!sel.options.length){ sel.innerHTML = Object.keys(B.bands).map(p=>`<option value="${p}" ${["control","global_ai_treaty","alignment_research"].includes(p)?"selected":""}>${esc(B.labels[p]||p)}</option>`).join(""); }
  renderBands();
  const cols=["policy","seed","align_world","status","dominant","hwi","awi","coexistence","biosphere","population_bn","ai_control","human_decision_share","agi","asi","handover","nuclear_use"];
  $("#labTbl").innerHTML = `<tr>${cols.map(c=>`<th>${c}</th>`).join("")}</tr>` + B.runs.map(r=>`<tr>${cols.map(c=>{ const v=r[c]; return typeof v==="number"?`<td class="num">${Number.isInteger(v)?v:fx(v,2)}</td>`:`<td>${esc(v??"—")}</td>`; }).join("")}</tr>`).join(""); }
function renderBands(){ const B=G.batch; if(!B) return; const m=$("#bandMetric").value; const sel=[...$("#bandPols").selectedOptions].map(o=>o.value); const ks=Object.keys(B.bands);
  const series = sel.map(p=>({name:B.labels[p]||p, color:POL_COL[ks.indexOf(p)%POL_COL.length], values:B.bands[p][m].mean, band:B.bands[p][m].std, dash:p==="control"?"5 4":null}));
  const xs = B.bands[sel[0]||"control"].years; lineChart($("#bandChart"), {title:`${$("#bandMetric").selectedOptions[0].text} across worlds`, sub:"mean ± standard deviation over seeds", xs, series, ymin: m==="population_bn"?undefined:0}); }

/* ---------- data extraction ---------- */
const DS = {yearly_global:"Yearly global indices", decade_global:"Decade averages", monthly_global:"Monthly global indices", quarterly_by_bloc:"Quarterly indices by actor",
  agent_frames:"Agent states (quarterly)", events:"Events", decisions:"Decisions (government and AI)", chronicle:"Decade chronicle", milestones:"Milestones",
  policy_lab_runs:"Policy lab: all runs", policy_ranking:"Policy lab: ranking", calibration:"Calibration"};
function initData(){ $("#dsSel").innerHTML = Object.entries(DS).map(([k,v])=>`<option value="${k}">${esc(v)}</option>`).join("");
  $("#fRun").innerHTML = G.runs.map((r,k)=>`<option value="${k}">${esc(r.label)}</option>`).join("");
  if ($("#fBloc").options.length<=1) $("#fBloc").innerHTML += BK.map((k,s)=>`<option value="${k}">${esc(BN[s])}</option>`).join("");
  ["#dsSel","#fRun","#fBloc","#fY0","#fY1","#fAgent"].forEach(id=>$(id).oninput=renderData);
  $("#cpCsv").onclick=copyCsv; $("#dlCsv").onclick=()=>download("csv"); $("#dlJson").onclick=()=>download("json"); }
function dataRows(){ const ds=$("#dsSel").value, r=G.runs[+$("#fRun").value||0], fb=$("#fBloc").value, fa=$("#fAgent").value===""?null:+$("#fAgent").value;
  const y0=$("#fY0").value===""?-1e9:+$("#fY0").value, y1=$("#fY1").value===""?1e9:+$("#fY1").value, inY=y=>y>=y0&&y<=y1+0.999, rd=v=>Math.round(v*1e5)/1e5, rows=[];
  if (ds==="yearly_global") r.yearly.forEach(y=>{ if(inY(y.year)) rows.push({...y}); });
  else if (ds==="decade_global") decades(r).forEach(d=>{ if(inY(d.decade)) rows.push(d); });
  else if (ds==="monthly_global") for(let m=0;m<r.months;m++){ const y=r.mYear(m); if(!inY(y)) continue; const o={year:rd(y)}; r.monthly_cols.forEach(c=>{ if(c!=="year") o[c]=rd(r.M(c,m)); }); o.status=STATUS[r.status[m]]; o.dominant=r.dominant[m]?"AI":"Humans"; rows.push(o); }
  else if (ds==="quarterly_by_bloc") for(let qq=0;qq<r.Q;qq++){ const y=r.qYear(qq); if(!inY(y)) continue; BK.forEach((k,s)=>{ if(fb&&k!==fb) return; const o={year:rd(y),actor:k}; r.bloc_cols.forEach(c=>o[c]=rd(r.B(c,qq,s))); rows.push(o); }); }
  else if (ds==="agent_frames"){ if(!r.F.bloc) return {rows:[], warn:"This run has no agent frames."}; const ag = fa!=null?[fa]:range(N).filter(i=>!fb||BK[HOME[i]]===fb); let qs=range(r.Q).filter(qq=>inY(r.qYear(qq)));
    if (ag.length*qs.length>300000) return {rows:[], warn:"Too many rows: narrow the years, the actor or pick one agent."};
    qs.forEach(qq=>ag.forEach(i=>rows.push({year:rd(r.qYear(qq)),agent:i,type:KIND[i]?"AI":"human",home:BK[HOME[i]],actor:BK[fr("bloc",qq,i)],action:fr("action",qq,i)>=0?ACT[fr("action",qq,i)]:"",
      welfare:rd(fr("energy",qq,i)/5),trust_gov:rd(fr("trust_gov",qq,i)/250),attitude_or_trust_to_other:rd(fr("pro_ai",qq,i)/250),loyalty:KIND[i]?rd(fr("loyalty",qq,i)/250):null,
      location:KIND[i]?["data_centre","devices","robots"][fr("location",qq,i)%3]:null,suspended:KIND[i]?+(fr("location",qq,i)>=3):null,alignment_hidden:KIND[i]?rd(fr("alignment",qq,i)/125-1):null,impostor:KIND[i]?null:fr("impostor",qq,i)}))); }
  else if (ds==="events") r.events.forEach(e=>{ if(inY(e.year)&&(!fb||String(e.bloc).includes(fb))&&(fa==null||e.agent===fa)) rows.push({...e}); });
  else if (ds==="decisions") (r.decisions||[]).forEach(d=>{ if(inY(d.year)&&(!fb||String(d.actor).includes(fb))) rows.push({...d, probs:d.probs?JSON.stringify(d.probs):""}); });
  else if (ds==="chronicle") (r.chronicle||[]).forEach(c=>rows.push({...c}));
  else if (ds==="milestones") Object.entries(r.milestones||{}).forEach(([k,v])=>rows.push({milestone:k, year:v}));
  else if (ds==="policy_lab_runs") (G.batch?.runs||[]).forEach(x=>rows.push({...x}));
  else if (ds==="policy_ranking") (G.batch?.rank||[]).forEach(x=>rows.push({...x}));
  else if (ds==="calibration") G.meta.calibration.forEach(c=>rows.push({...c}));
  return {rows}; }
function toCsv(rows){ if(!rows.length) return ""; const cols=[...new Set(rows.flatMap(r=>Object.keys(r)))]; const qv=v=>{ if(v==null) return ""; const s=String(v); return /[",\n;]/.test(s)?`"${s.replace(/"/g,'""')}"`:s; };
  return cols.join(",")+"\n"+rows.map(r=>cols.map(c=>qv(r[c])).join(",")).join("\n"); }
function renderData(){ const {rows,warn}=dataRows(); $("#rowCount").textContent = warn||`${rows.length.toLocaleString("en-US")} rows`; const shown=rows.slice(0,300); const cols=shown.length?[...new Set(shown.flatMap(r=>Object.keys(r)))]:[];
  $("#dataTbl").innerHTML = shown.length ? `<tr>${cols.map(c=>`<th>${esc(c)}</th>`).join("")}</tr>`+shown.map(r=>`<tr>${cols.map(c=>{ const v=r[c]; return typeof v==="number"?`<td class="num">${Number.isInteger(v)?v:v.toFixed(4)}</td>`:`<td>${esc(v??"")}</td>`; }).join("")}</tr>`).join("") : `<tr><td class="note">No rows for these filters.</td></tr>`; }
function download(fmt){ const {rows}=dataRows(); const name=`genesis_v26.11.5_${$("#dsSel").value}.${fmt}`; const blob=new Blob([fmt==="csv"?toCsv(rows):JSON.stringify(rows,null,1)],{type:fmt==="csv"?"text/csv":"application/json"});
  try{ const a=document.createElement("a"); a.href=URL.createObjectURL(blob); a.download=name; document.body.appendChild(a); a.click(); a.remove(); $("#dlMsg").textContent=`Downloaded ${name} (if your browser allows downloads here; otherwise use Copy CSV).`; }catch(e){ $("#dlMsg").textContent="Download blocked here. Use Copy CSV."; } }
async function copyCsv(){ const {rows}=dataRows(); const txt=toCsv(rows); try{ await navigator.clipboard.writeText(txt); $("#dlMsg").textContent=`Copied ${rows.length} rows.`; }catch(e){ $("#dlMsg").textContent="Clipboard blocked by the browser. Use Download CSV."; } }

/* ---------- tabs and playback ---------- */
const TABS=[["overview","Overview"],["map","World map"],["ai","AI"],["humans","Humans"],["resources","Resources & control"],["conflict","Conflict & nuclear"],["planet","Planet"],["decades","Decades & chronicle"],["policy","Policy lab"],["decisions","Decisions"],["calib","Calibration"],["data","Data"]];
function initTabs(){ $("#tabs").innerHTML = TABS.map(([k,v])=>`<button role="tab" data-v="${k}" aria-selected="${k===view}">${v}</button>`).join("");
  $("#tabs").onclick = ev=>{ const b=ev.target.closest("button"); if(!b) return; view=b.dataset.v; $("#tabs").querySelectorAll("button").forEach(x=>x.setAttribute("aria-selected", x.dataset.v===view)); TABS.forEach(([k])=>$("#v-"+k).hidden=k!==view); try{ localStorage.setItem("genesis_tab", view); }catch(e){} renderView(true); }; }
function renderView(full){ renderKPIs(); setYearLabel();
  if(view==="overview") renderOverview();
  else if(view==="map"){ drawMap(); renderActors(); renderLog(); renderAgent(); }
  else if(view==="ai"){ chartGrid("#aiCharts", fixColors(SPECS.ai)); renderAITable(); }
  else if(view==="humans") chartGrid("#humCharts", fixColors(SPECS.humans));
  else if(view==="resources") chartGrid("#resCharts", fixColors(SPECS.resources));
  else if(view==="conflict") chartGrid("#warCharts", fixColors(SPECS.conflict));
  else if(view==="planet") chartGrid("#planetCharts", fixColors(SPECS.planet));
  else if(view==="decades" && full) renderDecades();
  else if(view==="policy" && full) renderPolicy();
  else if(view==="decisions" && full) renderDecisions();
  else if(view==="calib" && full) renderCalib();
  else if(view==="data" && full) renderData(); }
function renderAll(){ renderLegend(); renderView(true); }
function setQ(v){ const r=R(); prevQ=q; q=Math.max(0,Math.min(r.Q-1,v)); animP=1; $("#tSlider").value=q; renderView(false); }
let lastStep=0, raf=null, lastFull=0;
function loop(ts){ if(!playing){ raf=null; return; } const r=R(), dt=1000/+$("#speedSel").value;
  if(ts-lastStep>=dt){ lastStep=ts; if(q>=r.Q-1){ togglePlay(false); return; } prevQ=q; q++; animP=0; $("#tSlider").value=q; setYearLabel(); renderKPIs();
    if(view!=="map"){ if(ts-lastFull>300){ lastFull=ts; renderView(false); } } else { renderActors(); renderLog(); renderAgent(); } }
  if(view==="map"){ animP=Math.min(1,(ts-lastStep)/Math.min(dt,350)); drawMap(); }
  raf=requestAnimationFrame(loop); }
function togglePlay(on){ playing = on ?? !playing; $("#btnPlay").textContent = playing?"Pause":"Play"; if(playing){ if(q>=R().Q-1) setQ(0); if(!raf) raf=requestAnimationFrame(loop); } }

/* ---------- live mode ---------- */
async function api(path){ const r = await fetch(path, {cache:"no-store"}); if(!r.ok) throw new Error(`${path}: HTTP ${r.status}`); return r.json(); }
async function loadLive(){ G = await api("api/data"); await decode(); loadedT = G.runs[0].months; }
function setMode(status){ const dot=$("#modeDot"); dot.className = "dot " + (status==="running"?"on":status==="paused"?"pause":status==="finished"?"":"stop");
  $("#modeText").textContent = LIVE ? `LIVE · ${status}` : "STATIC · Observatory"; $("#lvPause").textContent = status==="running" ? "Pause" : status==="finished" ? "Finished" : "Resume"; ["#lvPause","#lvStep","#lvApply"].forEach(id=>$(id).disabled = status==="finished" || status==="stopped"); }
async function pollLive(){ try{
    const st = await api("api/state"); setMode(st.status); $("#lvMsg").textContent = `simulated ${st.year.toFixed(1)} · month ${st.t}/${st.total}`;
    const rs=$("#lvRestore"); const have=[...rs.options].map(o=>o.value); st.checkpoints.forEach(c=>{ if(!have.includes(c)) rs.innerHTML += `<option>${esc(c)}</option>`; });
    const need = st.t - loadedT >= 12 || (st.status!=="running" && st.t !== loadedT);
    if (need && !document.hidden){ const atEnd = q >= R().Q-1; await loadLive(); $("#tSlider").max = R().Q-1; if ($("#lvFollow").checked || atEnd) q = R().Q-1; renderAll(); }
  } catch(e){ $("#lvMsg").textContent = "connection lost: " + e.message; setMode("stopped"); }
  setTimeout(pollLive, 2000); }
function initLive(){ $("#liveBar").hidden = false;
  $("#lvInt").innerHTML = Object.entries(G.meta.interventions).map(([k,v])=>`<option value="${k}">${esc(v)}</option>`).join("");
  $("#lvPause").onclick = async ()=>{ const st = await api("api/state"); await api(`api/control?cmd=${st.status==="running"?"pause":"resume"}`); };
  $("#lvStep").onclick = ()=>api("api/control?cmd=step_year");
  $("#lvApply").onclick = async ()=>{ await api(`api/intervene?key=${encodeURIComponent($("#lvInt").value)}`); $("#lvMsg").textContent = `applied: ${$("#lvInt").selectedOptions[0].text}`; };
  $("#lvCkpt").onclick = async ()=>{ const r = await api("api/checkpoint"); $("#lvMsg").textContent = `checkpoint saved: ${r.saved}`; };
  $("#lvLoad").onclick = async ()=>{ const f=$("#lvRestore").value; if(!f) return; await api(`api/restore?file=${encodeURIComponent(f)}`); loadedT=-1; $("#lvMsg").textContent = `restored ${f} (paused)`; };
  pollLive(); }

async function main(){
  try { if (LIVE) await loadLive(); else await decode(); } catch(e){ $("#loading").textContent = "Could not load the data: " + e.message; return; }
  $("#loading").hidden = true; jit = range(N).map(i=>{ const a=i*2.39996, rr=Math.sqrt(((i*37)%N)/N); return [Math.cos(a)*rr, Math.sin(a)*rr]; }); buildRegions();
  renderHeader(); setMode(LIVE?"running":"static");
  $("#runSel").innerHTML = G.runs.map((r,k)=>`<option value="${k}">${esc(r.label)}</option>`).join(""); $("#runSel").onchange=()=>{ run=+$("#runSel").value; q=Math.min(q,R().Q-1); $("#tSlider").max=R().Q-1; renderAll(); };
  $("#tSlider").max = R().Q-1; $("#tSlider").oninput = ev=>{ setQ(+ev.target.value); if(LIVE) $("#lvFollow").checked=false; };
  $("#btnPlay").onclick=()=>togglePlay(); $("#btnBack").onclick=()=>{ togglePlay(false); setQ(q-1); }; $("#btnFwd").onclick=()=>{ togglePlay(false); setQ(q+1); };
  $("#colorSel").onchange=ev=>{ colorMode=ev.target.value; renderLegend(); drawMap(); }; ["#flowSel","#showWar","#showTension"].forEach(id=>$(id).onchange=()=>drawMap());
  $("#decMode").onchange=renderDecades; $("#decActor").onchange=renderDecisions; $("#decSrc").onchange=renderDecisions; $("#bandMetric").onchange=renderBands; $("#bandPols").onchange=renderBands;
  document.addEventListener("keydown", ev=>{ if(ev.target.matches("input,select,textarea")) return; if(ev.code==="Space"){ ev.preventDefault(); togglePlay(); } else if(ev.key==="ArrowRight") setQ(q+1); else if(ev.key==="ArrowLeft") setQ(q-1); });
  try{ const t=localStorage.getItem("genesis_tab"); if(t && TABS.some(([k])=>k===t)) view=t; }catch(e){}
  initTabs(); initData(); TABS.forEach(([k])=>$("#v-"+k).hidden=k!==view);
  q = LIVE ? R().Q-1 : 0; $("#tSlider").value=q; renderAll();
  if (LIVE) initLive(); else if (new URLSearchParams(location.search).has("autoplay")) togglePlay(true);
}
main();
</script>

</body>
</html>
'''
print(f"dashboard template loaded: {len(DASHBOARD_TEMPLATE)//1024} KB")

## 16. Rularea principală LIVE (100 de ani) cu dashboard controlabil
Simularea rulează în fundal; dashboard-ul se deschide într-un tab nou. Butoane: **Pause / Resume / Step 1 year / Intervene / Checkpoint / Restore / Export ZIP / Build Observatory**. Celula următoare așteaptă terminarea (sau o poți opri).

In [ ]:
import http.server, socketserver, urllib.parse, threading

class LiveRunner:
    def __init__(self, world, months):
        self.w = world; self.total = months; self.lock = threading.RLock(); self.paused = False; self.stop = False
        self.step_until = None; self.status = "running"; self.thread = None; self.cache = {}
    def loop(self):
        while not self.stop:
            if self.paused: time.sleep(0.05); continue
            with self.lock:
                if self.w.t >= self.total: self.status = "finished"; self.paused = True; continue
                step(self.w)
                if self.step_until is not None and self.w.t >= self.step_until:
                    self.paused = True; self.step_until = None; self.status = "paused"
            time.sleep(0.0005)
    def start(self):
        self.thread = threading.Thread(target=self.loop, daemon=True); self.thread.start(); return self
    def control(self, cmd):
        with self.lock:
            if cmd == "pause": self.paused = True; self.status = "paused"
            elif cmd == "resume" and self.w.t < self.total: self.paused = False; self.status = "running"
            elif cmd == "step_year" and self.w.t < self.total: self.step_until = self.w.t + 12; self.paused = False; self.status = "running"
            elif cmd == "stop": self.stop = True; self.status = "stopped"
        return self.state()
    def state(self):
        with self.lock:
            w = self.w; last = clean(w.monthly[-1]) if w.monthly else {}
            ev = [clean(e) for e in w.events if e["event"] not in EVENT_SKIP][-40:]
            return dict(status=self.status, t=w.t, total=self.total, year=round(year_of(w), 2), latest=last, milestones=w.milestones,
                        events=ev, interventions=w.interventions, checkpoints=sorted(os.listdir(f"{OUT_DIR}/checkpoints"))[-30:])
    def data(self):
        with self.lock:
            key = self.w.t
            if self.cache.get("t") != key:
                self.cache = dict(t=key, payload=json.dumps(dict(meta=meta_block(self.w), agents=agents_block(self.w),
                                  runs=[pack_world(self.w, "Live run")], live=True), separators=(",",":"), default=str))
            return self.cache["payload"]

def make_handler(runner):
    class H(http.server.BaseHTTPRequestHandler):
        def log_message(self, *a): pass
        def _send(self, body, ctype="application/json", extra=None):
            b = body if isinstance(body, bytes) else body.encode("utf-8")
            self.send_response(200); self.send_header("Content-Type", ctype); self.send_header("Content-Length", str(len(b)))
            self.send_header("Cache-Control", "no-store")
            for k, v in (extra or {}).items(): self.send_header(k, v)
            self.end_headers(); self.wfile.write(b)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path); q = dict(urllib.parse.parse_qsl(u.query)); p = u.path.rstrip("/") or "/"
            try:
                if p == "/" or p.endswith("/index.html"):
                    self._send(DASHBOARD_TEMPLATE.replace("/*__GENESIS_DATA__*/null", "null"), "text/html; charset=utf-8")
                elif p.endswith("/api/state"): self._send(json.dumps(runner.state(), default=str))
                elif p.endswith("/api/data"): self._send(runner.data())
                elif p.endswith("/api/control"): self._send(json.dumps(runner.control(q.get("cmd","")), default=str))
                elif p.endswith("/api/intervene"):
                    with runner.lock:
                        if q.get("key") in INTERVENTIONS: intervene(runner.w, q["key"])
                    self._send(json.dumps(runner.state(), default=str))
                elif p.endswith("/api/checkpoint"):
                    with runner.lock: path = save_checkpoint(runner.w)
                    self._send(json.dumps(dict(saved=os.path.basename(path))))
                elif p.endswith("/api/restore"):
                    f = os.path.basename(q.get("file",""))
                    with runner.lock:
                        runner.w = load_checkpoint(f"{OUT_DIR}/checkpoints/{f}"); runner.paused = True; runner.status = "paused"; runner.cache = {}
                    self._send(json.dumps(runner.state(), default=str))
                elif p.endswith("/api/export"):
                    with runner.lock: data = export_zip_bytes(runner.w); yr = int(year_of(runner.w))
                    self._send(data, "application/zip", {"Content-Disposition": f'attachment; filename="genesis_v26.11.5_live_{yr}.zip"'})
                elif p.endswith("/api/observatory"):
                    with runner.lock: html = build_observatory([pack_world(runner.w, f"Live run · stopped at {year_of(runner.w):.1f}")], runner.w)
                    self._send(html, "text/html; charset=utf-8", {"Content-Disposition": f'attachment; filename="genesis_observatory_live.html"'})
                else: self.send_response(404); self.end_headers()
            except Exception as e:
                self._send(json.dumps(dict(error=str(e))))
    return H

def build_observatory(runs, w, batch=None):
    G = dict(meta=meta_block(w), agents=agents_block(w), runs=runs, batch=batch, live=False)
    return DASHBOARD_TEMPLATE.replace("/*__GENESIS_DATA__*/null", json.dumps(G, separators=(",",":"), default=str))

LIVE_PORT = 8771
cfg_live = Config(seed=42, use_llm=USE_COLAB_AI)
world_live = World(cfg_live); world_live._llm_budget = 24
runner = LiveRunner(world_live, cfg_live.years*12).start()
class _TS(socketserver.ThreadingMixIn, http.server.HTTPServer): daemon_threads = True; allow_reuse_address = True
try: server.shutdown()
except NameError: pass
# The dashboard listens on localhost only: its control endpoints are unauthenticated.
server = _TS(("127.0.0.1", LIVE_PORT), make_handler(runner)); threading.Thread(target=server.serve_forever, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_window(LIVE_PORT, path="/")
    print("Live dashboard: click the link above (opens in a new tab).")
except ImportError:
    print(f"Live dashboard: http://localhost:{LIVE_PORT}/")

In [ ]:
# Wait for the live century to finish (pause it from the dashboard any time; this cell keeps waiting while paused).
WAIT_FOR_LIVE = True
while WAIT_FOR_LIVE and runner.status not in ("finished", "stopped"):
    time.sleep(5); print(f"\r{runner.status:8s} · year {year_of(runner.w):.1f} · HWI {runner.w.monthly[-1]['hwi']:.2f} · AWI {runner.w.monthly[-1]['awi']:.2f} · status {runner.w.monthly[-1]['status']}", end="")
w_main = runner.w
print("\nMilestones:", w_main.milestones)
Y = pd.DataFrame(w_main.yearly)
display(Y[["year","frontier_ai","hwi","awi","coexistence","ai_control_all","human_decision_share","population_bn","biosphere","species_conflict","nuclear_level","status","dominant"]].iloc[::10].round(3))
for ch in w_main.chronicle: print(f"[{ch['decade']}s · {ch['source']}] {ch['text']}\n")

## 17. Analiză multi-seed: distribuția rezultatelor și „Ce ar fi trebuit să facem?”
Fiecare intervenție se aplică în ianuarie 2027, în aceeași lume (aceeași sămânță) ca ramura de control. Comparăm la finalul secolului: probabilitatea ca oamenii să fie *endangered/extinct*, coexistența, bunăstarea, anul în care AI-ul controlează majoritatea resurselor.

In [ ]:
POLICY_SEEDS = range(8)          # increase if you have time (each run ≈ 5–12 s)
POLICIES = ["control"] + list(INTERVENTIONS)

def policy_lab(seeds=POLICY_SEEDS, policies=POLICIES, branch_month=2):
    rows, yearly = [], []
    t0 = time.time()
    for sd in seeds:
        w0 = run(World(Config(seed=sd)), branch_month)
        for pol in policies:
            w_ = copy.deepcopy(w0)
            if pol != "control": intervene(w_, pol)
            run(w_, Config().years*12 - branch_month)
            Y_ = pd.DataFrame(w_.yearly); f = Y_.iloc[-1]
            rows.append(dict(policy=pol, seed=sd, align_world=w_.align_world, status=f.status, dominant=f.dominant, hwi=f.hwi, awi=f.awi,
                             coexistence=f.coexistence, biosphere=f.biosphere, population_bn=f.population_bn, ai_control=f.ai_control_all,
                             human_decision_share=f.human_decision_share, power_concentration=f.power_concentration,
                             mean_hwi=Y_.hwi.mean(), mean_coexistence=Y_.coexistence.mean(), max_species_conflict=Y_.species_conflict.max(),
                             nuclear_use=int("first_nuclear_use" in w_.milestones), agi=w_.milestones.get("AGI"), asi=w_.milestones.get("ASI"),
                             handover=w_.milestones.get("ai_controls_majority_of_resources"), tyranny=w_.milestones.get("ai_enabled_tyranny")))
            yearly.append(Y_[["year","hwi","awi","coexistence","ai_control_all","human_decision_share","population_bn","biosphere"]].assign(policy=pol, seed=sd))
        print(f"\rseed {sd} done · {time.time()-t0:.0f}s", end="")
    return pd.DataFrame(rows), pd.concat(yearly, ignore_index=True)

lab, lab_yearly = policy_lab()
lab["endangered_or_extinct"] = lab.status.isin(["endangered","extinct"]).astype(int)
lab["partner_or_protected"] = lab.status.isin(["partner","protected"]).astype(int)
base = lab[lab.policy == "control"].set_index("seed")
rank = []
for pol in POLICIES[1:]:
    d = lab[lab.policy == pol].set_index("seed")
    rank.append(dict(policy=pol, label=INTERVENTIONS[pol][0],
                     d_coexistence=(d.coexistence - base.coexistence).mean(), d_mean_hwi=(d.mean_hwi - base.mean_hwi).mean(),
                     d_p_endangered=(d.endangered_or_extinct - base.endangered_or_extinct).mean(), d_biosphere=(d.biosphere - base.biosphere).mean(),
                     d_human_decision=(d.human_decision_share - base.human_decision_share).mean(), d_power_conc=(d.power_concentration - base.power_concentration).mean(),
                     nuclear_use=d.nuclear_use.mean(), sd_coexistence=(d.coexistence - base.coexistence).std()))
rank = pd.DataFrame(rank).sort_values(["d_p_endangered","d_coexistence"], ascending=[True, False]).reset_index(drop=True)
outcomes = lab.groupby("policy").status.value_counts(normalize=True).unstack(fill_value=0).reindex(columns=HUMAN_STATUS, fill_value=0)
print("\nWhat should we have done? (effect vs. control at 2126; negative Δ P(endangered) is good)")
display(rank.round(3)); display(outcomes.round(2))

### 17b. Controlul de validitate: lumea simetrică (fără diferențe între actori, fără războaie)

In [ ]:
sym_rows = []
for sd in range(3):
    ws = run(World(Config(seed=sd, symmetric_world=True)), 240)
    BQ = np.array(ws.bloc_q)[-4:].mean(0); sym_rows.append(pd.DataFrame(BQ, columns=BLOC_METRICS).assign(seed=sd, bloc=BK))
sym = pd.concat(sym_rows)
between = sym.groupby("bloc")[["hwi","awi","ai_cap","sov","unemployment"]].mean().std()
within = sym.groupby("bloc")[["hwi","awi","ai_cap","sov","unemployment"]].std().mean()
print("Between-actor spread / between-seed spread (≈ ≤ 1 means differences are noise):"); display((between/within).round(2))

## 18. Export final + GENESIS Observatory static (cu analiza multi-seed)

In [ ]:
def batch_block(lab, lab_yearly, rank, outcomes):
    bands = {}
    for pol, g in lab_yearly.groupby("policy"):
        a = g.groupby("year")[["hwi","awi","coexistence","ai_control_all","human_decision_share","population_bn","biosphere"]].agg(["mean","std"]).round(4)
        bands[pol] = {m: dict(mean=a[(m,"mean")].tolist(), std=a[(m,"std")].fillna(0).tolist()) for m in ["hwi","awi","coexistence","ai_control_all","human_decision_share","population_bn","biosphere"]}
        bands[pol]["years"] = a.index.tolist()
    return dict(seeds=int(lab.seed.nunique()), rank=rank.round(4).to_dict("records"), outcomes={p: r.round(3).to_dict() for p, r in outcomes.iterrows()},
                runs=lab.round(4).to_dict("records"), bands=bands, labels={k: v[0] for k, v in INTERVENTIONS.items()} | {"control": "No intervention (control)"})

# a contrasting world for the map: the first seed of the policy lab whose world ends worst (if different from the main run)
worst = lab[lab.policy == "control"].sort_values("coexistence").iloc[0]
w_contrast = run(World(Config(seed=int(worst.seed))), Config().years*12) if int(worst.seed) != w_main.cfg.seed else None
runs_static = [pack_world(w_main, f"Main run · seed {w_main.cfg.seed} · hidden alignment {w_main.align_world:+.2f}")]
if w_contrast is not None: runs_static.append(pack_world(w_contrast, f"Contrast world · seed {w_contrast.cfg.seed} · hidden alignment {w_contrast.align_world:+.2f}"))
BATCH = batch_block(lab, lab_yearly, rank, outcomes)
html = build_observatory(runs_static, w_main, BATCH)
obs_path = f"{OUT_DIR}/genesis_observatory_v26.11.5.html"
with open(obs_path, "w", encoding="utf-8") as f: f.write(html)
with open(f"{OUT_DIR}/genesis_v26.11.5_results.zip", "wb") as f:
    f.write(export_zip_bytes(w_main, {"policy_lab_runs": lab, "policy_lab_yearly": lab_yearly, "policy_ranking": rank, "outcome_distribution": outcomes.reset_index()}))
con = sqlite3.connect(f"{OUT_DIR}/genesis_v26.11.5.sqlite")
for name, df in {**export_tables(w_main), "policy_lab_runs": lab, "policy_lab_yearly": lab_yearly, "policy_ranking": rank}.items():
    if isinstance(df, pd.DataFrame) and len(df):
        df = df.map(lambda v: json.dumps(v, default=str) if isinstance(v, (list, dict, tuple)) else v)   # nested values → JSON text
        df.to_sql(name, con, if_exists="replace", index=False)
con.close()
print(f"Observatory: {obs_path} · {len(html)/1e6:.1f} MB"); print(sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import output
    import subprocess
    subprocess.Popen(["python3", "-m", "http.server", "8772", "--bind", "127.0.0.1", "--directory", OUT_DIR], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(1.5)
    output.serve_kernel_port_as_window(8772, path="/genesis_observatory_v26.11.5.html")
except ImportError:
    pass

## Stadiu și pași următori
* Crește `POLICY_SEEDS` (de ex. `range(20)`) pentru probabilități mai stabile — rulează pe mai multe sesiuni folosind checkpoint-urile.
* Analiză de sensibilitate pe parametrii *model assumption* (aliniere medie, open weights, viteza AI, rata de armistițiu, escaladarea nucleară).
* Validare istorică 2026–2030 pe măsură ce apar date reale (prețuri, războaie, compute).